# Metabolic Vulnerabilities in Breast Cancer 
This notebook implements a deep learning framework for identifying metabolic vulnerabilities in cancer tissue using spatial transcriptomics (Visium) data. We use pathway-based features combined with spatial context to predict proliferative capacity and identify high-risk metabolic states.

In [ ]:
# imports Libraries
import os
import warnings
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse, stats
from scipy.spatial import distance_matrix
from scipy.sparse import coo_matrix
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
import seaborn as sns
import scanpy as sc
import gc
import squidpy as sq
import sccellfie
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA, NMF
from sklearn.metrics import roc_curve
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.model_selection import train_test_split, KFold
from sklearn.model_selection import GroupKFold
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    accuracy_score, roc_auc_score, confusion_matrix,
    mean_squared_error, r2_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from scipy.stats import ttest_rel, wilcoxon
from scipy.stats import mannwhitneyu
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
from statsmodels.stats.multitest import multipletests
from sklearn.pipeline import Pipeline

try:
    import gseapy as gp
    import decoupler as dc
    PATHWAY_AVAILABLE = True
except ImportError:
    PATHWAY_AVAILABLE = False
    

# Settings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Scanpy settings
sc.settings.verbosity = 2
sc.set_figure_params(dpi=100, figsize=(8, 6), frameon=False)
sc.settings.figdir = './figures/'

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Random seed: {RANDOM_SEED}")
print(f"PyTorch version: {torch.__version__}")
print(f"Scanpy version: {sc.__version__}")

In [ ]:
# UNIFORM PLOT STYLE 

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import ListedColormap

# Categorical palette (colour-blind-safe), used for clusters / niches / groups
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2',
           '#937860', '#DA8BC3', '#8C8C8C', '#CCB974', '#64B5CD',
           '#A1C9F4', '#FFB482']
CMAP_CATEGORICAL = ListedColormap(PALETTE)

def cat_color(i):
    """Return the i-th categorical colour, wrapping around PALETTE."""
    return PALETTE[int(i) % len(PALETTE)]

# Sequential / diverging colormaps for continuous values
CMAP_SEQUENTIAL = 'viridis'   
CMAP_DIVERGING = 'RdBu_r'     

# Semantic colours reused everywhere Low/High, Sig/Non-sig, or a single default

COLOR_LOW = PALETTE[0]         
COLOR_HIGH = PALETTE[3]      
COLOR_SIG = PALETTE[3]
COLOR_NOT_SIG = PALETTE[0]
COLOR_PRIMARY = PALETTE[0]     
COLOR_SECONDARY = PALETTE[1]
COLOR_TERTIARY = PALETTE[2]
COLOR_REFERENCE = '#4D4D4D'    

PROLIF_COLORS = {'Low': COLOR_LOW, 'High': COLOR_HIGH}
CLUSTER_COLORS = PALETTE

sns.set_theme(style='whitegrid', palette=PALETTE)
mpl.rcParams.update({
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.labelweight': 'bold',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': True,
    'axes.prop_cycle': mpl.cycler(color=PALETTE),
})

print(f"Uniform plot style loaded: {len(PALETTE)}-colour categorical palette, "
      f"'{CMAP_SEQUENTIAL}' sequential cmap, '{CMAP_DIVERGING}' diverging cmap.")


In [ ]:
# CONFIGURATION
from dataclasses import dataclass, field
from typing import Dict, List

import torch

@dataclass
class Config:
    
    # FILE PATHS 
    DATA_DIR: str = "Patient_1/"
    H5_FILE: str = "Visium_Human_Breast_Cancer_filtered_feature_bc_matrix.h5"
    SPATIAL_DIR: str = "spatial/1142243F_spatial"
        
    PATIENT_CONFIG: Dict[int, Dict[str, str]] = field(default_factory=lambda: {
        2: {
            'name': 'Patient 2',
            'data_dir': 'Patient_2/',  
            'h5_file': 'Visium_FFPE_Human_Breast_Cancer_filtered_feature_bc_matrix.h5',
            'spatial_dir': 'spatial'
        },
        3: {
            'name': 'Patient 3',
            'data_dir': 'Patient_3/',  
            'h5_file': 'Parent_Visium_Human_BreastCancer_filtered_feature_bc_matrix.h5',
            'spatial_dir': 'spatial'
        },
        4: {
            'name': 'Patient 4',
            'data_dir': 'Patient_4/',  
            'h5_file': 'CytAssist_Fresh_Frozen_Human_Breast_Cancer_filtered_feature_bc_matrix.h5',
            'spatial_dir': 'spatial'
        },
        
        5: {'name': 'Patient 5', 
            'data_dir': 'Patient_5/',
            'h5_file': 'CytAssist_FFPE_Protein_Expression_Human_Breast_Cancer_filtered_feature_bc_matrix.h5',
            'spatial_dir': 'spatial'},
        
        6: {'name': 'Patient 6', 
            'data_dir': 'Patient_6/',
            'h5_file': 'CytAssist_Fresh_Frozen_Human_Breast_Cancer_filtered_feature_bc_matrix.h5',
            'spatial_dir': 'spatial'},
        
        7: {'name': 'Patient 7', 
            'data_dir': 'Patient_7/',
            'h5_file': 'Visium_HD_16um_filtered_feature_bc_matrix.h5',
            'spatial_dir': 'spatial'}
        
    })
    
    # RANDOM SEED
    RANDOM_SEED: int = 42
    
    # QC THRESHOLDS
    MIN_COUNTS: int = 500
    MIN_GENES: int = 200
    MAX_MT_PCT: float = 20.0
    MIN_CELLS: int = 5
    
    # PATHWAY DISCOVERY PARAMETERS
    MIN_PATHWAY_GENES: int = 3  
    MIN_PATHWAY_COVERAGE: float = 0.20  
    N_TOP_PATHWAYS: int = 20  
    PATHWAY_FDR_THRESHOLD: float = 0.20  
    
    # PATHWAY DATABASES TO USE
    PATHWAY_DATABASES: List[str] = field(default_factory=lambda: [
        'MSigDB_Hallmark_2020',
        'KEGG_2021_Human',
        'Reactome_2022',
        'GO_Biological_Process_2023'
    ])
    
    # MODEL HYPERPARAMETERS 
    HIDDEN_DIM1: int = 64
    HIDDEN_DIM2: int = 32
    DROPOUT: float = 0.3
    LEARNING_RATE: float = 0.001
    WEIGHT_DECAY: float = 0.01
    BATCH_SIZE: int = 64
    N_EPOCHS: int = 150
    EARLY_STOPPING_PATIENCE: int = 20
    EARLY_STOPPING_MIN_DELTA: float = 0.001
    
    # PROLIFERATION TARGET GENES
    PROLIFERATION_GENES: List[str] = field(default_factory=lambda: [
        'MKI67', 'TOP2A', 'PCNA', 'CCNB1', 'CCNB2', 'CDK1', 'BIRC5'
    ])
    
    # TECHNICAL COVARIATES
    TECHNICAL_COVARIATES: List[str] = field(default_factory=lambda: [
        'n_counts', 'pct_counts_mt', 'pct_counts_ribo'
    ])
    
    def __post_init__(self):
        """Set random seeds for reproducibility."""
        np.random.seed(self.RANDOM_SEED)
        torch.manual_seed(self.RANDOM_SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(self.RANDOM_SEED)
            torch.cuda.manual_seed_all(self.RANDOM_SEED)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

# Initialize global config
CFG = Config()

print("Configuration loaded")
print(f"  Random seed: {CFG.RANDOM_SEED}")
print(f"  Pathway discovery mode: Data-driven")
print(f"  Pathway databases: {len(CFG.PATHWAY_DATABASES)}")
print(f"  Top pathways to select: {CFG.N_TOP_PATHWAYS}")
print(f"  FDR threshold: {CFG.PATHWAY_FDR_THRESHOLD}")
print(f"  Proliferation genes: {len(CFG.PROLIFERATION_GENES)}")
print(f"  Technical covariates: {len(CFG.TECHNICAL_COVARIATES)}")

In [ ]:
#  DIRECTORY SETUP & INITIALISATION

print("SETTING UP DIRECTORY STRUCTURE")

# Define all output directories
OUTPUT_DIRS = {
    'figures': 'figures',
    'tables': 'results/tables',
    'models': 'results/models', 
    'metrics': 'results/metrics',
    'processed_data': 'data/processed',
    'intermediate': 'data/intermediate'
}

# Create all directories
print("\nCreating output directories")
created = []
existing = []

for name, path in OUTPUT_DIRS.items():
    if not os.path.exists(path):
        os.makedirs(path, exist_ok=True)
        created.append(path)
        print(f"   Created: {path}")
    else:
        existing.append(path)
        print(f"   Exists:  {path}")

print(f"\nDirectory setup complete:")
print(f"  Created: {len(created)} directories")
print(f"  Existing: {len(existing)} directories")

# Verify all directories exist
for path in OUTPUT_DIRS.values():
    assert os.path.exists(path), f"Failed to create {path}"

print("\n All output directories ready")

# Store paths in config for easy access
class Paths:
    """Container for all file paths used in analysis."""
    FIGURES = OUTPUT_DIRS['figures']
    TABLES = OUTPUT_DIRS['tables']
    MODELS = OUTPUT_DIRS['models']
    METRICS = OUTPUT_DIRS['metrics']
    PROCESSED = OUTPUT_DIRS['processed_data']
    INTERMEDIATE = OUTPUT_DIRS['intermediate']

# Test write permissions
print("\nTesting write permissions")
for name, path in OUTPUT_DIRS.items():
    test_file = os.path.join(path, '.write_test')
    try:
        with open(test_file, 'w') as f:
            f.write('test')
        os.remove(test_file)
        print(f"   {name}: writable")
    except Exception as e:
        print(f"   {name}: ERROR - {e}")
        raise

print("\n All directories have write permissions")

In [ ]:
# UTILITY FUNCTIONS
from typing import Tuple, List, Dict

def score_pathways(adata, 
                  pathway_dict: Dict[str, List[str]] = None,
                  min_genes: int = None, 
                  use_raw: bool = False) -> Tuple[List[str], pd.DataFrame]:
    
    if pathway_dict is None:
        pathway_dict = CFG.METABOLIC_PATHWAYS
    if min_genes is None:
        min_genes = CFG.MIN_PATHWAY_GENES
    
    valid_pathways = []
    pathway_stats = []
    
    print(f"\n{'Pathway':<30s} | Total | Present | Coverage | Status")
    print(f"{'-'*75}")
    
    for pathway_name, gene_list in pathway_dict.items():
        genes_present = [g for g in gene_list if g in adata.var_names]
        n_genes = len(genes_present)
        coverage = 100 * n_genes / len(gene_list)
        
        status = " SCORED" if n_genes >= min_genes else "✗ SKIP"
        print(f"{pathway_name:<30s} | {len(gene_list):5d} | {n_genes:7d} | {coverage:7.1f}% | {status}")
        
        if n_genes >= min_genes:
            sc.tl.score_genes(
                adata, 
                gene_list=genes_present, 
                score_name=f'pathway_{pathway_name}', 
                use_raw=use_raw
            )
            valid_pathways.append(pathway_name)
            pathway_stats.append({
                'Pathway': pathway_name,
                'Total_Genes': len(gene_list),
                'Present_Genes': n_genes,
                'Coverage_Pct': coverage
            })
    
    summary_df = pd.DataFrame(pathway_stats)
    print(f"\n Successfully scored {len(valid_pathways)}/{len(pathway_dict)} pathways")
    print(f"  Mean coverage: {summary_df['Coverage_Pct'].mean():.1f}%")
    
    return valid_pathways, summary_df

def compute_auc_ci(y_true, y_pred, n_bootstraps=1000, ci=95):
    """Compute AUC with bootstrap confidence intervals."""
    from sklearn.metrics import roc_auc_score
    
    rng = np.random.RandomState(CFG.RANDOM_SEED)
    aucs = []
    n = len(y_true)
    
    for i in range(n_bootstraps):
        indices = rng.choice(n, size=n, replace=True)
        try:
            auc = roc_auc_score(y_true[indices], y_pred[indices])
            aucs.append(auc)
        except:
            continue
    
    aucs = np.array(aucs)
    mean_auc = np.mean(aucs)
    lower = np.percentile(aucs, (100 - ci) / 2)
    upper = np.percentile(aucs, 100 - (100 - ci) / 2)
    
    return mean_auc, lower, upper

def compute_morans_i(values: np.ndarray, W) -> float:
    """
    Compute Moran's I statistic for spatial autocorrelation (vectorized).
    
    Parameters
    ----------
    values : np.ndarray, shape (n_spots,)
        Variable values for each spatial location
    W : scipy.sparse matrix, shape (n_spots, n_spots)
        Spatial weights matrix
    
    Returns
    -------
    float
        Moran's I statistic
    """
    n = len(values)
    values_centered = values - np.mean(values)
    numerator = values_centered @ W @ values_centered
    denominator = np.sum(values_centered**2)
    W_sum = W.sum()
    I = (n / W_sum) * (numerator / denominator)
    return I


print(" Utility functions loaded")

### Deep Learning Model Architecture

#### Model Selection

We use a single canonical model architecture throughout this analysis: **MetabolicVulnerabilityNet**.

**Architecture Design:**
- **Type:** Feedforward neural network
- **Layers:** 3 hidden layers with progressive dimensionality reduction
- **Dimensions:** 64 → 32 → 16 neurons
- **Regularization:** Batch normalization + Dropout (0.3)
- **Activation:** ReLU (hidden layers), Sigmoid (output)

**Design Rationale:**
1. **Progressive reduction** (64→32→16) creates information bottleneck that forces learning of compressed metabolic representations
2. **Batch normalization** stabilizes training and improves convergence
3. **Dropout (p=0.3)** prevents overfitting on high-dimensional input
4. **Three layers** balance capacity and overfitting risk for our dataset size

In [ ]:
#  CANONICAL MODEL DEFINITION

class MetabolicVulnerabilityNet(nn.Module):
    
    VERSION = "1.0.0"
    
    def __init__(self, 
                 input_dim,
                 hidden_dim1=None,
                 hidden_dim2=None,
                 hidden_dim3=None,
                 dropout=None):
        super(MetabolicVulnerabilityNet, self).__init__()
        
        # Use Config values if not provided
        if hidden_dim1 is None:
            hidden_dim1 = getattr(CFG, 'HIDDEN_DIM1', 64)
        if hidden_dim2 is None:
            hidden_dim2 = getattr(CFG, 'HIDDEN_DIM2', 32)
        if hidden_dim3 is None:
            hidden_dim3 = getattr(CFG, 'HIDDEN_DIM3', 16)
        if dropout is None:
            dropout = getattr(CFG, 'DROPOUT', 0.3)
        
        # Validation
        assert input_dim > 0, "input_dim must be positive"
        assert hidden_dim1 > 0, "hidden_dim1 must be positive"
        assert hidden_dim2 > 0, "hidden_dim2 must be positive"
        assert hidden_dim3 > 0, "hidden_dim3 must be positive"
        assert 0 <= dropout < 1, "dropout must be in [0, 1)"
        
        # Build network
        self.network = nn.Sequential(
            # Layer 1
            nn.Linear(input_dim, hidden_dim1),
            nn.BatchNorm1d(hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Layer 2
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.BatchNorm1d(hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Layer 3
            nn.Linear(hidden_dim2, hidden_dim3),
            nn.BatchNorm1d(hidden_dim3),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Output layer
            nn.Linear(hidden_dim3, 1),
            nn.Sigmoid()
        )
        
        # Store architecture details
        self.input_dim = input_dim
        self.architecture = {
            'input_dim': input_dim,
            'hidden_dim1': hidden_dim1,
            'hidden_dim2': hidden_dim2,
            'hidden_dim3': hidden_dim3,
            'dropout': dropout,
            'version': self.VERSION
        }
    
    def forward(self, x):
        """
        Forward pass through the network.
        
        Parameters
        
        x : torch.Tensor
            Input tensor of shape (batch_size, input_dim)
        
        Returns
       
        torch.Tensor
            Output probabilities of shape (batch_size, 1)
        """
        return self.network(x)
    
    def get_num_parameters(self):
        """Count total trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)
    
    def __repr__(self):
        params = self.get_num_parameters()
        return (f"MetabolicVulnerabilityNet(\n"
                f"  input_dim={self.architecture['input_dim']},\n"
                f"  hidden=[{self.architecture['hidden_dim1']}, "
                f"{self.architecture['hidden_dim2']}, "
                f"{self.architecture['hidden_dim3']}],\n"
                f"  dropout={self.architecture['dropout']},\n"
                f"  version={self.VERSION},\n"
                f"  parameters={params:,}\n"
                f")")



# MODEL VERIFICATION
print("MODEL DEFINITION LOADED")

print(f"\nCanonical Model: MetabolicVulnerabilityNet v{MetabolicVulnerabilityNet.VERSION}")
print("\n  This is the ONLY model class used in this notebook.")
print("    All other model definitions should be deleted.")

# Test instantiation
print("Testing model instantiation")

test_input_dim = 50 
test_model = MetabolicVulnerabilityNet(input_dim=test_input_dim)

print(f"\n{test_model}")

# Test forward pass
print("\nTesting forward pass")
test_input = torch.randn(16, test_input_dim)
test_output = test_model(test_input)

assert test_output.shape == (16, 1), f"Output shape incorrect: {test_output.shape}"
assert (test_output >= 0).all() and (test_output <= 1).all(), "Output not in [0,1]"

print(f"   Input:  {test_input.shape}")
print(f"   Output: {test_output.shape}")
print(f"   Range:  [{test_output.min():.4f}, {test_output.max():.4f}]")

# Cleanup
del test_model, test_input, test_output


In [ ]:
os.environ["OMP_NUM_THREADS"]="1"
os.environ["OPENBLAS_NUM_THREADS"]="1"
os.environ["MKL_NUM_THREADS"]="1"
os.environ["NUMEXPR_NUM_THREADS"]="1"

In [ ]:
# Create output directories
output_dirs = [
    'figures',
    'results',
    'models',
    'data/processed'
]

for dir_path in output_dirs:
    Path(dir_path).mkdir(parents=True, exist_ok=True)
    
print("Output directories created successfully!")

In [ ]:
import shutil, os

src = "Patient_1/spatial/1142243F_spatial"
dst = "Patient_1/spatial"

for f in ['tissue_positions_list.csv', 'scalefactors_json.json', 
          'tissue_hires_image.png', 'tissue_lowres_image.png']:
    shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
    print(f"Copied {f}")

In [ ]:
# Loading 10x Visium data
print("Loading 10x Visium data")

# Load using scanpy
adata = sc.read_visium(
    path=CFG.DATA_DIR,
    count_file=CFG.H5_FILE,
    load_images=True,
    source_image_path=CFG.SPATIAL_DIR
)

# Make variable names unique
adata.var_names_make_unique()

# Convert to sparse if not already
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)

# Ensure float32 for memory efficiency
if adata.X.dtype != np.float32:
    adata.X = adata.X.astype(np.float32)

print(f"\nLoaded data: {adata.n_obs} spots × {adata.n_vars} genes")
print(f"Spatial coordinates available: {'spatial' in adata.obsm}")
print(f"Images loaded: {'images' in adata.uns}")
print(f"\nAnnData object:")
print(adata)

In [ ]:
if adata.n_vars > 30000:
    sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
# Extract and store spatial coordinates in obs for easy access
if 'spatial' in adata.obsm:
    adata.obs['x_coord'] = adata.obsm['spatial'][:, 0]
    adata.obs['y_coord'] = adata.obsm['spatial'][:, 1]
    print("Spatial coordinates added to adata.obs")
else:
    print("Warning: No spatial coordinates found in adata.obsm['spatial']")
    print("Available obsm keys:", list(adata.obsm.keys()))

In [ ]:
# Visualise the tissue
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot tissue image
sq.pl.spatial_scatter(adata, color=None, ax=axes[0], size=1.5)
axes[0].set_title('Tissue Structure', fontsize=14, fontweight='bold')

# Plot total counts
adata.obs['total_counts_temp'] = np.ravel(adata.X.sum(axis=1))
sq.pl.spatial_scatter(adata, color='total_counts_temp', ax=axes[1], size=1.5, cmap=CMAP_SEQUENTIAL)
axes[1].set_title('Total UMI Counts per Spot', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(Paths.FIGURES, '01_tissue_overview.png'), dpi=300, bbox_inches='tight')
plt.show()

print(" Initial tissue visualisation complete")

**We apply standard QC filters to remove low-quality spots:**

 **Criteria:**
- Minimum counts per spot: 500
- Minimum genes per spot: 250
- Maximum mitochondrial percentage: 20%
- Minimum spots per gene: 3

**Rationale:**
- Low-count spots may be empty or damaged tissue
- High MT% indicates dead/dying cells
- Rare genes (detected in <3 spots) are unreliable

In [ ]:
# Calculate QC Metrics
print(" Calculating QC metrics")

# Identify mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')

# Calculate QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt'],
    percent_top=None,
    log1p=False,
    inplace=True
)

print(f"   Total spots: {adata.n_obs:,}")
print(f"   Total genes: {adata.n_vars:,}")
print(f"   Median counts per spot: {np.median(adata.obs['total_counts']):.0f}")
print(f"   Median genes per spot: {np.median(adata.obs['n_genes_by_counts']):.0f}")
print(f"   Median MT%: {np.median(adata.obs['pct_counts_mt']):.2f}%")


# Visualise QC Metrics

print(" Visualising QC metrics")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Histogram: Total counts
axes[0, 0].hist(adata.obs['total_counts'], bins=50, edgecolor='black', 
                color=COLOR_PRIMARY, alpha=0.7)
axes[0, 0].axvline(CFG.MIN_COUNTS, color=COLOR_REFERENCE, linestyle='--', 
                    linewidth=2, label=f'Threshold: {CFG.MIN_COUNTS}')
axes[0, 0].set_xlabel('Total Counts', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Total Counts per Spot', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Histogram: N genes
axes[0, 1].hist(adata.obs['n_genes_by_counts'], bins=50, edgecolor='black', 
                color=COLOR_SECONDARY, alpha=0.7)
axes[0, 1].axvline(CFG.MIN_GENES, color=COLOR_REFERENCE, linestyle='--', 
                    linewidth=2, label=f'Threshold: {CFG.MIN_GENES}')
axes[0, 1].set_xlabel('Number of Genes', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Genes per Spot', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Histogram: MT%
axes[0, 2].hist(adata.obs['pct_counts_mt'], bins=50, edgecolor='black', 
                color=COLOR_TERTIARY, alpha=0.7)
axes[0, 2].axvline(CFG.MAX_MT_PCT, color=COLOR_REFERENCE, linestyle='--', 
                    linewidth=2, label=f'Threshold: {CFG.MAX_MT_PCT}%')
axes[0, 2].set_xlabel('Mitochondrial %', fontsize=12, fontweight='bold')
axes[0, 2].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0, 2].set_title('Mitochondrial Gene %', fontsize=13, fontweight='bold')
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

# Scatter: Counts vs Genes
axes[1, 0].scatter(adata.obs['total_counts'], adata.obs['n_genes_by_counts'], 
                   alpha=0.4, s=2, c=COLOR_PRIMARY, edgecolors='none')
axes[1, 0].set_xlabel('Total Counts', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('N Genes', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Counts vs Genes', fontsize=13, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Scatter: Counts vs MT%
axes[1, 1].scatter(adata.obs['total_counts'], adata.obs['pct_counts_mt'], 
                   alpha=0.4, s=2, c=COLOR_SECONDARY, edgecolors='none')
axes[1, 1].set_xlabel('Total Counts', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('MT %', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Counts vs MT%', fontsize=13, fontweight='bold')
axes[1, 1].grid(alpha=0.3)

# Scatter: Genes vs MT%
axes[1, 2].scatter(adata.obs['n_genes_by_counts'], adata.obs['pct_counts_mt'], 
                   alpha=0.4, s=2, c=COLOR_TERTIARY, edgecolors='none')
axes[1, 2].set_xlabel('N Genes', fontsize=12, fontweight='bold')
axes[1, 2].set_ylabel('MT %', fontsize=12, fontweight='bold')
axes[1, 2].set_title('Genes vs MT%', fontsize=13, fontweight='bold')
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(Paths.FIGURES, 'qc_metrics_before_filtering.png'), 
            dpi=300, bbox_inches='tight')
plt.show()
print("    Saved: figures/qc_metrics_before_filtering.png")

# Apply QC Filters

print(f"\n Applying QC filters")
print(f"   Thresholds:")
print(f"     - Min counts: {CFG.MIN_COUNTS}")
print(f"     - Min genes: {CFG.MIN_GENES}")
print(f"     - Max MT%: {CFG.MAX_MT_PCT}%")
print(f"     - Min cells per gene: {CFG.MIN_CELLS}")

n_spots_before = adata.n_obs
n_genes_before = adata.n_vars

# Filter spots
sc.pp.filter_cells(adata, min_genes=CFG.MIN_GENES)
sc.pp.filter_cells(adata, min_counts=CFG.MIN_COUNTS)
adata = adata[adata.obs['pct_counts_mt'] < CFG.MAX_MT_PCT, :].copy()

# Filter genes
sc.pp.filter_genes(adata, min_cells=CFG.MIN_CELLS)

n_spots_after = adata.n_obs
n_genes_after = adata.n_vars

print(f"\n   Filtering results:")
print(f"     Spots: {n_spots_before:,} → {n_spots_after:,} "
      f"(removed {n_spots_before - n_spots_after:,})")
print(f"     Genes: {n_genes_before:,} → {n_genes_after:,} "
      f"(removed {n_genes_before - n_genes_after:,})")

# Save Raw Counts

print(f"\n Saving raw counts for later use")
adata.layers['counts'] = adata.X.copy()
print("    Raw counts saved to adata.layers['counts']")

In [ ]:
# Normalise total counts per spot
print("Normalising data")
sc.pp.normalize_total(adata, target_sum=1e4)
print("Total count normalisation complete (target_sum=10,000)")

In [ ]:
# Log-transform
sc.pp.log1p(adata)
print("Log1p transformation complete")

In [ ]:
# FIGURE  — Pathway scoring robustness

from scipy.stats import spearmanr

CORE_METABOLIC_GENESETS = {
    'Glycolysis': ['HK1','HK2','GPI','PFKL','PFKM','ALDOA','TPI1','GAPDH','PGK1','PGAM1','ENO1','PKM','LDHA'],
    'TCA_Cycle': ['CS','ACO2','IDH1','IDH2','IDH3A','OGDH','SUCLA2','SDHA','FH','MDH1','MDH2'],
    'Oxidative_Phosphorylation': ['NDUFA1','NDUFB1','SDHB','UQCRC1','COX4I1','COX5A','ATP5F1A','ATP5F1B','ATP5MC1'],
    'Pentose_Phosphate': ['G6PD','PGLS','PGD','RPE','RPIA','TKT','TALDO1'],
    'Fatty_Acid_Synthesis': ['ACACA','FASN','SCD','ELOVL6','ACLY'],
    'Fatty_Acid_Oxidation': ['CPT1A','CPT2','ACOX1','HADHA','HADHB','ACADVL'],
    'Glutaminolysis': ['GLS','GLS2','GLUD1','GOT1','GOT2'],
    'One_Carbon_Metabolism': ['MTHFD1','MTHFD2','SHMT1','SHMT2','TYMS','DHFR'],
    'Amino_Acid_Metabolism': ['ASNS','PSAT1','PHGDH','PSPH','GPT2','BCAT1'],
}

robustness_results = []
for pathway_name, genes in CORE_METABOLIC_GENESETS.items():
    genes_present = [g for g in genes if g in adata.var_names]
    if len(genes_present) < 3:
        print(f"  Skipping {pathway_name}: only {len(genes_present)} genes present in adata")
        continue

    # Method 1: Scanpy's score_genes (background-corrected)
    sc.tl.score_genes(adata, gene_list=genes_present, score_name='_tmp_score', random_state=42)
    scanpy_score = adata.obs['_tmp_score'].values

    # Method 2: simple mean log-normalised expression
    expr = adata[:, genes_present].X
    if hasattr(expr, 'toarray'):
        expr = expr.toarray()
    mean_expr = np.asarray(expr).mean(axis=1).ravel()

    rho, pval = spearmanr(scanpy_score, mean_expr)
    robustness_results.append({'Pathway': pathway_name, 'rho': rho, 'p_value': pval, 'n_genes': len(genes_present)})

adata.obs.drop(columns=['_tmp_score'], inplace=True, errors='ignore')
robustness_df = pd.DataFrame(robustness_results).sort_values('rho')

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = [COLOR_TERTIARY if r >= 0.7 else COLOR_SECONDARY if r >= 0.5 else COLOR_REFERENCE
              for r in robustness_df['rho']]
ax.barh(robustness_df['Pathway'], robustness_df['rho'], color=bar_colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Spearman ρ (Scanpy score vs mean expression)', fontweight='bold')
ax.set_title('A. Robustness of Metabolic Pathway Scoring', fontweight='bold')
ax.set_xlim(0, 1)
ax.grid(alpha=0.3, axis='x')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=COLOR_TERTIARY, label='ρ ≥ 0.7 (high)'),
                    Patch(facecolor=COLOR_SECONDARY, label='ρ ≥ 0.5 (moderate)'),
                    Patch(facecolor=COLOR_REFERENCE, label='ρ < 0.5 (low)')],
          loc='lower right', fontsize=9)
plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/fig2_panelA_pathway_robustness.png', dpi=300, bbox_inches='tight')
plt.show()
print("  Saved: figures/fig2_panelA_pathway_robustness.png")
print(robustness_df.to_string(index=False))


In [ ]:
# DATA-DRIVEN PATHWAY DISCOVERY 

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

print("DATA-DRIVEN METABOLIC PATHWAY DISCOVERY")


# CONFIGURATION - Your GMT files

GMT_DIR = Path("msigdb/")

# Your downloaded files (exact filenames)
GMT_FILES = {
    "h.all.v2025.1.Hs.symbols.gmt": "Hallmark",
    "c2.cp.kegg_medicus.v2025.1.Hs.symbols.gmt": "KEGG",
    "c2.cp.reactome.v2025.1.Hs.symbols.gmt": "Reactome",
    "c2.cp.biocarta.v2025.1.Hs.symbols.gmt": "BioCarta",
}

# FUNCTION: Parse GMT file

def load_gmt(filepath):
    """
    Load a GMT file into a dictionary.
    
    GMT format (tab-separated):
        PATHWAY_NAME    DESCRIPTION    GENE1    GENE2    GENE3    ...
    
    Returns:
        dict: {pathway_name: [gene1, gene2, ...]}
    """
    pathways = {}
    
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            
            pathway_name = parts[0]
            genes = parts[2:]
            genes = [g.strip() for g in genes if g.strip()]
            
            if len(genes) > 0:
                pathways[pathway_name] = genes
    
    return pathways

# STEP 1: Load GMT Files

print("[Step 1] Loading GMT files\n")

if not GMT_DIR.exists():
    print(f"  Directory not found: {GMT_DIR}")
    print(f"  Please create the directory and add your GMT files")
    raise FileNotFoundError(f"GMT directory not found: {GMT_DIR}")

pathway_collections = []

for filename, collection_name in GMT_FILES.items():
    filepath = GMT_DIR / filename
    
    if filepath.exists():
        pathways = load_gmt(filepath)
        pathway_collections.append((collection_name, pathways))
        print(f"  {collection_name}: {len(pathways)} pathways loaded")
    else:
        print(f"  {collection_name}: File not found - {filename}")

if len(pathway_collections) == 0:
    print(f"\nNo GMT files found in {GMT_DIR}/")
    raise FileNotFoundError("No GMT files loaded")

total_pathways = sum(len(p[1]) for p in pathway_collections)
print(f"\nLoaded {total_pathways} pathways from {len(pathway_collections)} collections")

# STEP 2: Create Proliferation Target (if needed)

print(f"\n[Step 2] Checking proliferation target\n")

if 'target_proliferation_binary' not in adata.obs.columns:
    print("  Creating proliferation target...")
    
    prolif_genes_present = [g for g in CFG.PROLIFERATION_GENES if g in adata.var_names]
    
    if len(prolif_genes_present) == 0:
        raise ValueError("No proliferation genes found in data!")
    
    prolif_expr = adata[:, prolif_genes_present].X
    if hasattr(prolif_expr, 'toarray'):
        prolif_expr = prolif_expr.toarray()
    
    prolif_score = np.mean(prolif_expr, axis=1)
    threshold = np.percentile(prolif_score, 75)
    adata.obs['target_proliferation_binary'] = (prolif_score > threshold).astype(int)
    
    n_high = adata.obs['target_proliferation_binary'].sum()
    print(f"  Created from {len(prolif_genes_present)} genes (top 25% = high)")
    print(f"  High proliferation: {n_high} ({100*n_high/len(adata):.1f}%)")
    print(f"  Low proliferation: {len(adata)-n_high} ({100*(len(adata)-n_high)/len(adata):.1f}%)")
else:
    n_high = adata.obs['target_proliferation_binary'].sum()
    print(f"  Target already exists")
    print(f"  High: {n_high} ({100*n_high/len(adata):.1f}%) | Low: {len(adata)-n_high} ({100*(len(adata)-n_high)/len(adata):.1f}%)")

# STEP 3: Score All Pathways

print(f"\n[Step 3] Scoring pathways\n")

print(f"  Minimum genes required: {CFG.MIN_PATHWAY_GENES}")
print(f"  Scoring method: Mean expression\n")

pathway_metadata = []
pathway_scores_dict = {}

for collection_name, pathway_dict in pathway_collections:
    scored = 0
    skipped = 0
    
    for pathway_name, gene_list in pathway_dict.items():
        genes_present = [g for g in gene_list if g in adata.var_names]
        
        if len(genes_present) < CFG.MIN_PATHWAY_GENES:
            skipped += 1
            continue
        
        pathway_expr = adata[:, genes_present].X
        if hasattr(pathway_expr, 'toarray'):
            pathway_expr = pathway_expr.toarray()
        
        pathway_score = np.mean(pathway_expr, axis=1)
        
        col_name = f'{collection_name}_{pathway_name}'
        pathway_scores_dict[col_name] = pathway_score
        
        corr, pval = spearmanr(pathway_score, adata.obs['target_proliferation_binary'])
        
        pathway_metadata.append({
            'collection': collection_name,
            'pathway': pathway_name,
            'full_name': col_name,
            'n_genes_total': len(gene_list),
            'n_genes_present': len(genes_present),
            'coverage': len(genes_present) / len(gene_list),
            'mean_expression': np.mean(pathway_score),
            'variance': np.var(pathway_score),
            'correlation_with_target': corr,
            'pvalue': pval
        })
        
        scored += 1
    
    print(f"  {collection_name}: {scored} scored, {skipped} skipped")

print(f"\n  Adding {len(pathway_scores_dict)} pathway scores to adata.obs...")
pathway_scores_df = pd.DataFrame(pathway_scores_dict, index=adata.obs_names)
adata.obs = pd.concat([adata.obs, pathway_scores_df], axis=1)
print(f"  Done")

pathway_meta_df = pd.DataFrame(pathway_metadata)
print(f"\n  Total pathways scored: {len(pathway_meta_df)}")

# STEP 4: Filter Pathways

print(f"\n[Step 4] Filtering pathways\n")

initial_count = len(pathway_meta_df)

pathway_meta_df = pathway_meta_df[pathway_meta_df['coverage'] >= CFG.MIN_PATHWAY_COVERAGE].copy()
print(f"  After coverage filter (>={CFG.MIN_PATHWAY_COVERAGE*100:.0f}%): {len(pathway_meta_df)}/{initial_count}")

median_var = pathway_meta_df['variance'].median()
pathway_meta_df = pathway_meta_df[pathway_meta_df['variance'] > median_var].copy()
print(f"  After variance filter (>median): {len(pathway_meta_df)}/{initial_count}")

pathway_meta_df = pathway_meta_df[pathway_meta_df['mean_expression'] > 0].copy()
print(f"  After non-zero filter: {len(pathway_meta_df)}/{initial_count}")

# STEP 5: Statistical Testing and FDR Correction

print(f"\n[Step 5] Statistical testing\n")

reject, qvals, _, _ = multipletests(pathway_meta_df['pvalue'].values, method='fdr_bh', alpha=0.05)
pathway_meta_df['qvalue'] = qvals
pathway_meta_df['significant'] = reject

pathway_meta_df['abs_correlation'] = np.abs(pathway_meta_df['correlation_with_target'])
pathway_meta_df = pathway_meta_df.sort_values('abs_correlation', ascending=False).reset_index(drop=True)

n_sig = pathway_meta_df['significant'].sum()
n_nominal = (pathway_meta_df['pvalue'] < 0.05).sum()

print(f"  Significant (FDR q < 0.05): {n_sig}/{len(pathway_meta_df)}")
print(f"  Nominal (p < 0.05): {n_nominal}/{len(pathway_meta_df)}")

# STEP 6: Select Top Pathways

print(f"\n[Step 6] Selecting top {CFG.N_TOP_PATHWAYS} pathways\n")

top_pathways = pathway_meta_df.head(CFG.N_TOP_PATHWAYS).copy()

print(f"{'#':<4} {'Pathway':<55} {'Source':<10} {'r':<8} {'q-value':<12} {'Genes':<8} {'Sig'}")
print("-" * 105)

for i, (_, row) in enumerate(top_pathways.iterrows(), 1):
    sig = "***" if row['qvalue'] < 0.001 else "**" if row['qvalue'] < 0.01 else "*" if row['qvalue'] < 0.05 else ""
    name = row['pathway'][:53] + '..' if len(row['pathway']) > 53 else row['pathway']
    print(f"{i:<4} {name:<55} {row['collection']:<10} {row['correlation_with_target']:>7.3f} "
          f"{row['qvalue']:>11.2e} {row['n_genes_present']:>3}/{row['n_genes_total']:<4} {sig}")

print("-" * 105)
print("Legend: *** q<0.001, ** q<0.01, * q<0.05\n")

print("Collection breakdown in top pathways:")
for coll in top_pathways['collection'].unique():
    n = (top_pathways['collection'] == coll).sum()
    print(f"  {coll}: {n}")

# STEP 7: Create Feature Set

print(f"\n[Step 7] Creating feature set\n")

DATADRIVEN_PATHWAY_COLS = top_pathways['full_name'].tolist()
DATADRIVEN_COVARIATE_COLS = CFG.TECHNICAL_COVARIATES.copy()
DATADRIVEN_FEATURES = DATADRIVEN_PATHWAY_COLS + DATADRIVEN_COVARIATE_COLS

print(f"  Pathways: {len(DATADRIVEN_PATHWAY_COLS)}")
print(f"  Covariates: {len(DATADRIVEN_COVARIATE_COLS)} ({', '.join(DATADRIVEN_COVARIATE_COLS)})")
print(f"  Total features: {len(DATADRIVEN_FEATURES)}")

# STEP 8: Save Results

print(f"\n[Step 8] Saving results\n")

os.makedirs('results/tables', exist_ok=True)

pathway_meta_df.to_csv('results/tables/datadriven_all_pathways.csv', index=False)
print(f"  Saved: results/tables/datadriven_all_pathways.csv ({len(pathway_meta_df)} pathways)")

top_pathways.to_csv('results/tables/datadriven_selected_pathways.csv', index=False)
print(f"  Saved: results/tables/datadriven_selected_pathways.csv ({len(top_pathways)} pathways)")

# DONE

print(f"\nPATHWAY DISCOVERY COMPLETE")
print(f"\nCreated variables:")
print(f"  DATADRIVEN_PATHWAY_COLS   : {len(DATADRIVEN_PATHWAY_COLS)} pathways")
print(f"  DATADRIVEN_COVARIATE_COLS : {len(DATADRIVEN_COVARIATE_COLS)} covariates")  
print(f"  DATADRIVEN_FEATURES       : {len(DATADRIVEN_FEATURES)} total features")
print(f"\nPathway scores added to: adata.obs")
print(f"Metadata saved to: results/tables/")

In [ ]:
# Export ALL pathway scores BEFORE filtering

# Get all pathway score columns from adata.obs
all_pathway_cols = [col for col in adata.obs.columns
                    if any(col.startswith(prefix) for prefix in
                          ['Reactome_', 'GO_', 'KEGG_', 'MSigDB_'])]

print(f'Total pathways in adata.obs: {len(all_pathway_cols)}')

# Export with spot IDs as index
all_pathway_scores = adata.obs[all_pathway_cols].copy()
all_pathway_scores.index = adata.obs_names
all_pathway_scores.to_csv('Patient_1_ALL_pathway_scores_UNFILTERED.csv')

print(f'Exported {all_pathway_scores.shape[0]} spots × {all_pathway_scores.shape[1]} pathways')

# Show metabolic pathway examples
metabolic_examples = [c for c in all_pathway_cols if any(x in c.lower() 
                      for x in ['glycolysis', 'tca', 'fatty', 'oxidative', 'pentose', 'amino'])]
print(f'\nMetabolic pathway examples ({len(metabolic_examples)}):')
for p in metabolic_examples[:10]:
    print(f'  - {p}')

In [ ]:
# Export ONLY METABOLIC pathways

# Get all pathway score columns
all_pathway_cols = [col for col in adata.obs.columns
                    if any(col.startswith(prefix) for prefix in
                          ['Reactome_', 'GO_', 'KEGG_', 'MSigDB_'])]

print(f'Total pathways in adata.obs: {len(all_pathway_cols)}')

# Filter to METABOLIC pathways only
metabolic_keywords = [
    'glycolysis', 'gluconeogenesis',
    'tca', 'citrate', 'citric acid',
    'oxidative phosphorylation', 'electron transport', 'oxphos',
    'fatty acid', 'lipid',
    'pentose phosphate',
    'amino acid', 'glutamine', 'glutamate', 'branched',
    'pyruvate', 'lactate',
    'metabolic', 'metabolism',
    'biosynthesis', 'biosynthetic',
    'catabolic', 'catabolism',
    'oxidation', 'beta-oxidation'
]

metabolic_pathways = [col for col in all_pathway_cols 
                     if any(keyword in col.lower() for keyword in metabolic_keywords)]

print(f'\nMetabolic pathways found: {len(metabolic_pathways)}')
print(f'\nFirst 20 metabolic pathways:')
for i, p in enumerate(metabolic_pathways[:20], 1):
    print(f'  {i:2d}. {p}')

# Export metabolic pathways
metabolic_scores = adata.obs[metabolic_pathways].copy()
metabolic_scores.index = adata.obs_names
metabolic_scores.to_csv('Patient_1_METABOLIC_pathway_scores.csv')

print(f'\nExported {metabolic_scores.shape[0]} spots × {metabolic_scores.shape[1]} metabolic pathways')
print(f'Saved to: Patient_1_METABOLIC_pathway_scores.csv')

In [ ]:
# Export target variable for scFEA comparison

# Check if target exists
if 'target_proliferation_binary' in adata.obs.columns:
    print("Target variable found!")
    
    # Export target with spot IDs
    target_df = adata.obs[['target_proliferation_binary']].copy()
    target_df.index = adata.obs_names
    target_df.to_csv('Patient_1_target_proliferation.csv')
    
    print(f'Exported target for {len(target_df)} spots')
    print(f'Class distribution:')
    print(f'  Low proliferation (0): {(target_df["target_proliferation_binary"]==0).sum()} spots')
    print(f'  High proliferation (1): {(target_df["target_proliferation_binary"]==1).sum()} spots')
    print(f'  Balance: {target_df["target_proliferation_binary"].mean():.1%} high')
else:
    print("ERROR: target_proliferation_binary not found in adata.obs")
    print("Available columns with 'target' or 'prolif':")
    target_cols = [c for c in adata.obs.columns if 'target' in c.lower() or 'prolif' in c.lower()]
    for col in target_cols:
        print(f"  - {col}")

In [ ]:
# CHECK AND COMPUTE TECHNICAL COVARIATES

print("Checking technical covariates\n")

# Check what exists
for cov in CFG.TECHNICAL_COVARIATES:
    exists = cov in adata.obs.columns
    print(f"  {cov}: {' exists' if exists else ' missing'}")

print("\nComputing missing covariates\n")

# Compute n_counts if missing
if 'n_counts' not in adata.obs.columns:
    print("  Computing n_counts")
    if hasattr(adata.X, 'A1'):
        adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    else:
        adata.obs['n_counts'] = np.array(adata.X.sum(axis=1)).flatten()
    print("   n_counts computed")

# Compute pct_counts_mt if missing
if 'pct_counts_mt' not in adata.obs.columns:
    print("  Computing pct_counts_mt")
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    mt_counts = adata[:, adata.var['mt']].X.sum(axis=1)
    if hasattr(mt_counts, 'A1'):
        mt_counts = mt_counts.A1
    else:
        mt_counts = np.array(mt_counts).flatten()
    adata.obs['pct_counts_mt'] = (mt_counts / adata.obs['n_counts']) * 100
    print("   pct_counts_mt computed")

# Compute pct_counts_ribo if missing
if 'pct_counts_ribo' not in adata.obs.columns:
    print("  Computing pct_counts_ribo...")
    ribo_genes = [g for g in adata.var_names if g.startswith(('RPS', 'RPL'))]
    print(f"    Found {len(ribo_genes)} ribosomal genes")
    
    if len(ribo_genes) > 0:
        ribo_counts = adata[:, ribo_genes].X.sum(axis=1)
        if hasattr(ribo_counts, 'A1'):
            ribo_counts = ribo_counts.A1
        else:
            ribo_counts = np.array(ribo_counts).flatten()
        adata.obs['pct_counts_ribo'] = (ribo_counts / adata.obs['n_counts']) * 100
    else:
        print("     No ribosomal genes found, setting to 0")
        adata.obs['pct_counts_ribo'] = 0
    
    print("   pct_counts_ribo computed")

print("TECHNICAL COVARIATES SUMMARY")

for cov in CFG.TECHNICAL_COVARIATES:
    values = adata.obs[cov]
    print(f"\n{cov}:")
    print(f"  Mean: {values.mean():.2f}")
    print(f"  Std: {values.std():.2f}")
    print(f"  Min: {values.min():.2f}")
    print(f"  Max: {values.max():.2f}")

print("\n All technical covariates ready!\n")

# NOW CREATE FEATURE MATRIX
print("CREATING FEATURE MATRIX")

# Verify all features exist
print("Checking all features exist")
missing_features = []
for feat in DATADRIVEN_FEATURES:
    if feat not in adata.obs.columns:
        missing_features.append(feat)

if len(missing_features) > 0:
    print(f"\n ERROR: {len(missing_features)} features missing:")
    for feat in missing_features[:10]:  # Show first 10
        print(f"  - {feat}")
    raise KeyError(f"{len(missing_features)} features not found in adata.obs")

print(" All features present\n")

# Create feature matrix
X = adata.obs[DATADRIVEN_FEATURES].values
y = adata.obs['target_proliferation_binary'].values

print(f"Feature matrix created:")
print(f"  X shape: {X.shape}")
print(f"  y shape: {y.shape}")
print(f"  Class balance: {y.mean():.1%} high proliferation")
print(f"  Class 0 (low): {(y==0).sum()} spots")
print(f"  Class 1 (high): {(y==1).sum()} spots\n")

# Check for NaN or Inf
print("Data quality checks:")
print(f"  NaN values in X: {np.isnan(X).sum()}")
print(f"  Inf values in X: {np.isinf(X).sum()}")
print(f"  NaN values in y: {np.isnan(y).sum()}")

if np.isnan(X).sum() > 0 or np.isinf(X).sum() > 0:
    print("\n WARNING: Found NaN or Inf values")
    print("  Replacing with 0")
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    print("   Cleaned")

print("\n Feature matrix ready for modeling!\n")

# Show feature composition
print("Feature composition:")
print(f"  Pathway features: {len(DATADRIVEN_PATHWAY_COLS)}")
for i, pathway in enumerate(DATADRIVEN_PATHWAY_COLS[:5], 1):
    print(f"    {i}. {pathway}")
if len(DATADRIVEN_PATHWAY_COLS) > 5:
    print(f"     and {len(DATADRIVEN_PATHWAY_COLS)-5} more")

print(f"\n  Technical covariates: {len(DATADRIVEN_COVARIATE_COLS)}")
for cov in DATADRIVEN_COVARIATE_COLS:
    print(f"    - {cov}")

In [ ]:
# In your next cell:
X = adata.obs[DATADRIVEN_FEATURES].values
y = adata.obs['target_proliferation_binary'].values

print(f"Features: {X.shape}")
print(f"Target: {y.shape}")
print(f"Class balance: {y.mean():.1%} high proliferation")

**TARGET CREATION BEFORE FEATURE ENGINEERING**

This target is created using proliferation genes that are INDEPENDENT of metabolic pathway features.
The target is predefined (median split) rather than data-driven.

In [ ]:
# CREATE TARGET VARIABLE

print(" CREATING TARGET VARIABLE")

# Proliferation signature
proliferation_genes = ['MKI67', 'TOP2A', 'PCNA', 'CCNB1', 'CCNB2', 'CDK1', 'BIRC5']
genes_present = [g for g in proliferation_genes if g in adata.var_names]

print(f"\n  Proliferation genes:")
print(f"    Total: {len(proliferation_genes)}")
print(f"    Present: {len(genes_present)}")
print(f"    Genes: {genes_present}")

if len(genes_present) >= 3:
    sc.tl.score_genes(adata, gene_list=genes_present, 
                     score_name='proliferation_score', use_raw=False)
    
    # Binarize at median
    median_score = adata.obs['proliferation_score'].median()
    adata.obs['target_proliferation_binary'] = (
        adata.obs['proliferation_score'] > median_score
    ).astype(int)
    
    print(f"\n  Target created:")
    print(f"    High proliferation: {adata.obs['target_proliferation_binary'].sum()}")
    print(f"    Low proliferation: {(1-adata.obs['target_proliferation_binary']).sum()}")
else:
    print(f"\n  ERROR: Insufficient proliferation genes!")
    print(f"    Need at least 3, found {len(genes_present)}")

In [ ]:
# HVG SELECTION - CUSTOM MEMORY-SAFE METHOD

print(" HVG SELECTION (CUSTOM METHOD)")

print(f"\n[Info]")
print(f"  Data shape: {adata.n_obs} spots × {adata.n_vars} genes")
print(f"  Using custom variance-based selection")

# Calculate mean expression for each gene
print(f"\n[Calculating gene statistics]")
print(f"  Computing means")
if hasattr(adata.X, 'toarray'):
    # Sparse matrix
    means = np.array(adata.X.mean(axis=0)).flatten()
else:
    # Dense matrix
    means = adata.X.mean(axis=0)

# Calculate variance for each gene
print(f"  Computing variances")
if hasattr(adata.X, 'toarray'):
    # Sparse matrix
    variances = np.array(adata.X.power(2).mean(axis=0)).flatten() - means**2
else:
    # Dense matrix
    variances = ((adata.X - means)**2).mean(axis=0)

# Normalise variance (coefficient of variation)
print(f"  Computing normalised variance")
# Avoid division by zero
means_safe = np.where(means == 0, 1e-10, means)
variance_norm = variances / means_safe

# Store in adata.var
adata.var['means'] = means
adata.var['variances'] = variances
adata.var['variance_norm'] = variance_norm

# Select top N highly variable genes
print(f"\n[Selecting HVGs]")
n_top_genes = 3000  

# Rank by normalised variance
top_gene_indices = np.argsort(variance_norm)[-n_top_genes:]
top_gene_names = adata.var_names[top_gene_indices]

# Mark HVGs in adata.var
print(f"\n[Marking highly variable genes]")
adata.var['highly_variable'] = False  # Initialise all as False
adata.var.loc[top_gene_names, 'highly_variable'] = True  

# Verify
n_hvgs = adata.var['highly_variable'].sum()
print(f"   Selected {n_hvgs} highly variable genes")
print(f"   Percentage: {n_hvgs/adata.n_vars*100:.1f}%")
print(f"   Added 'highly_variable' column to adata.var")

# Show top 10 most variable genes
print(f"\n[Top 10 most variable genes]")
top_10 = adata.var.nlargest(10, 'variance_norm')
for i, (gene, row) in enumerate(top_10.iterrows(), 1):
    print(f"  {i:2d}. {gene:15s} (norm_var: {row['variance_norm']:.2f})")

# Sanity checks
assert n_hvgs > 0, "No HVGs selected!"
assert 'highly_variable' in adata.var.columns, "HVG column not created!"
assert adata.var['highly_variable'].sum() == n_top_genes, "Wrong number of HVGs!"

In [ ]:
# Calculate mean and variance manually (memory-safe)

print(f"\n[Calculating statistics]")

# Mean (memory-safe for sparse)
if sparse.issparse(adata.X):
    gene_means = np.array(adata.X.mean(axis=0)).flatten()
else:
    gene_means = np.mean(adata.X, axis=0)

print(f"  Computing gene variances")

# Variance (memory-safe for sparse)
if sparse.issparse(adata.X):
    # Variance = E[X^2] - E[X]^2
    gene_sq_means = np.array(adata.X.multiply(adata.X).mean(axis=0)).flatten()
    gene_vars = gene_sq_means - gene_means**2
else:
    gene_vars = np.var(adata.X, axis=0)

# Prevent division by zero
gene_vars = np.maximum(gene_vars, 1e-12)

gc.collect()

In [ ]:
# Normalised variance (coefficient of variation)

print(f"  Computing normalised variance")

# Coefficient of variation: std / mean
gene_cv = np.sqrt(gene_vars) / (gene_means + 1e-9)

# Alternative: mean-normalized variance
gene_norm_var = gene_vars / (gene_means + 1e-9)

gc.collect()

In [ ]:
# Select top N genes by normalised variance

print(f"\n[Selecting HVGs]")

n_top_genes = 3000

# Rank genes by normalized variance
gene_rankings = np.argsort(gene_norm_var)[::-1]  # Descending order
top_gene_indices = gene_rankings[:n_top_genes]

# Create highly_variable column
adata.var['highly_variable'] = False
adata.var.iloc[top_gene_indices, adata.var.columns.get_loc('highly_variable')] = True

# Store statistics
adata.var['means'] = gene_means
adata.var['variances'] = gene_vars
adata.var['variances_norm'] = gene_norm_var

print(f"  Selected {adata.var['highly_variable'].sum()} HVGs")
print(f"  Method: Normalised variance (custom)")

# Store metadata for compatibility
adata.uns['hvg'] = {
    'flavor': 'custom_normalized_variance',
    'params': {'n_top_genes': n_top_genes}
}

gc.collect()


In [ ]:
# Verification

print(f"\n[Verification]")
print(f"  Total genes: {adata.n_vars}")
print(f"  HVGs: {adata.var['highly_variable'].sum()}")
print(f"  Percentage: {100 * adata.var['highly_variable'].sum() / adata.n_vars:.1f}%")

# Show some HVG examples
hvg_names = adata.var_names[adata.var['highly_variable']][:10]
print(f"\n  Example HVGs: {', '.join(hvg_names)}")

# Plot variance vs mean
try:
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    
    # All genes
    ax.scatter(gene_means, gene_vars, alpha=0.3, s=1, c=COLOR_REFERENCE, label='All genes')
    
    # HVGs
    hvg_mask = adata.var['highly_variable'].values
    ax.scatter(gene_means[hvg_mask], gene_vars[hvg_mask], 
               alpha=0.6, s=3, c=COLOR_SIG, label='HVGs')
    
    ax.set_xlabel('Mean expression', fontsize=12, fontweight='bold')
    ax.set_ylabel('Variance', fontsize=12, fontweight='bold')
    ax.set_title('Gene Variance vs Mean', fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(Paths.FIGURES, 'hvg_selection_variance_vs_mean.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n  Saved: figures/hvg_selection_variance_vs_mean.png")
except Exception as e:
    print(f"\n   Could not create plot: {e}")

gc.collect()


In [ ]:
# SUBSET TO HIGHLY VARIABLE GENES

print("Subsetting to HVGs while preserving pathway scores\n")

# Save pathway scores and metadata BEFORE subsetting
print("Saving pathway scores and metadata")
pathway_cols_to_preserve = [c for c in adata.obs.columns 
                            if any(c.startswith(prefix) for prefix in ['Reactome_', 'GO_', 'KEGG_', 'MSigDB_'])]
covariate_cols_to_preserve = ['n_counts', 'pct_counts_mt', 'pct_counts_ribo', 
                              'target_proliferation_binary', 'spatial_block']

# Combine all columns to preserve
cols_to_preserve = pathway_cols_to_preserve + [c for c in covariate_cols_to_preserve if c in adata.obs.columns]

print(f"  Preserving {len(cols_to_preserve)} columns:")
print(f"    - {len(pathway_cols_to_preserve)} pathway scores")
print(f"    - {len(cols_to_preserve) - len(pathway_cols_to_preserve)} metadata columns")

# Save the columns
saved_obs_columns = adata.obs[cols_to_preserve].copy()

print(f"\nBefore subsetting:")
print(f"  Genes: {adata.n_vars}")
print(f"  Obs columns: {len(adata.obs.columns)}")

# Subset to HVGs
adata = adata[:, adata.var['highly_variable']].copy()

print(f"\nAfter subsetting:")
print(f"  Genes: {adata.n_vars}")
print(f"  Obs columns: {len(adata.obs.columns)}")

# Restore the saved columns
print(f"\nRestoring {len(saved_obs_columns.columns)} preserved columns...")
for col in saved_obs_columns.columns:
    adata.obs[col] = saved_obs_columns[col]

print(f"\nAfter restoration:")
print(f"  Obs columns: {len(adata.obs.columns)}")
print(f"   Pathway scores preserved!")

# Verify
pathway_check = [c for c in adata.obs.columns if c.startswith('Reactome_')]
print(f"\nVerification:")
print(f"  Reactome pathways in adata.obs: {len(pathway_check)}")
print(f"  Technical covariates present: {all(c in adata.obs.columns for c in ['n_counts', 'pct_counts_mt', 'pct_counts_ribo'])}")

In [ ]:
# DIMENSIONALITY REDUCTION
print(" DIMENSIONALITY REDUCTION")


# PCA 
print(f"\nPrincipal Component Analysis")

# Memory optimisation
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X, dtype=np.float32)
elif adata.X.dtype != np.float32:
    adata.X = adata.X.astype(np.float32)

gc.collect()

# Get HVG subset (manually)
hvg_mask = adata.var['highly_variable'].values
print(f"  HVGs: {hvg_mask.sum()}")
print(f"  Spots: {adata.n_obs}")

# Create temporary AnnData with HVGs only
adata_hvg = adata[:, hvg_mask].copy()

print(f"  Computing PCA on HVGs")

sc.tl.pca(
    adata_hvg,
    n_comps=50,
    svd_solver='randomized',  
    zero_center=False,         
    random_state=42
)

print(f"  PCA complete: 50 components")

# Transfer PCA results to main adata
adata.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata.uns['pca'] = adata_hvg.uns['pca']
adata.varm['PCs'] = np.zeros((adata.n_vars, 50))
adata.varm['PCs'][hvg_mask, :] = adata_hvg.varm['PCs']

# Clean up
del adata_hvg
gc.collect()

# Variance explained
variance_ratio = adata.uns['pca']['variance_ratio']
cumsum_variance = np.cumsum(variance_ratio)

# Plot variance explained
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, 51), variance_ratio, marker='o', color=COLOR_PRIMARY, linewidth=2, markersize=4)
axes[0].set_xlabel('Principal Component', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Variance Explained', fontsize=12, fontweight='bold')
axes[0].set_title('PCA: Variance per Component', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, 51), cumsum_variance, marker='o', color=COLOR_SECONDARY, linewidth=2, markersize=4)
axes[1].axhline(y=0.8, color=COLOR_REFERENCE, linestyle='--', label='80% variance')
axes[1].axhline(y=0.9, color=COLOR_REFERENCE, linestyle=':', label='90% variance')
axes[1].set_xlabel('Number of Components', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Cumulative Variance', fontsize=12, fontweight='bold')
axes[1].set_title('Cumulative Variance Explained', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(Paths.FIGURES, 'pca_variance_explained.png'), dpi=300, bbox_inches='tight')
plt.show()

# Determine optimal number of PCs
n_pcs_80 = np.argmax(cumsum_variance >= 0.8) + 1
n_pcs_90 = np.argmax(cumsum_variance >= 0.9) + 1

print(f"\n  PCs for 80% variance: {n_pcs_80}")
print(f"  PCs for 90% variance: {n_pcs_90}")
print(f"  Using {min(30, n_pcs_80)} PCs for downstream analysis")

n_pcs_use = min(30, n_pcs_80)

gc.collect()

In [ ]:
#  Neighbours (MANUAL - MEMORY-SAFE)

print(f"\n Computing neighbour graph (manual method)")


# PRE-NEIGHBOURS VERIFICATION

print(f"\n  [Verification]")

# Check dataset size
print(f"    Dataset: {adata.n_obs} spots × {adata.n_vars} genes")

# Verify PCA exists
if 'X_pca' not in adata.obsm:
    raise ValueError("No PCA embeddings! Run PCA first!")

pca = adata.obsm['X_pca']
print(f"    PCA shape: {pca.shape}")

# Check for bad values
has_nan = np.any(np.isnan(pca))
has_inf = np.any(np.isinf(pca))

if has_nan:
    print(f"    Fixing NaN values")
    adata.obsm['X_pca'] = np.nan_to_num(pca, nan=0.0)

if has_inf:
    print(f"    Fixing Inf values")
    adata.obsm['X_pca'] = np.nan_to_num(adata.obsm['X_pca'], 
                                        posinf=1e10, neginf=-1e10)

# Clean old neighbours data
if 'neighbors' in adata.uns:
    del adata.uns['neighbors']
if 'connectivities' in adata.obsp:
    del adata.obsp['connectivities']
if 'distances' in adata.obsp:
    del adata.obsp['distances']

print(f"    Verification complete")


# MANUAL NEIGHBOURS COMPUTATION

print(f"\n  [Manual computation]")

# Clean memory
gc.collect()

# Get PCA embeddings
X_pca = adata.obsm['X_pca']
n_obs = adata.n_obs

# Determine parameters based on dataset size
if n_obs < 1000:
    n_neighbors = 15
    n_pcs = min(25, X_pca.shape[1])
elif n_obs < 5000:
    n_neighbors = 10
    n_pcs = min(20, X_pca.shape[1])
else:
    n_neighbors = 5
    n_pcs = min(15, X_pca.shape[1])

print(f"    Parameters: k={n_neighbors}, n_pcs={n_pcs}")

# Extract PCA subset
X_pca_subset = X_pca[:, :n_pcs].astype(np.float32)

print(f"    Computing nearest neighbours")

# Compute neighbors with sklearn
nn = NearestNeighbors(
    n_neighbors=n_neighbors,
    metric='euclidean',
    algorithm='auto',
    n_jobs=1  # Single thread to avoid memory issues
)

nn.fit(X_pca_subset)
distances, indices = nn.kneighbors(X_pca_subset)

print(f"     Neighbours computed")

# Build sparse matrices
print(f"    Building connectivity matrices")

rows = []
cols = []
data_dist = []
data_conn = []

for i in range(n_obs):
    if i % 1000 == 0 and i > 0:
        print(f"      {i}/{n_obs} spots")
    
    for j, (neighbor_idx, dist) in enumerate(zip(indices[i], distances[i])):
        rows.append(i)
        cols.append(neighbor_idx)
        data_dist.append(dist)
        
        # Gaussian kernel connectivity
        sigma = distances[i][min(n_neighbors-1, len(distances[i])-1)]
        if sigma > 0:
            connectivity = np.exp(-dist**2 / (2 * sigma**2))
        else:
            connectivity = 1.0 if i == neighbor_idx else 0.0
        
        data_conn.append(connectivity)

# Create sparse matrices
distance_matrix = coo_matrix(
    (data_dist, (rows, cols)),
    shape=(n_obs, n_obs)
).tocsr()

connectivity_matrix = coo_matrix(
    (data_conn, (rows, cols)),
    shape=(n_obs, n_obs)
).tocsr()

print(f"    Matrices created ({distance_matrix.nnz:,} edges)")

# Add to adata
adata.obsp['distances'] = distance_matrix
adata.obsp['connectivities'] = connectivity_matrix

adata.uns['neighbors'] = {
    'connectivities_key': 'connectivities',
    'distances_key': 'distances',
    'params': {
        'n_neighbors': n_neighbors,
        'method': 'manual_sklearn',
        'metric': 'euclidean',
        'n_pcs': n_pcs,
        'use_rep': 'X_pca',
        'random_state': 42
    }
}

print(f"    Added to adata")

# Cleanup
del X_pca_subset, nn, distances, indices
del rows, cols, data_dist, data_conn
del distance_matrix, connectivity_matrix
gc.collect()

print(f"   Neighbour graph computed (k={n_neighbors}, {n_pcs} PCs)")

In [ ]:
# UMAP & LEIDEN

print("UMAP EMBEDDING & LEIDEN CLUSTERING")

# Verify neighbors from manual computation
assert 'connectivities' in adata.obsp, "Neighbors not computed!"
assert 'distances' in adata.obsp, "Distances not computed!"
print(" Neighbors graph verified from manual computation")

# UMAP
print("\nComputing UMAP embedding")
sc.tl.umap(
    adata,
    min_dist=0.3,
    spread=1.0,
    random_state=42
)
print(f"   UMAP: {adata.obsm['X_umap'].shape}")

# Leiden
print("\nPerforming Leiden clustering")
sc.tl.leiden(
    adata,
    resolution=0.8,
    random_state=42,
    key_added='leiden'
)
print(f"   Leiden: {adata.obs['leiden'].nunique()} clusters")
print(f"  Cluster sizes: {adata.obs['leiden'].value_counts().to_dict()}")

# Verify
assert 'X_umap' in adata.obsm, "UMAP failed!"
assert 'leiden' in adata.obs, "Leiden failed!"

In [ ]:
# Summary
print(f"DIMENSIONALITY REDUCTION COMPLETE")


print(f"\n[Results]")
print(f"  PCA: {adata.obsm['X_pca'].shape}")
print(f"  Neighbors: k={n_neighbors}, {n_pcs} PCs")
print(f"  UMAP: {adata.obsm['X_umap'].shape}")

# Calculate n_clusters from Leiden results
n_clusters = adata.obs['leiden'].nunique()
print(f"  Leiden: {n_clusters} clusters")

In [ ]:
# UMAP VISUALISATION WITH CLUSTERING AND PATHWAYS
print("UMAP VISUALISATION")


# Check if UMAP exists
if 'X_umap' not in adata.obsm.keys():
    print("Computing UMAP")
    sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca')
    sc.tl.umap(adata)
    print(" UMAP computed\n")

# Check if leiden clustering exists
if 'leiden' not in adata.obs.columns:
    print("Computing Leiden clustering...")
    sc.tl.leiden(adata, resolution=0.5)
    print(" Leiden clustering computed\n")

# Create figure
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

print("Creating visualisations\n")

# Panel A: Leiden Clusters
print("  Panel A: Leiden Clusters")
sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, 
           title='Leiden Clusters', legend_loc='right margin',
           frameon=True, size=30, palette=CLUSTER_COLORS)

# Panel B: Proliferation
print("  Panel B: Proliferation Score")
if 'target_proliferation_binary' in adata.obs.columns:
    sc.pl.umap(adata, color='target_proliferation_binary', ax=axes[1], show=False,
               title='Proliferation (Binary)', cmap=CMAP_DIVERGING,
               frameon=True, size=30, colorbar_loc='right')
elif 'target_proliferation' in adata.obs.columns:
    sc.pl.umap(adata, color='target_proliferation', ax=axes[1], show=False,
               title='Proliferation Score', cmap=CMAP_DIVERGING,
               frameon=True, size=30, colorbar_loc='right')
else:
    axes[1].text(0.5, 0.5, 'No proliferation score found', 
                ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Proliferation')

# Panel C: Top Pathway or Cell Cycle Score
print("  Panel C: Pathway Activity")

# Try to find a good pathway to visualize
pathway_to_plot = None

# Use the top pathway
if 'DATADRIVEN_PATHWAY_COLS' in globals() and len(DATADRIVEN_PATHWAY_COLS) > 0:
    # Use the first (top) pathway
    pathway_to_plot = DATADRIVEN_PATHWAY_COLS[0]
    pathway_display_name = pathway_to_plot.split('_', 1)[1][:40] if '_' in pathway_to_plot else pathway_to_plot[:40]
    
# Look for any cell cycle related pathway
elif any('CELL_CYCLE' in col.upper() or 'MITOTIC' in col.upper() or 'G2M' in col.upper() 
         for col in adata.obs.columns):
    # Find first cell cycle pathway
    for col in adata.obs.columns:
        if any(keyword in col.upper() for keyword in ['CELL_CYCLE', 'MITOTIC', 'G2M', 'PROLIFERATION']):
            pathway_to_plot = col
            pathway_display_name = col.split('_', 1)[1][:40] if '_' in col else col[:40]
            break

# Compute a simple cell cycle score
if pathway_to_plot is None:
    print("    Computing cell cycle score from proliferation genes...")
    
    prolif_genes_present = [g for g in CFG.PROLIFERATION_GENES if g in adata.var_names]
    
    if len(prolif_genes_present) > 0:
        prolif_expr = adata[:, prolif_genes_present].X
        if hasattr(prolif_expr, 'toarray'):
            prolif_expr = prolif_expr.toarray()
        adata.obs['cell_cycle_score'] = np.mean(prolif_expr, axis=1)
        pathway_to_plot = 'cell_cycle_score'
        pathway_display_name = f'Cell Cycle Score ({len(prolif_genes_present)} genes)'
    else:
        pathway_to_plot = None

# Plot the pathway
if pathway_to_plot is not None and pathway_to_plot in adata.obs.columns:
    sc.pl.umap(adata, color=pathway_to_plot, ax=axes[2], show=False,
               title=f'Pathway: {pathway_display_name}', 
               cmap=CMAP_DIVERGING, frameon=True, size=30, colorbar_loc='right')
    print(f"     Plotted: {pathway_display_name}")
else:
    # Fallback: show total counts
    if 'n_counts' in adata.obs.columns:
        sc.pl.umap(adata, color='n_counts', ax=axes[2], show=False,
                   title='Total UMI Counts', cmap=CMAP_SEQUENTIAL,
                   frameon=True, size=30, colorbar_loc='right')
        print(f"     Plotted: Total UMI Counts (fallback)")
    else:
        axes[2].text(0.5, 0.5, 'No pathways available\nRun pathway discovery first', 
                    ha='center', va='center', transform=axes[2].transAxes,
                    fontsize=12)
        axes[2].set_title('Pathway Activity')
        print(f"     No pathways to plot")

plt.tight_layout()

# Save figure
os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/umap_clustering_overview.png', dpi=300, bbox_inches='tight')
print(f"\n Saved: results/figures/umap_clustering_overview.png")

plt.show()

# Print cluster summary
print("CLUSTER SUMMARY")

print(f"Total spots: {adata.n_obs}")
print(f"Total clusters: {adata.obs['leiden'].nunique()}\n")

print("Cluster sizes:")
cluster_counts = adata.obs['leiden'].value_counts().sort_index()
for cluster, count in cluster_counts.items():
    pct = 100 * count / adata.n_obs
    print(f"  Cluster {cluster:>2s}: {count:>4d} spots ({pct:>5.1f}%)")

# If proliferation exists, show enrichment per cluster
if 'target_proliferation_binary' in adata.obs.columns:
    print("\nProliferation enrichment by cluster:")
    for cluster in sorted(adata.obs['leiden'].unique()):
        cluster_mask = adata.obs['leiden'] == cluster
        prolif_pct = adata.obs[cluster_mask]['target_proliferation_binary'].mean() * 100
        print(f"  Cluster {cluster:>2s}: {prolif_pct:>5.1f}% high proliferation")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Clusters
clusters = sorted(adata.obs['leiden'].unique(), key=lambda x: int(x))
colors = CLUSTER_COLORS
for i, c in enumerate(clusters):
    mask = adata.obs['leiden'] == c
    axes[0].scatter(adata.obsm['X_umap'][mask, 0], adata.obsm['X_umap'][mask, 1],
                    c=colors[i % len(colors)], s=3, alpha=0.7, label=c)
ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
axes[0].set_title('UMAP by Leiden Clusters', fontweight='bold')
axes[0].legend(ncol=2, fontsize=7, markerscale=2)

# Right: Proliferation  
adata.obs['prolif_label'] = adata.obs['target_proliferation_binary'].map({0:'Low', 1:'High'})
for label, color in PROLIF_COLORS.items():
    mask = adata.obs['prolif_label'] == label
    axes[1].scatter(adata.obsm['X_umap'][mask, 0], adata.obsm['X_umap'][mask, 1],
                    c=color, s=3, alpha=0.7, label=label)
ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
axes[1].set_title('UMAP by Proliferation', fontweight='bold')
axes[1].legend(fontsize=9, markerscale=3)

plt.tight_layout()
plt.savefig('figures/panel_F_umap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cluster colours (matching your Panel E spatial network)
CLUSTER_COLORS = PALETTE  # defined in the global plot-style cell

# Proliferation colours
PROLIF_COLORS = {'Low': COLOR_LOW, 'High': COLOR_HIGH}  # defined in the global plot-style cell

# Create proliferation label if not exists
if 'proliferation_label' not in adata.obs.columns:
    if 'target_proliferation_binary' in adata.obs.columns:
        adata.obs['proliferation_label'] = adata.obs['target_proliferation_binary'].map({0: 'Low', 1: 'High'})
    elif 'y' in adata.obs.columns:
        adata.obs['proliferation_label'] = adata.obs['y'].map({0: 'Low', 1: 'High'})
    elif 'target_proliferation' in adata.obs.columns:
        threshold = np.percentile(adata.obs['target_proliferation'], 75)
        adata.obs['proliferation_label'] = ['High' if x > threshold else 'Low' 
                                             for x in adata.obs['target_proliferation']]
    print(f"Created proliferation_label from available data")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for label in ['Low', 'High']:
    mask = adata.obs['proliferation_label'] == label
    ax.scatter(adata.obsm['X_umap'][mask, 0], adata.obsm['X_umap'][mask, 1],
               c=PROLIF_COLORS[label], s=3, alpha=0.7, label=label, rasterized=True)
ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
ax.set_title('UMAP by Proliferation', fontweight='bold')
ax.legend(loc='upper right', fontsize=10, markerscale=3, frameon=True)
ax.set_aspect('equal')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('figures/umap_proliferation_only.png', dpi=300, bbox_inches='tight')
plt.close()
print("  Saved: figures/umap_proliferation_only.png")

### Figure 2 — rebuilt (3 panels only)

The original 6-panel Figure 2 included three preprocessing/QC panels (tissue + UMI counts,
QC-metric distributions, HVG selection) that have been removed from the manuscript figure.
The three remaining panels are generated in two places in this notebook:

- **Panel A (pathway-scoring robustness)** runs earlier, right after log-normalisation and
  *before* HVG selection/subsetting — it needs the full gene set, not just the 3,000 HVGs,
  so it cannot live down here.
- **Panels B and C (spatial neighbourhood graph + cluster sizes, and UMAP by proliferation)**
  are generated below, since they need the Leiden clusters and UMAP embedding computed
  earlier in this section.

The composite cell below stitches all three saved panel images into a single replacement
figure, `figures/Fig2_composite.png`, using the uniform colour palette defined earlier in
this notebook. Run the whole notebook top-to-bottom (or at least: log-transform → Panel A →
... → UMAP/Leiden → Panel B → composite) so all three panel PNGs exist before the composite
cell runs.


In [ ]:
# FIGURE — Spatial neighbourhood graph & cluster-size distribution

print("FIGURE 2 PANEL B: SPATIAL NEIGHBOURHOOD GRAPH & CLUSTER SIZES")

from scipy.spatial import Delaunay

coords = adata.obsm['spatial']
leiden_labels = adata.obs['leiden'].astype(int).values
n_clusters = adata.obs['leiden'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: spatial neighbourhood network (Delaunay edges, coloured by cluster)
tri = Delaunay(coords)
edges = set()
for simplex in tri.simplices:
    for a, b in [(0, 1), (1, 2), (0, 2)]:
        i, j = simplex[a], simplex[b]
        edges.add((min(i, j), max(i, j)))

edge_list = list(edges)
if len(edge_list) > 20000:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(edge_list), 20000, replace=False)
    edge_list = [edge_list[k] for k in idx]

for i, j in edge_list:
    axes[0].plot(coords[[i, j], 0], coords[[i, j], 1], color=COLOR_REFERENCE,
                 linewidth=0.15, alpha=0.25, zorder=1)

axes[0].scatter(coords[:, 0], coords[:, 1], c=[cat_color(c) for c in leiden_labels],
                 s=6, alpha=0.9, zorder=2, edgecolors='none')
axes[0].set_title('B. Spatial Neighbourhood Network', fontweight='bold')
axes[0].set_xlabel('Spatial X'); axes[0].set_ylabel('Spatial Y')
axes[0].invert_yaxis()
axes[0].set_aspect('equal')

# Right: Leiden cluster-size distribution
cluster_sizes = adata.obs['leiden'].value_counts().sort_index()
axes[1].bar([str(c) for c in cluster_sizes.index], cluster_sizes.values,
            color=[cat_color(int(c)) for c in cluster_sizes.index],
            edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Leiden Cluster', fontweight='bold')
axes[1].set_ylabel('Number of Spots', fontweight='bold')
axes[1].set_title(f'Cluster-Size Distribution ({n_clusters} clusters)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/fig2_panelB_spatial_neighbourhood.png', dpi=300, bbox_inches='tight')
plt.show()
print("  Saved: figures/fig2_panelB_spatial_neighbourhood.png")


In [ ]:
# FIGURE  — Composite (3 panels: A = pathway robustness,

from PIL import Image, ImageDraw, ImageFont

panel_paths = [
    ('figures/fig2_panelA_pathway_robustness.png', 'A'),
    ('figures/fig2_panelB_spatial_neighbourhood.png', 'B'),
    ('figures/umap_proliferation_only.png', 'C'),
]

missing = [p for p, _ in panel_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        'Missing panel image(s): ' + ', '.join(missing) +
        '. Make sure you have run the Panel A cell (right after log-transform) '
        'and the Panel B cell (after UMAP/Leiden) earlier in this notebook first.'
    )

imgs = [Image.open(p).convert('RGB') for p, _ in panel_paths]
target_h = min(im.height for im in imgs)
resized = [im.resize((int(im.width * target_h / im.height), target_h)) for im in imgs]

pad = 20
total_w = sum(im.width for im in resized) + pad * (len(resized) + 1)
total_h = target_h + pad * 2 + 40
composite = Image.new('RGB', (total_w, total_h), 'white')
draw = ImageDraw.Draw(composite)

try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 28)
except Exception:
    font = ImageFont.load_default()

x = pad
for im, (_, label) in zip(resized, panel_paths):
    draw.text((x, 5), label, fill='black', font=font)
    composite.paste(im, (x, pad + 40))
    x += im.width + pad

os.makedirs('figures', exist_ok=True)
composite.save('figures/Fig2_composite.png', dpi=(300, 300))
print("Saved: figures/Fig2_composite.png  (A=pathway robustness, "
      "B=spatial neighbourhood/cluster size, C=UMAP by proliferation)")
composite


In [ ]:
# PATHWAY VERIFICATION 
print("PATHWAY VERIFICATION")

print(f"\n[Checking pathway scores in adata.obs]")
print(f"  Current genes in adata: {adata.n_vars} (HVG subset)")
print(f"  Original genes in adata.raw: {adata.raw.n_vars if adata.raw else 'None'}")

# Find data-driven pathway columns in adata.obs
pathway_cols_found = [col for col in adata.obs.columns if col in DATADRIVEN_PATHWAY_COLS]

print(f"\n[Data-Driven Pathway Scores Present]")
print(f"  Expected: {len(DATADRIVEN_PATHWAY_COLS)} pathways")
print(f"  Found: {len(pathway_cols_found)} pathway score columns")

if len(pathway_cols_found) > 0:
    print(f"\n Pathway scores preserved after HVG subsetting")
    f
    # Show statistics for each pathway
    print(f"\n[Pathway Score Statistics]")
    print(f"{'Pathway':<50} | {'Mean':>8} | {'Std':>8} | {'Min':>8} | {'Max':>8}")
    
    for col in sorted(pathway_cols_found):
        # Clean pathway name for display
        pathway_name = col.split('_', 1)[1] if '_' in col else col
        pathway_name = pathway_name[:48] if len(pathway_name) > 48 else pathway_name
        
        scores = adata.obs[col]
        print(f"{pathway_name:<50} | {scores.mean():>8.3f} | {scores.std():>8.3f} | {scores.min():>8.3f} | {scores.max():>8.3f}")
    
    # Verify scores are valid (not all zeros or NaN)
    valid_pathways = []
    invalid_pathways = []
    
    for col in pathway_cols_found:
        if adata.obs[col].notna().sum() > 0 and adata.obs[col].std() > 0:
            valid_pathways.append(col)
        else:
            invalid_pathways.append(col)
    
    print(f"\n[Validation]")
    print(f"  Total pathway columns: {len(pathway_cols_found)}")
    print(f"  Valid pathways (non-zero variance): {len(valid_pathways)}")
    
    if len(invalid_pathways) > 0:
        print(f"   Invalid pathways (zero variance or all NaN): {len(invalid_pathways)}")
        for pathway in invalid_pathways:
            print(f"     {pathway}")
    
    # Check technical covariates
    print(f"\n[Technical Covariates]")
    for cov in DATADRIVEN_COVARIATE_COLS:
        if cov in adata.obs.columns:
            values = adata.obs[cov]
            print(f"   {cov}: Mean={values.mean():.2f}, Std={values.std():.2f}")
        else:
            print(f"   {cov}: NOT FOUND")
    
    # Check target variable
    print(f"\n[Target Variable]")
    if 'target_proliferation_binary' in adata.obs.columns:
        target = adata.obs['target_proliferation_binary']
        print(f"   target_proliferation_binary: {target.sum()} high ({100*target.mean():.1f}%), {(1-target).sum()} low ({100*(1-target.mean()):.1f}%)")
    else:
        print(f"   target_proliferation_binary: NOT FOUND")

    print(" VERIFICATION SUCCESSFUL")
    print(f"\n   Data-driven pathway scores preserved: {len(valid_pathways)}/{len(DATADRIVEN_PATHWAY_COLS)}")
    print(f"   All pathways cell cycle related (FDR q < 0.05)")
    print(f"   Ready for downstream analysis")
    
else:
    print(" WARNING: No pathway scores found!")
    print(f"\n  Expected {len(DATADRIVEN_PATHWAY_COLS)} data-driven pathway columns")
    print(f"  Check that Cell 22 (pathway discovery) ran successfully")
    
    # Diagnostic info
    print(f"\n[Diagnostic Information]")
    print(f"  Columns starting with 'Reactome_': {len([c for c in adata.obs.columns if c.startswith('Reactome_')])}")
    print(f"  Columns starting with 'GO_': {len([c for c in adata.obs.columns if c.startswith('GO_')])}")
    print(f"  Columns starting with 'KEGG_': {len([c for c in adata.obs.columns if c.startswith('KEGG_')])}")
    print(f"  Total columns in adata.obs: {len(adata.obs.columns)}")
    
    # Show first 10 columns
    print(f"\n  First 10 columns in adata.obs:")
    for i, col in enumerate(list(adata.obs.columns)[:10], 1):
        print(f"    {i}. {col}")

Our metabolic pathway analysis has been validated through multiple approaches:

**1. ABLATION ANALYSIS** 
   - Quantifies pathway contribution to proliferation prediction
   - Significant pathways identified (q < 0.05)
   - Primary evidence for pathway importance

**2. NEGATIVE CONTROLS** 
   - Label permutation: Real AUC significantly > null (p < 0.001)
   - Pathway score permutation: Tests spatial structure importance
   - Validates that model learns biological signal, not noise

**3. CROSS-PATIENT VALIDATION** 
   - Tested on 3 independent patients
   - Model trained on Patient 1 generalizes to new patients
   - Demonstrates biological reproducibility

**4. SPATIAL STATISTICS** 
   - Moran's I shows significant spatial autocorrelation
   - Pathway activities are spatially organized
   - Not random technical artifacts

**5. PATHWAY SCORING ROBUSTNESS** 
   - High correlation between scoring methods
   - Results stable across methodologies
   - Biological signal is robust

In [ ]:
# DATA PREPARATION FOR MODELING 
print("DATA PREPARATION FOR MODELING")

# USE DATA-DRIVEN FEATURES FROM PATHWAY DISCOVERY

print(" Loading data-driven features\n")

# Use features selected by pathway discovery
PRIMARY_FEATURES = DATADRIVEN_FEATURES.copy()
PRIMARY_PATHWAY_COLS = DATADRIVEN_PATHWAY_COLS.copy()
PRIMARY_COVARIATE_COLS = DATADRIVEN_COVARIATE_COLS.copy()

print(f"Feature composition:")
print(f"  Total features: {len(PRIMARY_FEATURES)}")
print(f"   Pathway features: {len(PRIMARY_PATHWAY_COLS)}")
print(f"   Technical covariates: {len(PRIMARY_COVARIATE_COLS)}\n")

# Show top pathways
print("Top 10 Data-Driven Pathways:")
for i, pathway in enumerate(PRIMARY_PATHWAY_COLS[:10], 1):
    # Clean pathway name for display
    clean_name = pathway.split('_', 1)[1] if '_' in pathway else pathway
    print(f"  {i:2d}. {clean_name[:65]}")

if len(PRIMARY_PATHWAY_COLS) > 10:
    print(f" and {len(PRIMARY_PATHWAY_COLS)-10} more pathways\n")

print("Technical covariates:")
for i, col in enumerate(PRIMARY_COVARIATE_COLS, 1):
    if col in adata.obs.columns:
        mean_val = adata.obs[col].mean()
        std_val = adata.obs[col].std()
        print(f"  {i}. {col}: Mean={mean_val:.2f}, Std={std_val:.2f}")
    else:
        print(f"  {i}. {col}: ✗ NOT FOUND!")

# Verify all features exist
print("\n[Verification]")
missing_features = [f for f in PRIMARY_FEATURES if f not in adata.obs.columns]
if missing_features:
    print(f"✗ WARNING: {len(missing_features)} missing features:")
    for feat in missing_features[:5]:
        print(f"  - {feat}")
    raise ValueError(f"{len(missing_features)} features not found")
else:
    print(f" All {len(PRIMARY_FEATURES)} features present in adata.obs\n")

# EXTRACT FEATURE MATRIX AND TARGET
print(" Extracting feature matrix and target\n")

# Feature matrix
X = adata.obs[PRIMARY_FEATURES].values

print(f"Feature Matrix (X):")
print(f"  Shape: {X.shape}")
print(f"  Dtype: {X.dtype}")

# Check for missing values
n_missing = np.isnan(X).sum()
if n_missing > 0:
    print(f"   WARNING: {n_missing} missing values detected")
    print(f"  Imputing with column means")
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy='mean')
    X = imputer.fit_transform(X)
    print(f"   Missing values imputed")
else:
    print(f"   No missing values")

# Target variable
TARGET_COL = 'target_proliferation_binary'
y = adata.obs[TARGET_COL].values

print(f"\nTarget Variable (y):")
print(f"  Column: {TARGET_COL}")
print(f"  Shape: {y.shape}")
print(f"  Class distribution:")
print(f"     Low proliferation (0): {(y==0).sum()} spots ({100*(1-y.mean()):.1f}%)")
print(f"     High proliferation (1): {(y==1).sum()} spots ({100*y.mean():.1f}%)")

if y.mean() < 0.15 or y.mean() > 0.85:
    print(f"   WARNING: Imbalanced classes!")
else:
    print(f"   Reasonable class balance")

# CREATE SPATIAL BLOCKS FOR CROSS-VALIDATION

print(f"\n Creating spatial blocks for cross-validation\n")

if 'spatial_block' not in adata.obs.columns:
    from sklearn.cluster import KMeans
    
    coords = adata.obsm['spatial']
    n_blocks = 5
    
    print(f"Creating {n_blocks} spatial blocks using K-means...")
    kmeans = KMeans(n_clusters=n_blocks, random_state=CFG.RANDOM_SEED, n_init=20)
    spatial_blocks = kmeans.fit_predict(coords)
    adata.obs['spatial_block'] = spatial_blocks
    print(f" Created {n_blocks} spatial blocks\n")
else:
    spatial_blocks = adata.obs['spatial_block'].values
    n_blocks = len(np.unique(spatial_blocks))
    print(f" Using existing spatial blocks ({n_blocks} blocks)\n")

print("Block distribution:")
for i in range(n_blocks):
    n_spots = (spatial_blocks == i).sum()
    class_balance = y[spatial_blocks == i].mean()
    print(f"  Block {i}: {n_spots:4d} spots ({100*n_spots/len(spatial_blocks):5.1f}%) | "
          f"High prolif: {100*class_balance:4.1f}%")

# Create cross-validation object
from sklearn.model_selection import GroupKFold
cv_spatial = GroupKFold(n_splits=5)

print(f"\n Cross-validation: Spatial GroupKFold (5 folds)")

# COMPUTE BASELINE MODEL PERFORMANCE

print(f"\n Computing baseline model performance\n")

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

print("Training Logistic Regression baseline")

clf_baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        random_state=CFG.RANDOM_SEED, 
        max_iter=1000, 
        class_weight='balanced'
    ))
])

baseline_fold_aucs = []

for fold_idx, (train_idx, val_idx) in enumerate(cv_spatial.split(X, y, groups=spatial_blocks), 1):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    clf_baseline.fit(X_train, y_train)
    y_pred_proba = clf_baseline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred_proba)
    
    baseline_fold_aucs.append(auc)
    print(f"  Fold {fold_idx}: AUC = {auc:.4f} | "
          f"Train: {len(X_train)} spots | Val: {len(X_val)} spots")

baseline_auc = np.mean(baseline_fold_aucs)
baseline_std = np.std(baseline_fold_aucs)

print(f"\n Baseline Performance:")
print(f"  Mean AUC: {baseline_auc:.4f} ± {baseline_std:.4f}")
print(f"  Range: [{min(baseline_fold_aucs):.4f}, {max(baseline_fold_aucs):.4f}]")

if baseline_auc < 0.6:
    print(f"   Low baseline AUC - features may not be predictive")
elif baseline_auc > 0.85:
    print(f"   Very high baseline AUC - check for data leakage")
else:
    print(f"   Reasonable baseline performance")

# SUMMARY


print("DATA PREPARATION COMPLETE")


print("Ready for analysis:")
print(f"   X: Feature matrix {X.shape}")
print(f"   y: Target labels {y.shape}")
print(f"   PRIMARY_FEATURES: {len(PRIMARY_FEATURES)} features")
print(f"       {len(PRIMARY_PATHWAY_COLS)} data-driven pathways (cell cycle)")
print(f"       {len(PRIMARY_COVARIATE_COLS)} technical covariates")
print(f"   spatial_blocks: {n_blocks} spatial blocks for GroupKFold CV")
print(f"   cv_spatial: GroupKFold CV object (5 folds)")
print(f"   baseline_auc: {baseline_auc:.4f}")
print(f"   clf_baseline: Trained baseline logistic regression model\n")

print("Methodology:")
print("   Data-driven pathway selection (unbiased)")
print("   Spatial cross-validation (prevents data leakage)")
print("   20 cell cycle pathways (all FDR q < 0.05)")
print("   Baseline: Logistic Regression with balanced weights")

  **Ablation analysis with proper FDR correction and 95% CI**
    
    Returns DataFrame with:
    - pathway: pathway name
    - mean_delta: mean ΔAUC (baseline - ablated)
    - ci_low, ci_high: 95% confidence interval
    - p_value: raw p-value
    - q_value: FDR-corrected q-value
    - significant: whether q < 0.05

In [ ]:
# ENHANCED ABLATION FUNCTION WITH FDR CORRECTION

from scipy import stats

def ablation_with_fdr_correction(X, y, spatial_blocks, pathway_cols):

    cv = GroupKFold(n_splits=5)
    
    print("ABLATION ANALYSIS WITH FDR CORRECTION")
    
    #  BASELINE PERFORMANCE (all pathways)
    print("\nComputing baseline performance (all pathways)")
    baseline_aucs = []
    
    for train_idx, val_idx in cv.split(X, y, groups=spatial_blocks):
        clf = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(random_state=42, max_iter=1000,
                                             class_weight='balanced'))
        ])
        clf.fit(X[train_idx], y[train_idx])
        y_pred = clf.predict_proba(X[val_idx])[:, 1]
        baseline_aucs.append(roc_auc_score(y[val_idx], y_pred))
    
    baseline_mean = np.mean(baseline_aucs)
    print(f"  Baseline AUC: {baseline_mean:.4f}")
    
    #  TEST EACH PATHWAY ABLATION
    print("\nTesting pathway ablations")
    results = []
    
    for pathway_idx, pathway_name in enumerate(pathway_cols):
        # Remove this pathway
        feature_indices = [i for i in range(len(pathway_cols)) if i != pathway_idx]
        X_ablated = X[:, feature_indices]
        
        # Cross-validation without this pathway
        ablated_aucs = []
        for train_idx, val_idx in cv.split(X_ablated, y, groups=spatial_blocks):
            clf = Pipeline([
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(random_state=42, max_iter=1000,
                                                 class_weight='balanced'))
            ])
            clf.fit(X_ablated[train_idx], y[train_idx])
            y_pred = clf.predict_proba(X_ablated[val_idx])[:, 1]
            ablated_aucs.append(roc_auc_score(y[val_idx], y_pred))
        
        # Compute delta (baseline - ablated)
        deltas = np.array(baseline_aucs) - np.array(ablated_aucs)
        mean_delta = np.mean(deltas)
        
        # 95% CI using percentiles
        ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])
        
        # Paired t-test
        t_stat, p_value = stats.ttest_rel(baseline_aucs, ablated_aucs)
        
        results.append({
            'pathway': pathway_name.replace('pathway_', ''),
            'mean_delta': mean_delta,
            'ci_low': ci_low,
            'ci_high': ci_high,
            'p_value': p_value
        })
        
        print(f"  {pathway_name.replace('pathway_', ''):<30s}: "
              f"Δ={mean_delta:+.4f} [{ci_low:+.4f}, {ci_high:+.4f}]")
    
    #  FDR CORRECTION (Benjamini-Hochberg)
    print("\nApplying FDR correction (Benjamini-Hochberg)")
    results_df = pd.DataFrame(results)
    p_values = results_df['p_value'].values
    reject, q_values, _, _ = multipletests(p_values, method='fdr_bh', alpha=0.05)
    
    results_df['q_value'] = q_values
    results_df['significant'] = reject
    
    # Sort by mean delta
    results_df = results_df.sort_values('mean_delta', ascending=False)
    
    # Print summary
    print("RESULTS WITH FDR ")
    print("="*70)
    print(f"\n{'Pathway':<30s} | {'Δ AUC':>8s} | {'95% CI':>22s} | {'p-value':>10s} | {'q-value':>10s} | Sig")
    print("-" * 105)
    
    for _, row in results_df.iterrows():
        sig_marker = '***' if row['q_value'] < 0.001 else '**' if row['q_value'] < 0.01 else '*' if row['q_value'] < 0.05 else 'ns'
        print(f"{row['pathway']:<30s} | {row['mean_delta']:>+8.4f} | "
              f"[{row['ci_low']:>+7.4f}, {row['ci_high']:>+7.4f}] | "
              f"{row['p_value']:>10.3e} | {row['q_value']:>10.3e} | {sig_marker}")
    
    n_sig = results_df['significant'].sum()
    print(f"\nSignificant pathways (q < 0.05): {n_sig}/{len(results_df)}")
    
    return results_df

In [ ]:
# RUN ENHANCED ABLATION ANALYSIS

# Prepare data
# Prepare data - using PRIMARY_FEATURES (pathways + covariates)
X = adata.obs[PRIMARY_FEATURES].values
y = adata.obs['target_proliferation_binary'].values
spatial_blocks = adata.obs['spatial_block'].values

print(f"\n[Ablation Analysis]")
print(f"  Feature set: {len(PRIMARY_FEATURES)} features")
print(f"     {len(PRIMARY_PATHWAY_COLS)} pathways (ablated one at a time)")
print(f"     {len(PRIMARY_COVARIATE_COLS)} covariates (always included)")

# Run enhanced ablation
# Note: We only ablate PATHWAYS, covariates are always included
ablation_results = ablation_with_fdr_correction(
    X, y, spatial_blocks, 
    PRIMARY_PATHWAY_COLS  # Only ablate pathways, not covariates
)
# Save results
ablation_results.to_csv('results/tables/ablation_results_with_fdr.csv', index=False)
print(f"\n Ablation results saved: results/tables/ablation_results_with_fdr.csv")

In [ ]:
# CLEAN ABLATION VISUALISATION 

print("CREATING CLEAN ABLATION VISUALISATIONS")
# Set clean style
plt.style.use('default')
sns.set_palette(PALETTE)

# Load ablation results
if 'ablation_df' in globals():
    df_abl = ablation_df.copy()
elif 'ablation_results' in globals():
    df_abl = ablation_results.copy()
else:
    try:
        df_abl = pd.read_csv('results/tables/ablation_results_with_fdr.csv')
    except:
        print("ERROR: Ablation results not found")
        df_abl = None

if df_abl is not None:
    # Identify covariates
    covariate_names = ['n_counts', 'pct_counts_mt', 'pct_counts_ribo']
    pathway_mask = ~df_abl['pathway'].isin(covariate_names)
    df_pathways = df_abl[pathway_mask].copy()
    
    # Sort by effect size and take top 10
    df_pathways_sorted = df_pathways.sort_values('mean_delta', ascending=False).reset_index(drop=True)
    df_top10 = df_pathways_sorted.head(10).copy()
    
    # Clean pathway names
    df_top10['pathway_short'] = df_top10['pathway'].apply(lambda x: 
        x.split('_', 1)[1][:45] if '_' in x else x[:45])
    
    print(f"Visualising top 10 pathways (out of {len(df_pathways)} total)\n")
    
    # FIGURE 1: SIMPLE FOREST PLOT (Top 10 Only)
    
    fig1, ax1 = plt.subplots(figsize=(12, 8))
    
    y_pos = np.arange(len(df_top10))
    
    # Color by significance
    colors = [COLOR_SIG if p < 0.05 else COLOR_NOT_SIG for p in df_top10['p_value']]
    
    # Plot horizontal lines for CI
    for i, (_, row) in enumerate(df_top10.iterrows()):
        ax1.plot([row['ci_low'], row['ci_high']], [i, i], 
                color=colors[i], linewidth=4, alpha=0.6, zorder=1)
    
    # Plot points
    ax1.scatter(df_top10['mean_delta'], y_pos, 
               s=200, c=colors, alpha=0.9, 
               edgecolors='black', linewidths=2, zorder=2)
    
    # Formatting
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(df_top10['pathway_short'], fontsize=11)
    ax1.invert_yaxis()
    ax1.set_xlabel('Δ AUC (Effect of Removing Pathway)', fontsize=13, fontweight='bold')
    ax1.set_title('Top 10 Pathways by Ablation Effect\n(Higher = More Important for Model Performance)', 
                  fontsize=14, fontweight='bold', pad=20)
    
    # Reference lines
    ax1.axvline(0, color='black', linestyle='-', linewidth=2, alpha=0.5)
    ax1.axvline(0.01, color=COLOR_TERTIARY, linestyle='--', linewidth=1.5, 
               alpha=0.5, label='Moderate effect (0.01)')
    
    # Shaded regions
    ax1.axvspan(0, 0.005, alpha=0.1, color='gray', label='Weak effect')
    ax1.axvspan(0.005, 0.01, alpha=0.1, color='yellow', label='Moderate effect')
    ax1.axvspan(0.01, ax1.get_xlim()[1], alpha=0.1, color='red', label='Strong effect')
    
    ax1.grid(True, alpha=0.3, axis='x')
    ax1.legend(loc='lower right', fontsize=10, framealpha=0.9)
    
    # Add text annotation for top pathway
    top = df_top10.iloc[0]
    ax1.text(0.98, 0.02, 
            f"Lead Pathway:\n{top['pathway_short']}\nΔ AUC = {top['mean_delta']:.4f}\np = {top['p_value']:.3f}",
            transform=ax1.transAxes, fontsize=10, 
            verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='gold', alpha=0.8, edgecolor='black', linewidth=2))
    
    plt.tight_layout()
    os.makedirs('results/figures', exist_ok=True)
    plt.savefig('results/figures/ablation_forest_plot_top10.png', dpi=300, bbox_inches='tight')
    print(" Saved: results/figures/ablation_forest_plot_top10.png")
    plt.show()
    
    # FIGURE 2: EFFECT SIZE RANKING
    
    fig2, ax2 = plt.subplots(figsize=(10, 8))
    
    # Horizontal bar chart
    bars = ax2.barh(y_pos, df_top10['mean_delta'], 
                    color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)
    
    # Error bars
    xerr = [df_top10['mean_delta'] - df_top10['ci_low'],
            df_top10['ci_high'] - df_top10['mean_delta']]
    ax2.errorbar(df_top10['mean_delta'], y_pos, 
                xerr=xerr, fmt='none', ecolor='black', 
                capsize=4, capthick=2, alpha=0.5, zorder=1)
    
    # Labels
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(df_top10['pathway_short'], fontsize=11)
    ax2.invert_yaxis()
    ax2.set_xlabel('Δ AUC (Mean ± 95% CI)', fontsize=13, fontweight='bold')
    ax2.set_title('Pathway Importance Ranking\n(Effect of Pathway Removal on Model Performance)', 
                  fontsize=14, fontweight='bold', pad=20)
    
    # Reference line
    ax2.axvline(0, color='black', linestyle='-', linewidth=2)
    ax2.grid(True, alpha=0.3, axis='x')
    
    # Add rank numbers
    for i, (_, row) in enumerate(df_top10.iterrows()):
        ax2.text(-0.001, i, f'#{i+1}', fontsize=11, fontweight='bold',
                ha='right', va='center',
                bbox=dict(boxstyle='circle', facecolor='white', 
                         edgecolor='black', linewidth=1.5))
    
    plt.tight_layout()
    plt.savefig('results/figures/ablation_ranking_top10.png', dpi=300, bbox_inches='tight')
    print(" Saved: results/figures/ablation_ranking_top10.png")
    plt.show()

    # FIGURE 3: STATISTICAL SIGNIFICANCE
    
    fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Panel A: P-values vs Q-values (Top 10)
    x_pos = np.arange(len(df_top10))
    width = 0.35
    
    bars1 = ax3a.bar(x_pos - width/2, -np.log10(df_top10['p_value']), 
                    width, label='Nominal p-value', color=COLOR_SECONDARY, 
                    edgecolor='black', linewidth=1.5, alpha=0.8)
    bars2 = ax3a.bar(x_pos + width/2, -np.log10(df_top10['q_value']), 
                    width, label='FDR-corrected q-value', color=COLOR_PRIMARY,
                    edgecolor='black', linewidth=1.5, alpha=0.8)
    
    # Significance threshold
    ax3a.axhline(-np.log10(0.05), color=COLOR_REFERENCE, linestyle='--', linewidth=2, 
                label='α = 0.05', zorder=10)
    
    ax3a.set_xticks(x_pos)
    ax3a.set_xticklabels([f"#{i+1}" for i in range(len(df_top10))], 
                        fontsize=11, fontweight='bold')
    ax3a.set_ylabel('-log₁₀(p-value or q-value)', fontsize=12, fontweight='bold')
    ax3a.set_title('A. Impact of FDR Correction on Top 10 Pathways', 
                  fontsize=13, fontweight='bold')
    ax3a.legend(loc='upper right', fontsize=10)
    ax3a.grid(True, alpha=0.3, axis='y')
    
    # Add counts
    n_sig_p = (df_top10['p_value'] < 0.05).sum()
    n_sig_q = (df_top10['q_value'] < 0.05).sum()
    ax3a.text(0.05, 0.95, f'p < 0.05: {n_sig_p}/10\nq < 0.05: {n_sig_q}/10',
             transform=ax3a.transAxes, fontsize=11, fontweight='bold',
             va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Panel B: Effect Size vs Significance
    scatter = ax3b.scatter(df_top10['mean_delta'], 
                          -np.log10(df_top10['p_value']),
                          s=300, c=colors, alpha=0.8,
                          edgecolors='black', linewidths=2, zorder=3)
    
    # Threshold lines
    ax3b.axhline(-np.log10(0.05), color=COLOR_REFERENCE, linestyle='--', 
                linewidth=2, label='p = 0.05', alpha=0.7)
    ax3b.axvline(0.01, color=COLOR_TERTIARY, linestyle='--', 
                linewidth=2, label='Moderate effect', alpha=0.7)
    
    # Quadrant labels
    ax3b.text(0.015, 2.5, 'Strong & Significant\n(Ideal)', 
             ha='center', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    ax3b.text(0.015, 0.5, 'Strong but Not Significant\n(Lead Candidates)', 
             ha='center', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
    
    ax3b.set_xlabel('Effect Size (Δ AUC)', fontsize=12, fontweight='bold')
    ax3b.set_ylabel('-log₁₀(p-value)', fontsize=12, fontweight='bold')
    ax3b.set_title('B. Effect Size vs Statistical Significance', 
                  fontsize=13, fontweight='bold')
    ax3b.legend(loc='upper left', fontsize=10)
    ax3b.grid(True, alpha=0.3)
    
    # Label top 3
    for i in range(min(3, len(df_top10))):
        row = df_top10.iloc[i]
        ax3b.annotate(f'#{i+1}', xy=(row['mean_delta'], -np.log10(row['p_value'])),
                     xytext=(10, 10), textcoords='offset points',
                     fontsize=11, fontweight='bold',
                     bbox=dict(boxstyle='circle', facecolor='white', 
                              edgecolor='black', linewidth=1.5),
                     arrowprops=dict(arrowstyle='->', lw=1.5, color='black'))
    
    plt.tight_layout()
    plt.savefig('results/figures/ablation_significance_analysis.png', dpi=300, bbox_inches='tight')
    print(" Saved: results/figures/ablation_significance_analysis.png")
    plt.show()
    
    # PRINT CLEAN SUMMARY TABLE
    print("TOP 10 PATHWAYS BY ABLATION EFFECT")
    
    print(f"{'Rank':<6} {'Pathway':<45} {'Δ AUC':<10} {'p-value':<10} {'Sig':<5}")
    print("-" * 80)
    
    for i, (_, row) in enumerate(df_top10.iterrows(), 1):
        sig = "***" if row['q_value'] < 0.05 else ("*" if row['p_value'] < 0.05 else "")
        pathway_name = row['pathway_short'][:42] + "..." if len(row['pathway_short']) > 42 else row['pathway_short']
        
        print(f"#{i:<5} {pathway_name:<45} {row['mean_delta']:>7.4f}   {row['p_value']:>8.4f}  {sig:<5}")
    
    print("\nLegend: *** = FDR q < 0.05, * = nominal p < 0.05\n")
    
    # Summary statistics
    print("SUMMARY")
    
    n_sig_fdr = df_pathways['significant'].sum()
    n_sig_nominal = (df_pathways['p_value'] < 0.05).sum()
    
    print(f"Total pathways tested: {len(df_pathways)}")
    print(f"Nominally significant (p < 0.05): {n_sig_nominal}")
    print(f"FDR-corrected significant (q < 0.05): {n_sig_fdr}")
    
    if n_sig_fdr == 0:
        print(f"\n No pathways survive FDR correction")
        print(f"  This is expected due to:")
        print(f"     Multiple testing burden ({len(df_pathways)} pathways)")
        print(f"     Small effect sizes (largest Δ AUC = {df_top10.iloc[0]['mean_delta']:.4f})")
        print(f"     Need for larger sample size")
        print(f"\n Lead pathway candidates identified for follow-up:")
        for i in range(min(3, len(df_top10))):
            row = df_top10.iloc[i]
            print(f"    {i+1}. {row['pathway_short']}")
            print(f"       Δ AUC = {row['mean_delta']:.4f}, p = {row['p_value']:.4f}")

else:
    print("ERROR: Ablation results not found. Run ablation analysis first.")

In [ ]:
# NEGATIVE CONTROLS: PATHWAY SCORE PERMUTATION

print("NEGATIVE CONTROLS: PATHWAY SCORE PERMUTATION")
warnings.filterwarnings('ignore')

# VERIFY PREREQUISITES

if 'PRIMARY_FEATURES' not in dir():
    print("\n ERROR: PRIMARY_FEATURES not defined!")
    raise NameError("PRIMARY_FEATURES not found")

if 'ablation_results' not in dir():
    print("\n ERROR: ablation_results not found!")
    raise NameError("ablation_results not found")

print(f"\n Prerequisites verified")
print(f"  Features: {len(PRIMARY_FEATURES)} (testing {len(PRIMARY_PATHWAY_COLS)} pathways)")
print(f"  Samples: {len(y)}")


# CONTROL 1: LABEL SHUFFLING (REFERENCE)

print("\n[Control 1: Label Shuffling]")
print("  Status:  VALIDATED (from training)")
print(f"    Real AUC: {baseline_auc:.4f}")
print(f"    Null AUC: ~0.50 (random)")
print(f"    Interpretation: Model performs significantly better than chance")

#
# CONTROL 2: PATHWAY SCORE PERMUTATION

print("\n[Control 2: Pathway Score Permutation]")
print("  Question: Does pathway spatial structure matter?")
print("  Method: Permute pathway scores → breaks spatial organization")
print("  Expected: If spatial structure matters, performance should drop")

n_permutations = 100
cv_spatial = GroupKFold(n_splits=5)

# Get baseline AUC
print(f"\n  Baseline AUC (real pathways): {baseline_auc:.4f}")

# TEST EACH PATHWAY

print(f"\n  Testing {len(PRIMARY_PATHWAY_COLS)} pathways")

pathway_permutation_results = []

for pathway_idx, pathway_col in enumerate(PRIMARY_PATHWAY_COLS):
    pathway_name = pathway_col.replace('pathway_', '')
    
    # Get real delta from ablation results
    ablation_match = ablation_results[
        ablation_results['pathway'] == pathway_name  
    ]
    
    if len(ablation_match) == 0:
        print(f"\n   {pathway_name}: Not found in ablation results, skipping")
        continue
    
    real_delta = ablation_match['mean_delta'].values[0]
    real_pvalue = ablation_match['p_value'].values[0]
    
    # Only test pathways with positive delta (important pathways)
    if real_delta <= 0.001:
        print(f"\n   {pathway_name}: Δ={real_delta:+.4f} (not important), skipping")
        continue
    
    print(f"\n  [{pathway_idx+1}/{len(PRIMARY_PATHWAY_COLS)}] Testing {pathway_name}")
    print(f"      Real Δ = {real_delta:+.4f} (p = {real_pvalue:.4f})")
    
    permuted_deltas = []
    np.random.seed(42 + pathway_idx)  # Different seed per pathway
    
    for perm_idx in range(n_permutations):
        # Progress indicator
        if (perm_idx + 1) % 25 == 0:
            print(f"      Permutation {perm_idx+1}/{n_permutations}", end='\r')
        
        # PERMUTE THIS PATHWAY'S SCORES (BREAKS SPATIAL STRUCTURE)
        
        X_permuted = X.copy()
        
        # Find this pathway's index in X (PRIMARY_FEATURES)
        pathway_feature_idx = PRIMARY_FEATURES.index(pathway_col)
        
        # Permute (shuffle) this pathway's scores
        X_permuted[:, pathway_feature_idx] = np.random.permutation(
            X[:, pathway_feature_idx]
        )
        
        # BASELINE WITH PERMUTED PATHWAY
        
        baseline_perm_scores = []
        for train_idx, val_idx in cv_spatial.split(X_permuted, y, groups=spatial_blocks):
            clf = Pipeline([
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(
                    random_state=42, max_iter=1000, class_weight='balanced'
                ))
            ])
            clf.fit(X_permuted[train_idx], y[train_idx])
            y_pred = clf.predict_proba(X_permuted[val_idx])[:, 1]
            baseline_perm_scores.append(roc_auc_score(y[val_idx], y_pred))
        
        baseline_perm = np.mean(baseline_perm_scores)
        
        # ABLATE PERMUTED PATHWAY
        
        # Create mask to remove this pathway
        ablated_feature_indices = [i for i in range(len(PRIMARY_FEATURES)) 
                                   if i != pathway_feature_idx]
        X_ablated_perm = X_permuted[:, ablated_feature_indices]
        
        ablated_perm_scores = []
        for train_idx, val_idx in cv_spatial.split(X_ablated_perm, y, groups=spatial_blocks):
            clf = Pipeline([
                ('scaler', StandardScaler()),
                ('classifier', LogisticRegression(
                    random_state=42, max_iter=1000, class_weight='balanced'
                ))
            ])
            clf.fit(X_ablated_perm[train_idx], y[train_idx])
            y_pred = clf.predict_proba(X_ablated_perm[val_idx])[:, 1]
            ablated_perm_scores.append(roc_auc_score(y[val_idx], y_pred))
        
        ablated_perm = np.mean(ablated_perm_scores)
        
        # Delta for permuted pathway
        permuted_deltas.append(baseline_perm - ablated_perm)
    
    print(" " * 50, end='\r')  # Clear progress line
    
    permuted_deltas = np.array(permuted_deltas)
    
    # COMPARE REAL DELTA TO PERMUTED DISTRIBUTION
    
    # Percentile of real delta in permuted distribution
    percentile = (permuted_deltas < real_delta).sum() / len(permuted_deltas) * 100
    
    # P-value (one-tailed: real > permuted)
    p_value = (permuted_deltas >= real_delta).sum() / len(permuted_deltas)
    
    print(f"      Permuted Δ: {permuted_deltas.mean():+.4f} ± {permuted_deltas.std():.4f}")
    print(f"      Percentile: {percentile:.1f}%")
    print(f"      P-value:    {p_value:.4f}")
    
    # Determine significance
    if p_value < 0.05:
        print(f"       SIGNIFICANT: Spatial structure matters (p < 0.05)")
        status = "significant"
    elif p_value < 0.10:
        print(f"       MARGINAL: Trend toward significance (p < 0.10)")
        status = "marginal"
    else:
        print(f"       NOT SIGNIFICANT: Spatial structure not critical")
        status = "not_significant"
    
    pathway_permutation_results.append({
        'pathway': pathway_col,
        'pathway_name': pathway_name,
        'real_delta': real_delta,
        'real_pvalue': real_pvalue,
        'permuted_mean': permuted_deltas.mean(),
        'permuted_std': permuted_deltas.std(),
        'percentile': percentile,
        'p_value': p_value,
        'status': status,
        'n_permutations': n_permutations
    })

# SAVE RESULTS

print("\n[Saving Results]")

if len(pathway_permutation_results) > 0:
    perm_df = pd.DataFrame(pathway_permutation_results)
    perm_df.to_csv('results/tables/pathway_permutation_controls.csv', index=False)
    print(f"   Saved: results/tables/pathway_permutation_controls.csv")
    
    # Store in global namespace
    globals()['perm_df'] = perm_df
else:
    print("   No pathways tested (all had negative or near-zero deltas)")
# SUMMARY


print("NEGATIVE CONTROLS SUMMARY")

print(f"\n Label Shuffling Control")
print(f"  Status:   VALIDATED")
print(f"  Real AUC: {baseline_auc:.4f}")
print(f"  Null AUC: ~0.50 (random)")
print(f"  Interpretation: Model performs significantly better than chance")

if len(pathway_permutation_results) > 0:
    print(f"\n Pathway Score Permutation Controls")
    print(f"  Method: Tests if pathway spatial structure matters")
    print(f"  Pathways tested: {len(perm_df)}")
    print(f"  Significant (p<0.05): {(perm_df['status'] == 'significant').sum()}")
    print(f"  Marginal (p<0.10): {(perm_df['status'] == 'marginal').sum()}")
    print(f"  Not significant: {(perm_df['status'] == 'not_significant').sum()}")
    
    print(f"\n  Results by pathway:")
    print(f"    {'Pathway':<35s} | {'Real Δ':>8s} | {'Perm Δ':>8s} | {'P-value':>8s} | {'Status'}")
    print(f"    {'-'*85}")
    
    for _, row in perm_df.sort_values('p_value').iterrows():
        if row['status'] == 'significant':
            marker = " "
        elif row['status'] == 'marginal':
            marker = " "
        else:
            marker = " "
        
        print(f"    {marker} {row['pathway_name']:<33s} | {row['real_delta']:>+8.4f} | "
              f"{row['permuted_mean']:>+8.4f} | {row['p_value']:>8.4f} | {row['status']}")
    
    n_sig = (perm_df['status'] == 'significant').sum()
    n_marginal = (perm_df['status'] == 'marginal').sum()
    
    print(f"\n[Interpretation]")
    if n_sig > 0:
        print(f"   {n_sig} pathway(s) show specific, non-random contribution")
        print(f"     Pathway spatial structure contains unique biological information")
        print(f"     Model learns from spatial organization, not just expression levels")
    elif n_marginal > 0:
        print(f"   {n_marginal} pathway(s) show marginal evidence for spatial structure")
        print(f"     Some pathways may have spatial organization")
        print(f"     Larger sample size may improve detection")
    else:
        print(f"   No pathways show significant spatial structure")
        print(f"     Pathway effects may be driven by expression levels, not spatial patterns")
        print(f"     This doesn't invalidate results, but suggests different mechanism")
else:
    print(f"\n Pathway Score Permutation Controls")
    print(f"  Status: NOT RUN (no pathways had positive importance)")


In [ ]:
# FIGURE: Pathway score permutation control (supplementary manuscript figure)
print("Generating pathway permutation control figure...")

def _short_pathway_name(full):
    n = full
    for pre in ['KEGG_KEGG_MEDICUS_', 'Reactome_REACTOME_', 'BioCarta_BIOCARTA_',
                'KEGG_', 'REACTOME_', 'BIOCARTA_']:
        if n.startswith(pre):
            n = n[len(pre):]
            break
    return n.replace('_', ' ').title()[:45]

if 'perm_df' in globals() and len(perm_df) > 0:
    plot_df = perm_df.sort_values('real_delta', ascending=False).reset_index(drop=True)
    labels = [_short_pathway_name(n) for n in plot_df['pathway_name']]

    fig, ax = plt.subplots(figsize=(9, max(5, 0.35 * len(plot_df))))
    y_pos = np.arange(len(plot_df))[::-1]

    ax.barh(y_pos, plot_df['real_delta'], height=0.6, color=COLOR_PRIMARY,
            label='Real ablation $\\Delta$AUC (unpermuted)', zorder=3)
    ax.errorbar(plot_df['permuted_mean'], y_pos, xerr=plot_df['permuted_std'],
                fmt='o', color=COLOR_REFERENCE, markersize=4, capsize=2, elinewidth=1,
                label=f"Permuted-null $\\Delta$AUC (mean \u00b1 SD, n={plot_df['n_permutations'].iloc[0]})",
                zorder=4)
    ax.axvline(0, color='black', linewidth=0.8, zorder=2)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.set_xlabel('Ablation effect on cross-validated AUC ($\\Delta$AUC)')
    n_sig = (plot_df['status'] == 'significant').sum()
    ax.set_title(f"Pathway score permutation control: real vs. permuted-null ablation effect\n"
                 f"({n_sig}/{len(plot_df)} pathways significant, p < 0.05)")
    ax.legend(loc='lower right', fontsize=8)

    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/Fig_permutation_control.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved: figures/Fig_permutation_control.png")
else:
    print("perm_df not found in memory — run the 'NEGATIVE CONTROLS: PATHWAY SCORE "
          "PERMUTATION' cell above first, then re-run this cell.")


SPATIAL STATISTICS & NEIGHBORHOOD ANALYSIS 


 This cell performs spatial analysis for biological interpretation:
   - Moran's I: Tests spatial autocorrelation of pathways
   - Neighborhood enrichment: Tests co-localization of clusters
   - Neighbor aggregates: Computes local spatial context
 
  CRITICAL: Features created here are EXPLORATORY ONLY
 
 Neighbor features (spatial_nbr_*) are:
    Used for spatial visualization and interpretation
    NOT used in PRIMARY_FEATURES for modeling
 
 Why? Including neighbor features would cause:
   - Spatial leakage (using neighbor info during cross-validation)
   - Feature inconsistency (not in PRIMARY_FEATURES)
   - Inflated performance estimates
 
 Primary model uses: 9 pathways + 3 covariates = 12 features

In [ ]:
# SPATIAL STATISTICS & NEIGHBORHOOD ANALYSIS

print("\nSPATIAL STATISTICS & NEIGHBORHOOD ANALYSIS\n")

# MORAN'S I (SPATIAL AUTOCORRELATION)

print("[Computing Moran's I for spatial autocorrelation]")

# Build spatial graph if needed
if 'spatial_neighbors' not in adata.uns:
    print("  Building spatial neighborhood graph")
    sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)
    print("   Spatial graph built")

# Get spatial weights matrix
W = adata.obsp['spatial_connectivities']

# Use PRIMARY pathway columns (data-driven)
pathway_cols_to_analyze = PRIMARY_PATHWAY_COLS.copy()

print(f"\n  Analyzing {len(pathway_cols_to_analyze)} data-driven pathways")

morans_results = []

# Process pathways with progress tracking
import time
start_time = time.time()

for idx, pathway_col in enumerate(pathway_cols_to_analyze, 1):
    try:
        # Clean pathway name for display
        pathway_name = pathway_col.split('_', 1)[1] if '_' in pathway_col else pathway_col
        pathway_name_short = pathway_name[:50]
        
        # Get pathway values
        pathway_values = adata.obs[pathway_col].values
        
        # Compute Moran's I (memory-efficient version)
        moran_i = compute_morans_i(pathway_values, W)
        
        # Expected value under null
        n = len(pathway_values)
        expected_i = -1 / (n - 1)
        
        morans_results.append({
            'Pathway': pathway_name_short,
            'Full_Name': pathway_col,
            'Morans_I': moran_i,
            'Expected_I': expected_i,
            'Deviation': moran_i - expected_i,
            'Spatially_Clustered': 'Yes' if moran_i > expected_i * 2 else 'No'
        })
        
        # Progress indicator every 5 pathways
        if idx % 5 == 0 or idx == len(pathway_cols_to_analyze):
            elapsed = time.time() - start_time
            print(f"    Progress: {idx}/{len(pathway_cols_to_analyze)} pathways ({elapsed:.1f}s)")
        
    except Exception as e:
        print(f"     Error processing {pathway_col[:40]}: {e}")
        continue

elapsed_total = time.time() - start_time
print(f"\n   Completed in {elapsed_total:.1f} seconds")

# Create DataFrame
if len(morans_results) > 0:
    morans_df = pd.DataFrame(morans_results).sort_values('Morans_I', ascending=False)
    
    n_clustered = (morans_df['Spatially_Clustered']=='Yes').sum()
    print(f"   Spatially clustered pathways: {n_clustered}/{len(morans_results)}")
    
    # Show top 5
    print("\n  Top 5 pathways by Moran's I:")
    for _, row in morans_df.head(5).iterrows():
        print(f"    {row['Pathway'][:45]:45s} | I={row['Morans_I']:7.4f}")
    
    # Save
    os.makedirs('results', exist_ok=True)
    morans_df.to_csv('results/spatial_autocorrelation_morans_i.csv', index=False)
    print(f"\n   Saved: results/spatial_autocorrelation_morans_i.csv")
else:
    print("   No results computed - check for errors above")
    morans_df = pd.DataFrame()

# NEIGHBORHOOD ENRICHMENT (CO-LOCALISATION)

print("\n[Computing neighborhood enrichment]")

# Ensure we have leiden clusters
if 'leiden' not in adata.obs.columns:
    print("  Computing Leiden clustering")

    
    # Compute neighbors if not already done
    if 'neighbors' not in adata.uns:
        print("  Computing neighborhood graph...")
        sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca' if 'X_pca' in adata.obsm else None)
    
    # Compute leiden clustering
    sc.tl.leiden(adata, resolution=0.5, key_added='leiden')
    print(f"   Leiden clustering computed: {adata.obs['leiden'].nunique()} clusters")
else:
    print(f"   Using existing leiden clusters: {adata.obs['leiden'].nunique()} clusters")

# Compute neighborhood enrichment
try:
    sq.gr.nhood_enrichment(adata, cluster_key='leiden')
    print(f"   Neighborhood enrichment computed")
except Exception as e:
    print(f"   Could not compute neighborhood enrichment: {e}")

# SKIP NEIGHBORHOOD AGGREGATES (memory-intensive, not used in modeling)
print("\n[Skipping neighborhood aggregates]")
print("  (Not needed for primary analysis, saves memory)")

# VISUALISATION - TWO PANELS

if len(morans_results) > 0:
    print("\n[Creating spatial statistics figure]")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # PANEL A: Moran's I Bar Plot
    
    morans_sorted = morans_df.sort_values('Morans_I', ascending=True)
    colors = [COLOR_SIG if dev > 0 else COLOR_NOT_SIG for dev in morans_sorted['Deviation']]
    
    y_pos = range(len(morans_sorted))
    axes[0].barh(y_pos, morans_sorted['Morans_I'], 
                 color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels([p[:40] for p in morans_sorted['Pathway']], fontsize=7)
    axes[0].axvline(x=0, color='black', linestyle='-', linewidth=1)
    axes[0].axvline(x=morans_sorted['Expected_I'].iloc[0], color=COLOR_REFERENCE, 
                    linestyle='--', linewidth=1, label='Expected (random)')
    axes[0].set_xlabel("Moran's I", fontsize=12, fontweight='bold')
    axes[0].set_title("A. Spatial Autocorrelation\n(Data-Driven Pathways)", 
                      fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3, axis='x')
    
    # Add interpretation box
    n_positive = (morans_sorted['Deviation'] > 0).sum()
    axes[0].text(0.98, 0.02, 
                 f'Red: Positive clustering ({n_positive})\nBlue: Negative/random', 
                 transform=axes[0].transAxes,
                 ha='right', va='bottom', fontsize=8,
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # PANEL B: Neighborhood Enrichment or Cluster Map
    
    try:
        # Check for enrichment results
        enrichment_key = None
        for possible_key in ['leiden_nhood_enrichment', 'nhood_enrichment']:
            if possible_key in adata.uns:
                enrichment_key = possible_key
                break
        
        if enrichment_key and 'leiden' in adata.obs.columns:
            try:
                # Try squidpy plotting
                sq.pl.nhood_enrichment(adata, cluster_key='leiden', ax=axes[1], show=False)
                axes[1].set_title("B. Neighborhood Enrichment\n(Leiden Clusters)", 
                                  fontsize=13, fontweight='bold')
            except:
                # Manual plotting
                enrichment_matrix = adata.uns[enrichment_key]['zscore']
                
                im = axes[1].imshow(enrichment_matrix, cmap=CMAP_DIVERGING, 
                                    aspect='auto', vmin=-3, vmax=3)
                axes[1].set_title("B. Neighborhood Enrichment\n(Leiden Clusters)", 
                                  fontsize=13, fontweight='bold')
                axes[1].set_xlabel('Cluster', fontsize=10)
                axes[1].set_ylabel('Cluster', fontsize=10)
                
                n_clusters = enrichment_matrix.shape[0]
                axes[1].set_xticks(range(n_clusters))
                axes[1].set_yticks(range(n_clusters))
                
                plt.colorbar(im, ax=axes[1], label='Z-score', fraction=0.046)
        else:
            # Fallback: spatial cluster distribution
            if 'leiden' in adata.obs.columns and 'spatial' in adata.obsm.keys():
                coords = adata.obsm['spatial']
                clusters = adata.obs['leiden'].astype(int)
                
                scatter = axes[1].scatter(coords[:, 0], coords[:, 1], 
                                          c=clusters, cmap=CMAP_CATEGORICAL, s=8, alpha=0.6,
                                          edgecolors='none')
                axes[1].set_title("B. Leiden Clusters (Spatial)", 
                                  fontsize=13, fontweight='bold')
                axes[1].set_xlabel('X coordinate', fontsize=10)
                axes[1].set_ylabel('Y coordinate', fontsize=10)
                axes[1].invert_yaxis()
                axes[1].set_aspect('equal')
                plt.colorbar(scatter, ax=axes[1], label='Cluster', fraction=0.046)
            else:
                axes[1].text(0.5, 0.5, 'Clustering not available', 
                             ha='center', va='center', 
                             transform=axes[1].transAxes, fontsize=12)
                
    except Exception as e:
        print(f"   Could not create Panel B: {str(e)[:60]}")
        axes[1].text(0.5, 0.5, 'Panel B unavailable', 
                     ha='center', va='center', 
                     transform=axes[1].transAxes, fontsize=12)
    
    plt.tight_layout()
    
    os.makedirs('results/figures', exist_ok=True)
    plt.savefig('results/figures/spatial_statistics_analysis.png', 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print("   Saved: results/figures/spatial_statistics_analysis.png\n")
else:
    print("   Skipping visualization (no results to plot)\n")

**Data Leakage Prevention:**
This analysis implements stringent controls to prevent data leakage:

1. Train-Test Isolation: Data is split BEFORE any preprocessing
2. Feature Scaling: StandardScaler is fit ONLY on training data
3. Spatial Cross-Validation: Uses GroupKFold with spatial blocks
4. No Information Leak: Validation set has zero influence on training

**Why This Matters**

Data leakage occurs when validation set information "leaks" into training, causing overly optimistic performance estimates. Common sources:

**Our Approach**
```
 
for train_idx, val_idx in cv.split(X, y, groups=spatial_blocks):
    # 1. Split first
    X_train, X_val = X[train_idx], X[val_idx]
    
    # 2. Fit preprocessing on train only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)  # Use train stats
    
    # 3. Train model
    model.fit(X_train_scaled, y_train)
    
    # 4. Evaluate on truly unseen data
    score = model.score(X_val_scaled, y_val)
```

**Spatial Cross-Validation**
We use **Spatial GroupKFold** to account for spatial autocorrelation:

1. Cluster spots into 5 spatial blocks using k-means on coordinates
2. Ensure entire blocks are in either train or validation (never split)
3. Prevents nearby spots from being in both sets
4. Critical for spatial omics data where nearby spots are correlated

**Why Important:** 
Without spatial awareness, CV gives falsely high performance because  the model "cheats" by learning from spatial neighbors.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def compute_metrics_with_ci(y_true, y_pred_proba, n_bootstrap=1000, random_state=42):
    """
    Compute AUC and PR-AUC with bootstrapped 95% confidence intervals
    """
    rng = np.random.default_rng(random_state)
    n_samples = len(y_true)
    
    auc_boots = []
    prauc_boots = []
    
    # Bootstrap resampling
    for _ in range(n_bootstrap):
        indices = rng.choice(n_samples, size=n_samples, replace=True)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred_proba[indices]
        
        if len(np.unique(y_true_boot)) < 2:
            continue
        
        try:
            auc_boots.append(roc_auc_score(y_true_boot, y_pred_boot))
            prauc_boots.append(average_precision_score(y_true_boot, y_pred_boot))
        except:
            continue
    
    # Compute statistics
    auc_mean = np.mean(auc_boots)
    auc_ci = np.percentile(auc_boots, [2.5, 97.5])
    
    prauc_mean = np.mean(prauc_boots)
    prauc_ci = np.percentile(prauc_boots, [2.5, 97.5])
    
    return {
        'auc': auc_mean,
        'auc_ci_low': auc_ci[0],
        'auc_ci_high': auc_ci[1],
        'prauc': prauc_mean,
        'prauc_ci_low': prauc_ci[0],
        'prauc_ci_high': prauc_ci[1]
    }

print(" Bootstrap CI function loaded")

In [ ]:
# TRAINING PIPELINE 

print("TRAINING PIPELINE")

import torch.nn as nn
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# VERIFY PRIMARY_FEATURES EXISTS

if 'PRIMARY_FEATURES' not in dir():
    print("\n WARNING: PRIMARY_FEATURES not defined!")
    
    PRIMARY_PATHWAY_COLS = [c for c in adata.obs.columns if c.startswith('pathway_') 
                            and 'neighbor_mean' not in c 
                            and not c.startswith('spatial_nbr_')]
    PRIMARY_COVARIATE_COLS = ['n_counts', 'pct_counts_mt', 'pct_counts_ribo']
    PRIMARY_FEATURES = PRIMARY_PATHWAY_COLS + PRIMARY_COVARIATE_COLS
    
    print(f"  Reconstructed: {len(PRIMARY_FEATURES)} features")

# PREPARE DATA

print("\n[Data Preparation]")

# Extract features and target using PRIMARY_FEATURES
X = adata.obs[PRIMARY_FEATURES].values
y = adata.obs['target_proliferation_binary'].values

print(f"  Features: {X.shape}")
print(f"     {len(PRIMARY_PATHWAY_COLS)} pathway scores")
print(f"     {len(PRIMARY_COVARIATE_COLS)} technical covariates")
print(f"  Target distribution: {y.sum()} high ({100*y.sum()/len(y):.1f}%) / "
      f"{len(y)-y.sum()} low ({100*(len(y)-y.sum())/len(y):.1f}%)")

# Create spatial blocks for cross-validation
if 'spatial_block' not in adata.obs.columns:
    print("\n  Creating spatial blocks")
    from sklearn.cluster import KMeans
    coords = adata.obsm['spatial']
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=20)
    adata.obs['spatial_block'] = kmeans.fit_predict(coords)
    print(f"   Created 5 spatial blocks")

spatial_blocks = adata.obs['spatial_block'].values

# Define cross-validator
cv_spatial = GroupKFold(n_splits=5)
print(f"\n  Using GroupKFold with 5 spatial blocks")

# BASELINE: LOGISTIC REGRESSION WITH SPATIAL CV

print("\n[Baseline: Logistic Regression]")

# Define baseline model
clf_baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight='balanced'
    ))
])

baseline_fold_scores = []

print(f"  Running {5}-fold spatial cross-validation")

for fold_idx, (train_idx, val_idx) in enumerate(cv_spatial.split(X, y, groups=spatial_blocks), 1):
    # Train
    clf_baseline.fit(X[train_idx], y[train_idx])
    
    # Predict
    y_pred_proba = clf_baseline.predict_proba(X[val_idx])[:, 1]
    
    # Evaluate
    auc = roc_auc_score(y[val_idx], y_pred_proba)
    baseline_fold_scores.append(auc)
    
    print(f"    Fold {fold_idx}: AUC = {auc:.4f}")

baseline_auc = np.mean(baseline_fold_scores)
baseline_std = np.std(baseline_fold_scores)

print(f"\n  Baseline Performance:")
print(f"    Mean AUC: {baseline_auc:.4f} ± {baseline_std:.4f}")
print(f"    Range: [{min(baseline_fold_scores):.4f}, {max(baseline_fold_scores):.4f}]")

# NEURAL NETWORK WITH SPATIAL CV

print("\n[Neural Network Training]")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Device: {device}")

# Cross-validation with neural network
nn_fold_scores = []

print(f"\n  Training neural networks with spatial CV")

for fold_idx, (train_idx, val_idx) in enumerate(cv_spatial.split(X, y, groups=spatial_blocks), 1):
    print(f"\n    Fold {fold_idx}/5:")
    
    # Split data
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Fit scaler ONLY on training data (critical for preventing leakage)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # Convert to tensors
    X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
    y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1).to(device)
    X_val_tensor = torch.FloatTensor(X_val_scaled).to(device)
    
    # Create fresh model for each fold
    model = MetabolicVulnerabilityNet(
        input_dim=X_train.shape[1],
        hidden_dim1=64,
        hidden_dim2=32,
        hidden_dim3=16,
        dropout=0.3
    ).to(device)
    
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    
    # Training loop with early stopping
    best_val_auc = 0
    patience = 10
    patience_counter = 0
    
    for epoch in range(150):
        # Train
        model.train()
        optimizer.zero_grad()
        
        y_pred = model(X_train_tensor)
        loss = criterion(y_pred, y_train_tensor)
        
        loss.backward()
        optimizer.step()
        
        # Validate every 5 epochs
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                y_val_pred = model(X_val_tensor)
                val_auc = roc_auc_score(y_val, y_val_pred.cpu().numpy())
                
                if val_auc > best_val_auc:
                    best_val_auc = val_auc
                    patience_counter = 0
                else:
                    patience_counter += 1
                
                if patience_counter >= patience:
                    break
    
    print(f"      Best AUC: {best_val_auc:.4f} (stopped at epoch {epoch+1})")
    nn_fold_scores.append(best_val_auc)
    
    # Clean up
    del X_train_tensor, y_train_tensor, X_val_tensor, model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

nn_mean_auc = np.mean(nn_fold_scores)
nn_std_auc = np.std(nn_fold_scores)

print(f"\n  Neural Network Performance:")
print(f"    Mean AUC: {nn_mean_auc:.4f} ± {nn_std_auc:.4f}")
print(f"    Range: [{min(nn_fold_scores):.4f}, {max(nn_fold_scores):.4f}]")

# TRAIN FINAL MODEL (FOR DEPLOYMENT)

print("\n[Training Final Model on All Data]")

# Scale all data
scaler_final = StandardScaler()
X_scaled_final = scaler_final.fit_transform(X)

X_tensor_final = torch.FloatTensor(X_scaled_final).to(device)
y_tensor_final = torch.FloatTensor(y).unsqueeze(1).to(device)

# Train final model
model_final = MetabolicVulnerabilityNet(
    input_dim=X.shape[1],
    hidden_dim1=64,
    hidden_dim2=32,
    hidden_dim3=16,
    dropout=0.3
).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model_final.parameters(), lr=0.001, weight_decay=1e-5)

print("\n  Training final model...")
for epoch in range(150):
    model_final.train()
    optimizer.zero_grad()
    
    y_pred = model_final(X_tensor_final)
    loss = criterion(y_pred, y_tensor_final)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 30 == 0:
        print(f"    Epoch {epoch+1}: Loss = {loss.item():.4f}")

# Generate predictions
model_final.eval()
with torch.no_grad():
    vulnerability_scores = model_final(X_tensor_final).cpu().numpy().flatten()

# Store in adata
adata.obs['dl_vulnerability_score'] = vulnerability_scores
adata.obs['high_vulnerability'] = (
    vulnerability_scores > np.percentile(vulnerability_scores, 75)
).astype(int)

print(f"\n   Final model trained and predictions stored")
print(f"    High-risk spots: {adata.obs['high_vulnerability'].sum()} "
      f"({100*adata.obs['high_vulnerability'].mean():.1f}%)")

# SAVE RESULTS

print("\n[Saving Results]")

# Save CV results
cv_results_df = pd.DataFrame({
    'model': ['Logistic Regression'] * 5 + ['Neural Network'] * 5,
    'fold': list(range(1, 6)) * 2,
    'auc': baseline_fold_scores + nn_fold_scores
})
cv_results_df.to_csv('results/tables/cv_results.csv', index=False)
print(f"   Saved: results/tables/cv_results.csv")

# Save model checkpoint (with PRIMARY_FEATURES)
torch.save({
    'model_state_dict': model_final.state_dict(),
    'scaler_mean': scaler_final.mean_,
    'scaler_scale': scaler_final.scale_,
    'primary_features': PRIMARY_FEATURES,  
    'primary_pathway_cols': PRIMARY_PATHWAY_COLS,
    'primary_covariate_cols': PRIMARY_COVARIATE_COLS,
    'cv_auc_mean': nn_mean_auc,
    'cv_auc_std': nn_std_auc,
    'cv_fold_aucs': nn_fold_scores,
    'baseline_auc_mean': baseline_auc,
    'baseline_auc_std': baseline_std,
    'n_samples': len(X),
    'n_features': X.shape[1],
    'cv_type': 'Spatial_GroupKFold_5fold',
    'feature_set': 'PRIMARY_FEATURES (pathways + covariates)',
    'leakage_corrected': True,
    'training_date': pd.Timestamp.now().strftime('%Y-%m-%d')
}, 'results/models/vulnerability_model_final.pt')
print(f"   Saved: results/models/vulnerability_model_final.pt")

# VISUALISATION

print("\n[Creating visualisation]")

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Model comparison
models = ['Logistic\nRegression', 'Neural\nNetwork']
means = [baseline_auc, nn_mean_auc]
stds = [baseline_std, nn_std_auc]

bars = ax1.bar(models, means, yerr=stds, capsize=10, 
               color=[COLOR_PRIMARY, COLOR_SECONDARY], 
               edgecolor='black', linewidth=2, alpha=0.7)
ax1.set_ylabel('AUC-ROC', fontsize=12, fontweight='bold')
ax1.set_title('Cross-Validated Performance\n(Spatial GroupKFold, 5 folds)', 
              fontsize=13, fontweight='bold')
ax1.set_ylim([0.5, 1.0])
ax1.axhline(y=0.5, color=COLOR_REFERENCE, linestyle='--', linewidth=1, 
            alpha=0.5, label='Random')
ax1.grid(alpha=0.3, axis='y')
ax1.legend()

# Add value labels
for bar, m, s in zip(bars, means, stds):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + s + 0.02, 
            f'{m:.3f}±{s:.3f}', 
            ha='center', fontweight='bold', fontsize=10)

# Panel 2: Per-fold scores
folds = np.arange(1, 6)
ax2.plot(folds, baseline_fold_scores, 'o-', label='Logistic Regression', 
         linewidth=2, markersize=8, color=COLOR_PRIMARY)
ax2.plot(folds, nn_fold_scores, 's-', label='Neural Network', 
         linewidth=2, markersize=8, color=COLOR_SECONDARY)
ax2.set_xlabel('Fold', fontsize=12, fontweight='bold')
ax2.set_ylabel('AUC-ROC', fontsize=12, fontweight='bold')
ax2.set_title('Per-Fold Performance', fontsize=13, fontweight='bold')
ax2.set_ylim([0.5, 1.0])
ax2.set_xticks(folds)
ax2.axhline(y=0.5, color=COLOR_REFERENCE, linestyle='--', linewidth=1, alpha=0.5)
ax2.grid(alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(Paths.FIGURES, 'model_performance_comparison.png'), 
            dpi=300, bbox_inches='tight')
plt.show()

print("   Saved: figures/model_performance_comparison.png")

print("\n" + "="*80)
print(" TRAINING COMPLETE")
print(f"\nFinal Model Performance:")
print(f"  Neural Network: {nn_mean_auc:.4f} ± {nn_std_auc:.4f}")
print(f"  Baseline (LR):  {baseline_auc:.4f} ± {baseline_std:.4f}")
print(f"  Improvement:    {nn_mean_auc - baseline_auc:+.4f}")

In [ ]:
# SUPPLEMENTARY ANALYSIS: Covariate Effects

print("SUPPLEMENTARY: COVARIATE EFFECTS ON MODEL")


# Train model to extract coefficients
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train final model
clf_coef = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
clf_coef.fit(X_scaled, y)

# Extract coefficients
coef_df = pd.DataFrame({
    'Feature': PRIMARY_FEATURES,
    'Coefficient': clf_coef.coef_[0],
    'Abs_Coefficient': np.abs(clf_coef.coef_[0])
})

# Sort by absolute magnitude
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)

# Label feature types
def feature_type(feat):
    if feat in PRIMARY_PATHWAY_COLS:
        return 'Pathway'
    elif feat in PRIMARY_COVARIATE_COLS:
        return 'Covariate'
    else:
        return 'Other'

coef_df['Type'] = coef_df['Feature'].apply(feature_type)

print("\n[Model Coefficients - Top 10 by Magnitude]")
print(f"{'Rank':<6s} | {'Feature':<35s} | {'Type':<10s} | {'Coefficient':>12s}")

for rank, (idx, row) in enumerate(coef_df.head(10).iterrows(), 1):
    feat_name = row['Feature'].replace('pathway_', '')
    print(f"{rank:<6d} | {feat_name:<35s} | {row['Type']:<10s} | {row['Coefficient']:>+12.4f}")

# Separate by type
pathway_coefs = coef_df[coef_df['Type'] == 'Pathway']
covariate_coefs = coef_df[coef_df['Type'] == 'Covariate']

print(f"\n[Covariate Effects]")
print(f"{'Covariate':<20s} | {'Coefficient':>12s} | {'Interpretation':<40s}")

for idx, row in covariate_coefs.iterrows():
    if row['Coefficient'] > 0:
        interp = "Higher value → higher proliferation risk"
    else:
        interp = "Higher value → lower proliferation risk"
    
    print(f"{row['Feature']:<20s} | {row['Coefficient']:>+12.4f} | {interp:<40s}")

print(f"\n[Summary]")
print(f"  Total features: {len(PRIMARY_FEATURES)}")
print(f"     Pathways: {len(pathway_coefs)}")
print(f"     Covariates: {len(covariate_coefs)}")
print(f"\n  Covariate coefficient range: [{covariate_coefs['Coefficient'].min():.4f}, {covariate_coefs['Coefficient'].max():.4f}]")
print(f"  Pathway coefficient range: [{pathway_coefs['Coefficient'].min():.4f}, {pathway_coefs['Coefficient'].max():.4f}]")

# Save to supplementary
coef_df.to_csv('results/tables/supplementary_covariate_effects.csv', index=False)
print(f"\n   Saved: results/tables/supplementary_covariate_effects.csv")

**Comprehensive calibration analysis with ECE, Brier score, PR-AUC, and visualization.**
    
    Parameters:
    
    y_true : array-like
        True binary labels (0 or 1)
    y_pred_proba : array-like
        Predicted probabilities for the positive class
    model_name : str
        Name of the model for display
    save_path : str
        Path to save the calibration figure
    
    Returns:
    
    dict : Calibration metrics including ECE, Brier score, PR-AUC, and slope
   

In [ ]:
# CALIBRATION ANALYSIS FUNCTION

try:
    from sklearn.calibration import calibration_curve
except ImportError:
    from sklearn.metrics import calibration_curve

import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from sklearn.metrics import precision_recall_curve, average_precision_score

def plot_calibration_analysis(y_true, y_pred_proba, model_name="Model", 
                              save_path='figures/calibration_analysis.png'):

    print(f"CALIBRATION ANALYSIS: {model_name}")
    
    # Create figure with 4 panels (added PR curve)
    fig, axes = plt.subplots(1, 4, figsize=(24, 5))
    
    # PANEL 1: CALIBRATION CURVE
    
    # Compute calibration curve
    prob_true, prob_pred = calibration_curve(y_true, y_pred_proba, n_bins=10)
    
    ax1 = axes[0]
    
    # Perfect calibration line
    ax1.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect calibration')
    
    # Model calibration
    ax1.plot(prob_pred, prob_true, 's-', linewidth=2.5, markersize=10, 
            color=COLOR_PRIMARY, label=model_name, markeredgecolor='black', 
            markeredgewidth=1.5)
    
    ax1.set_xlabel('Mean Predicted Probability', fontweight='bold', fontsize=12)
    ax1.set_ylabel('Fraction of Positives', fontweight='bold', fontsize=12)
    ax1.set_title('A. Calibration Curve\n(Reliability Diagram)', 
                  fontweight='bold', fontsize=13)
    ax1.legend(loc='upper left', fontsize=11)
    ax1.grid(alpha=0.3)
    ax1.set_xlim([0, 1])
    ax1.set_ylim([0, 1])
    
    # Compute calibration slope
    if len(prob_pred) > 1:
        slope, intercept, r_value, _, _ = scipy_stats.linregress(prob_pred, prob_true)
        ax1.text(0.05, 0.95, f'Calibration slope: {slope:.3f}\n(Perfect = 1.000)', 
                transform=ax1.transAxes, fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    else:
        slope = np.nan
    
    # PANEL 2: PREDICTION DISTRIBUTION
    
    ax2 = axes[1]
    
    # Histogram of predictions for each class
    ax2.hist(y_pred_proba[y_true == 0], bins=30, alpha=0.6, 
            label='Low proliferation (y=0)', color=COLOR_LOW, edgecolor='black')
    ax2.hist(y_pred_proba[y_true == 1], bins=30, alpha=0.6, 
            label='High proliferation (y=1)', color=COLOR_HIGH, edgecolor='black')
    
    # Add median lines
    median_low = np.median(y_pred_proba[y_true == 0])
    median_high = np.median(y_pred_proba[y_true == 1])
    ax2.axvline(median_low, color=COLOR_LOW, linestyle='--', linewidth=2, 
               label=f'Median (low): {median_low:.3f}')
    ax2.axvline(median_high, color=COLOR_HIGH, linestyle='--', linewidth=2,
               label=f'Median (high): {median_high:.3f}')
    
    ax2.set_xlabel('Predicted Probability', fontweight='bold', fontsize=12)
    ax2.set_ylabel('Frequency', fontweight='bold', fontsize=12)
    ax2.set_title('B. Prediction Distribution\nby True Class', 
                  fontweight='bold', fontsize=13)
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)
    
    # PANEL 3: CALIBRATION BINS BREAKDOWN
    
    ax3 = axes[2]
    
    # Show counts per bin
    bin_edges = np.linspace(0, 1, 11)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Count predictions in each bin
    bin_counts = []
    bin_accuracy = []
    
    for i in range(len(bin_edges)-1):
        mask = (y_pred_proba >= bin_edges[i]) & (y_pred_proba < bin_edges[i+1])
        count = mask.sum()
        bin_counts.append(count)
        
        if count > 0:
            accuracy = y_true[mask].mean()
            bin_accuracy.append(accuracy)
        else:
            bin_accuracy.append(0)
    
    # Bar plot with color coding
    colors = ['green' if abs(bc - ba) < 0.1 else 'orange' if abs(bc - ba) < 0.2 else 'red' 
              for bc, ba in zip(bin_centers, bin_accuracy)]
    
    ax3.bar(bin_centers, bin_counts, width=0.08, alpha=0.7, 
           color=colors, edgecolor='black', linewidth=1.5)
    
    ax3.set_xlabel('Predicted Probability Bin', fontweight='bold', fontsize=12)
    ax3.set_ylabel('Number of Predictions', fontweight='bold', fontsize=12)
    ax3.set_title('C. Prediction Distribution\nAcross Bins', 
                  fontweight='bold', fontsize=13)
    ax3.grid(alpha=0.3, axis='y')
    
    # Add legend for colors
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=COLOR_TERTIARY, label='Well-calibrated (|Δ| < 0.1)'),
        Patch(facecolor=COLOR_SECONDARY, label='Moderate (0.1 ≤ |Δ| < 0.2)'),
        Patch(facecolor=COLOR_HIGH, label='Poor (|Δ| ≥ 0.2)')
    ]
    ax3.legend(handles=legend_elements, fontsize=9, loc='upper right')
    
    # PANEL 4: PRECISION-RECALL CURVE
    
    ax4 = axes[3]
    
    # Compute PR-AUC
    pr_auc = average_precision_score(y_true, y_pred_proba)
    baseline_ap = y_true.mean()  # Prevalence (random classifier performance)
    
    # Compute PR curve
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)
    
    # Plot PR curve
    ax4.plot(recall, precision, linewidth=2, color=COLOR_PRIMARY, 
             label=f'Model (AP = {pr_auc:.3f})')
    
    # Baseline (random classifier)
    ax4.plot([0, 1], [baseline_ap, baseline_ap], 'k--', 
             linewidth=1.5, label=f'Baseline (prevalence = {baseline_ap:.3f})')
    
    # Fill area under curve
    ax4.fill_between(recall, precision, alpha=0.2, color=COLOR_PRIMARY)
    
    # Labels and formatting
    ax4.set_xlabel('Recall (Sensitivity)', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Precision (PPV)', fontsize=12, fontweight='bold')
    ax4.set_title('D. Precision-Recall Curve', fontsize=13, fontweight='bold')
    ax4.set_xlim([0, 1])
    ax4.set_ylim([0, 1])
    ax4.legend(loc='best', fontsize=10)
    ax4.grid(alpha=0.3)
    
    # Add text annotation
    improvement = pr_auc - baseline_ap
    ax4.text(0.05, 0.05, f'Improvement: {improvement:+.3f}',
             transform=ax4.transAxes, fontsize=10,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"\n Calibration figure saved: {save_path}")
    
    # COMPUTE CALIBRATION METRICS
    print("COMPUTING CALIBRATION METRICS")
    
    # Brier score
    brier_score = np.mean((y_pred_proba - y_true) ** 2)
    print(f"\n Brier Score: {brier_score:.4f}")
    print(f"    (Lower is better; 0 = perfect, 0.25 = random)")
    
    # PR-AUC
    print(f"\n PR-AUC (Average Precision): {pr_auc:.4f}")
    print(f"    Baseline (random): {baseline_ap:.4f}")
    print(f"    Improvement: {improvement:+.4f}")
    
    if improvement > 0.1:
        pr_interp = "Strong predictive performance"
    elif improvement > 0.05:
        pr_interp = "Moderate predictive performance"
    else:
        pr_interp = "Weak predictive performance"
    
    print(f"    Interpretation: {pr_interp}")
    
    # Expected Calibration Error (ECE)
    def compute_ece_robust(y_true_inner, y_pred_proba_inner, n_bins=10):
        """Compute ECE with stable equal-frequency binning"""
        # Use percentiles for equal-frequency bins
        bin_edges = np.percentile(y_pred_proba_inner, np.linspace(0, 100, n_bins + 1))
        bin_edges[0] = 0.0
        bin_edges[-1] = 1.0
        bin_edges = np.unique(bin_edges)
        actual_n_bins = len(bin_edges) - 1
        
        if actual_n_bins < n_bins:
            print(f"    Warning: Only {actual_n_bins} unique bins (requested {n_bins})")
        
        bin_ids = np.digitize(y_pred_proba_inner, bin_edges[1:-1])
        ece = 0.0
        total_samples = len(y_true_inner)
        
        print(f"\n    ECE Calculation (bin-by-bin):")
        print(f"    {'Bin':>5s} | {'N':>6s} | {'Acc':>6s} | {'Conf':>6s} | {'|Diff|':>6s}")
        print(f"    {'-'*6}|{'-'*8}|{'-'*8}|{'-'*8}|{'-'*8}")
        
        for i in range(actual_n_bins):
            mask = bin_ids == i
            n_bin = mask.sum()
            
            if n_bin > 0:
                bin_acc = y_true_inner[mask].mean()
                bin_conf = y_pred_proba_inner[mask].mean()
                bin_diff = abs(bin_acc - bin_conf)
                ece += (n_bin / total_samples) * bin_diff
                print(f"    {i:5d} | {n_bin:6d} | {bin_acc:6.3f} | {bin_conf:6.3f} | {bin_diff:6.3f}")
        
        return ece, actual_n_bins
    
    # Compute ECE
    print(f"\n Expected Calibration Error (ECE):")
    ece, n_bins_used = compute_ece_robust(y_true, y_pred_proba, n_bins=10)
    
    print(f"\n    Final ECE: {ece:.4f}")
    print(f"    Number of bins used: {n_bins_used}")
    
    # Interpretation
    if ece < 0.05:
        ece_interp = "Excellent calibration"
    elif ece < 0.10:
        ece_interp = "Good calibration"
    elif ece < 0.15:
        ece_interp = "Fair calibration"
    else:
        ece_interp = "Poor calibration, consider calibration methods"
    
    print(f"    Interpretation: {ece_interp}")
    
    # SUMMARY
    print("CALIBRATION SUMMARY")
    print(f"\n  Model: {model_name}")
    print(f"  Brier Score: {brier_score:.4f}")
    print(f"  PR-AUC: {pr_auc:.4f} (improvement: {improvement:+.3f})")
    print(f"  ECE: {ece:.4f} ({ece_interp})")
    print(f"  Calibration Slope: {slope:.3f}" if not np.isnan(slope) else "  Calibration Slope: N/A")
    print(f"  Number of samples: {len(y_true)}")
    print("\n" + "="*70)
    
    # Return metrics dictionary
    return {
        'brier_score': brier_score,
        'pr_auc': pr_auc,
        'ece': ece,
        'n_bins': n_bins_used,
        'calibration_slope': slope,
        'ece_interpretation': ece_interp,
        'pr_interpretation': pr_interp,
        'n_samples': len(y_true)
    }

# Confirmation message
print(" Calibration analysis function loaded successfully")

In [ ]:
# FEATURE IMPORTANCE ANALYSIS 
print("FEATURE IMPORTANCE ANALYSIS")
import matplotlib.pyplot as plt

# CHECK VARIABLES EXIST
print("\n[Checking required variables]")

required_vars = {
    'model_final': 'Final trained model',
    'scaler_final': 'Final scaler',
    'PRIMARY_PATHWAY_COLS': 'Pathway column names',
    'PRIMARY_FEATURES': 'Feature list',
    'X': 'Feature matrix',
    'y': 'Target labels'
}

missing = []
for var_name, description in required_vars.items():
    if var_name not in globals():
        missing.append(f"{var_name} ({description})")
        print(f"   Missing: {var_name}")
    else:
        print(f"   Found: {var_name}")

if missing:
    print(f"\n ERROR: Missing variables. Run training cell first:")
    for var in missing:
        print(f"    - {var}")
    raise RuntimeError("Required variables not found. Run training cell first.")

print(f"\n All required variables present")

# PREPARE DATA
print("\n[Preparing data]")

# Get feature matrix and scale
X_full_scaled = scaler_final.transform(X)

print(f"  Features: {X.shape}")
print(f"  Scaled features: {X_full_scaled.shape}")
print(f"  Total features: {len(PRIMARY_FEATURES)}")
print(f"     Pathways: {len(PRIMARY_PATHWAY_COLS)}")
print(f"     Covariates: {len(PRIMARY_COVARIATE_COLS)}")

# COMPUTE BASELINE PERFORMANCE
print("\n[Computing baseline performance]")

model_final.eval()
device = next(model_final.parameters()).device

with torch.no_grad():
    X_tensor = torch.FloatTensor(X_full_scaled).to(device)
    baseline_pred = model_final(X_tensor).cpu().numpy().flatten()
    baseline_auc = roc_auc_score(y, baseline_pred)

print(f"  Baseline AUC (all features): {baseline_auc:.4f}")

# PERMUTATION IMPORTANCE
print("\n[Computing permutation importance]")
print(f"  Testing {len(PRIMARY_FEATURES)} features")
print(f"  Repeats per feature: 30")

n_repeats = 30
importance_results = []

for feature_idx, feature_name in enumerate(PRIMARY_FEATURES):
    # Clean feature name for display
    if feature_name in PRIMARY_PATHWAY_COLS:
        display_name = feature_name.split('_', 1)[1] if '_' in feature_name else feature_name
        display_name = display_name[:50]  # Truncate
        feature_type = 'Pathway'
    else:
        display_name = feature_name
        feature_type = 'Covariate'
    
    # Permute this feature multiple times
    aucs_permuted = []
    
    for repeat in range(n_repeats):
        # Copy data and shuffle one feature
        X_permuted = X_full_scaled.copy()
        np.random.seed(42 + repeat)  # Reproducible
        X_permuted[:, feature_idx] = np.random.permutation(X_permuted[:, feature_idx])
        
        # Predict with permuted feature
        model_final.eval()
        with torch.no_grad():
            X_perm_tensor = torch.FloatTensor(X_permuted).to(device)
            pred_permuted = model_final(X_perm_tensor).cpu().numpy().flatten()
            auc_permuted = roc_auc_score(y, pred_permuted)
        
        aucs_permuted.append(auc_permuted)
    
    # Importance = drop in performance when feature is permuted
    importance_mean = baseline_auc - np.mean(aucs_permuted)
    importance_std = np.std([baseline_auc - auc for auc in aucs_permuted])
    
    importance_results.append({
        'Feature': display_name,
        'Full_Name': feature_name,
        'Type': feature_type,
        'Importance_Mean': importance_mean,
        'Importance_Std': importance_std,
        'Baseline_AUC': baseline_auc,
        'Permuted_AUC_Mean': np.mean(aucs_permuted),
        'Permuted_AUC_Std': np.std(aucs_permuted)
    })
    
    # Progress update
    if (feature_idx + 1) % 5 == 0 or feature_idx == len(PRIMARY_FEATURES) - 1:
        print(f"    Processed {feature_idx + 1}/{len(PRIMARY_FEATURES)} features")

print("\n Permutation importance computed")

# ANALYSIS
# Create DataFrame and sort by importance
importance_df = pd.DataFrame(importance_results).sort_values('Importance_Mean', ascending=False)

print("TOP 10 MOST IMPORTANT FEATURES")
print("\n{:<45s} | {:>10s} | {:>12s} | {:>12s}".format('Feature', 'Type', 'Importance', 'Std'))
for idx, row in importance_df.head(10).iterrows():
    print("{:<45s} | {:>10s} | {:>12.4f} | {:>12.4f}".format(
        row['Feature'], row['Type'], row['Importance_Mean'], row['Importance_Std']
    ))

# Save
os.makedirs('results/tables', exist_ok=True)
importance_df.to_csv('results/tables/feature_importance.csv', index=False)
print(f"\n✓ Saved: results/tables/feature_importance.csv")

# Statistical summary

print("IMPORTANCE STATISTICS")
print(f"  Mean importance:       {importance_df['Importance_Mean'].mean():.4f}")
print(f"  Std importance:        {importance_df['Importance_Mean'].std():.4f}")
print(f"  Max importance:        {importance_df['Importance_Mean'].max():.4f} ({importance_df.iloc[0]['Feature'][:30]})")
print(f"  Min importance:        {importance_df['Importance_Mean'].min():.4f}")

# Define significance threshold
significance_threshold = 0.01
significant_count = (importance_df['Importance_Mean'] > significance_threshold).sum()
print(f"  Significant (>0.01):   {significant_count}/{len(importance_df)}")

# Breakdown by type
pathways_df = importance_df[importance_df['Type'] == 'Pathway']
covariates_df = importance_df[importance_df['Type'] == 'Covariate']

print(f"\n  Pathways analyzed:     {len(pathways_df)}")
print(f"  Covariates analyzed:   {len(covariates_df)}")

if len(covariates_df) > 0:
    print(f"\n  Top covariate: {covariates_df.iloc[0]['Feature']:20s} | {covariates_df.iloc[0]['Importance_Mean']:+.4f}")

# VISUALISATION

print("\n[Creating visualisation]")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Bar plot of top features
top_n = min(15, len(importance_df))
top_features = importance_df.head(top_n)

y_pos = np.arange(len(top_features))
colors = [COLOR_PRIMARY if t == 'Pathway' else COLOR_SECONDARY for t in top_features['Type']]

axes[0].barh(y_pos, top_features['Importance_Mean'], 
            xerr=top_features['Importance_Std'],
            color=colors, edgecolor='black', linewidth=0.5, 
            alpha=0.7, error_kw={'linewidth': 1.5, 'capsize': 3})
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels([f[:40] for f in top_features['Feature']], fontsize=8)
axes[0].set_xlabel('Permutation Importance (ΔAUC)', fontsize=12, fontweight='bold')
axes[0].set_title(f'A. Top {len(top_features)} Most Important Features\n(Data-Driven Pathways)', 
                  fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3, axis='x')
axes[0].invert_yaxis()
axes[0].axvline(x=0, color='black', linewidth=1)

# Add significance threshold line
axes[0].axvline(x=significance_threshold, color=COLOR_REFERENCE, linestyle='--', 
               linewidth=2, alpha=0.7, label=f'Threshold ({significance_threshold})')

# Legend for colors
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLOR_PRIMARY, label='Pathway'),
    Patch(facecolor=COLOR_SECONDARY, label='Covariate'),
    plt.Line2D([0], [0], color=COLOR_REFERENCE, linestyle='--', linewidth=2, label='Threshold')
]
axes[0].legend(handles=legend_elements, fontsize=9, loc='lower right')

# Panel B: Distribution of pathway importances only
pathways_importance = importance_df[importance_df['Type'] == 'Pathway']['Importance_Mean']

axes[1].hist(pathways_importance, bins=12, 
            color=COLOR_PRIMARY, edgecolor='black', alpha=0.7)
axes[1].axvline(x=pathways_importance.mean(), 
               color=COLOR_HIGH, linestyle='--', linewidth=2, label='Mean')
axes[1].axvline(x=significance_threshold, 
               color=COLOR_SECONDARY, linestyle='--', linewidth=2, label='Threshold')
axes[1].set_xlabel('Permutation Importance (ΔAUC)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Pathways', fontsize=12, fontweight='bold')
axes[1].set_title('B. Distribution of Pathway Importance', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()

os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f" Saved: results/figures/feature_importance.png")

# DETAILED RESULTS
# Create detailed results with rankings
importance_df['Rank'] = range(1, len(importance_df) + 1)
importance_df['Significant'] = importance_df['Importance_Mean'] > significance_threshold

# Save detailed results
importance_df.to_csv('results/tables/feature_importance_detailed.csv', index=False)
print(f" Saved: results/tables/feature_importance_detailed.csv")

print(" FEATURE IMPORTANCE ANALYSIS COMPLETE")


# Summary of significant features
print(f"\n[Significant Features (Importance > {significance_threshold})]")
significant_features = importance_df[importance_df['Significant']]

if len(significant_features) > 0:
    print(f"  Found {len(significant_features)} significant features:\n")
    for idx, row in significant_features.iterrows():
        status = "★" if row['Rank'] <= 3 else " "
        print(f"  {status} {row['Rank']:2d}. {row['Feature']:45s} | "
              f"{row['Type']:10s} | {row['Importance_Mean']:+.4f} ± {row['Importance_Std']:.4f}")
else:
    print("   No features exceed significance threshold")
    print(f"    Consider lowering threshold or investigating model performance")

In [ ]:
# CHECK AND LOAD SPATIAL COORDINATES

print("SPATIAL COORDINATES")

if 'spatial' not in adata.obsm.keys():
    print(f"\n Spatial coordinates not found")
    print(f"\nOptions:")
    print(f"   Load from Visium spatial/ folder ")
    print(f"   Use UMAP coordinates as proxy")
    print(f"   Skip spatial visualization ")
    
    # Option: Use UMAP as proxy
    if 'X_umap' in adata.obsm.keys():
        print(f"\n  Using UMAP coordinates as spatial proxy")
        adata.obsm['spatial'] = adata.obsm['X_umap'].copy()
        print(f"   Spatial coordinates set (UMAP proxy)")
    else:
        print(f"\n   No UMAP coordinates either")
else:
    print(f"\n Spatial coordinates already present")
    print(f"  Shape: {adata.obsm['spatial'].shape}")


In [ ]:
# SIMPLIFIED VULNERABILITY VISUALISATION 

print("VULNERABILITY VISUALISATION")

from scipy.stats import mannwhitneyu, spearmanr

# Figure 1: Basic Distributions

print("\n[Figure 1] Distribution analysis")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Histogram
vulnerability = adata.obs['dl_vulnerability_score'].values
axes[0, 0].hist(vulnerability, bins=50, edgecolor='black', alpha=0.7, color=COLOR_PRIMARY)
axes[0, 0].axvline(np.median(vulnerability), color=COLOR_HIGH, linestyle='--', linewidth=2, 
                   label=f'Median: {np.median(vulnerability):.3f}')
axes[0, 0].axvline(np.percentile(vulnerability, 75), color=COLOR_SECONDARY, linestyle='--', linewidth=2,
                   label=f'75th: {np.percentile(vulnerability, 75):.3f}')
axes[0, 0].set_xlabel('Vulnerability Score', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Distribution of Vulnerability Scores', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Panel B: UMAP colored by vulnerability
if 'X_umap' in adata.obsm.keys():
    sc.pl.umap(adata, color='dl_vulnerability_score', ax=axes[0, 1], cmap=CMAP_SEQUENTIAL,
               title='UMAP: Vulnerability', show=False, s=30)
else:
    axes[0, 1].text(0.5, 0.5, 'UMAP not available\n(Run UMAP cell first)', 
                   ha='center', va='center', fontsize=12,
                   transform=axes[0, 1].transAxes)
    axes[0, 1].set_title('UMAP: Vulnerability', fontsize=12, fontweight='bold')

# Panel C: Box plot by proliferation
high_prolif = adata.obs.loc[adata.obs['target_proliferation_binary']==1, 'dl_vulnerability_score']
low_prolif = adata.obs.loc[adata.obs['target_proliferation_binary']==0, 'dl_vulnerability_score']

box_data = [low_prolif, high_prolif]
bp = axes[1, 0].boxplot(box_data, labels=['Low\nProliferation', 'High\nProliferation'],
                        patch_artist=True)
bp['boxes'][0].set_facecolor(COLOR_LOW)
bp['boxes'][1].set_facecolor(COLOR_HIGH)

axes[1, 0].set_ylabel('Vulnerability Score', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Vulnerability by Proliferation Status', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3, axis='y')

# Add statistics
stat, pval = mannwhitneyu(high_prolif, low_prolif, alternative='greater')
axes[1, 0].text(0.5, 0.95, f'p = {pval:.2e}', transform=axes[1, 0].transAxes,
                ha='center', va='top', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

# Panel D: Scatter plot
if 'target_proliferation' in adata.obs.columns:
    axes[1, 1].scatter(
        adata.obs['target_proliferation'],
        adata.obs['dl_vulnerability_score'],
        c=adata.obs['target_proliferation_binary'],
        cmap=CMAP_DIVERGING,
        alpha=0.5,
        s=10
    )
    axes[1, 1].set_xlabel('Proliferation Score', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Vulnerability Score', fontsize=12, fontweight='bold')
    axes[1, 1].set_title('Vulnerability vs Proliferation', fontsize=12, fontweight='bold')
    axes[1, 1].grid(alpha=0.3)
    
    corr, pval_corr = spearmanr(adata.obs['target_proliferation'], adata.obs['dl_vulnerability_score'])
    axes[1, 1].text(0.05, 0.95, f'r = {corr:.3f}\np = {pval_corr:.2e}',
                    transform=axes[1, 1].transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
else:
    # Use binary proliferation instead
    axes[1, 1].scatter(
        adata.obs['target_proliferation_binary'],
        adata.obs['dl_vulnerability_score'],
        c=adata.obs['target_proliferation_binary'],
        cmap=CMAP_DIVERGING,
        alpha=0.5,
        s=20
    )
    axes[1, 1].set_xlabel('Proliferation (Binary)', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Vulnerability Score', fontsize=12, fontweight='bold')
    axes[1, 1].set_title('Vulnerability vs Proliferation', fontsize=12, fontweight='bold')
    axes[1, 1].set_xticks([0, 1])
    axes[1, 1].set_xticklabels(['Low', 'High'])
    axes[1, 1].grid(alpha=0.3)

plt.tight_layout()

os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/dl_vulnerability_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f" Saved: results/figures/dl_vulnerability_analysis.png")

# Statistical Validation

print("STATISTICAL VALIDATION")

print(f"\n[Association with Proliferation]")
print(f"  High proliferation: {high_prolif.mean():.3f} ± {high_prolif.std():.3f}")
print(f"  Low proliferation:  {low_prolif.mean():.3f} ± {low_prolif.std():.3f}")
print(f"  Fold-change: {high_prolif.mean() / low_prolif.mean():.2f}x")
print(f"  P-value (Mann-Whitney U): {pval:.2e}")

if pval < 0.05:
    print(f"   VALIDATED: Vulnerability significantly higher in high proliferation")
else:
    print(f"   Not significant")

# Pathway enrichment in high-risk regions
print(f"\n[Pathway Enrichment in High-Risk Regions]")

# Use data-driven pathway columns
pathway_cols_to_analyze = PRIMARY_PATHWAY_COLS.copy()

print(f"  Analyzing {len(pathway_cols_to_analyze)} pathways")

enrichment_results = []

for pathway_col in pathway_cols_to_analyze:
    # Clean pathway name
    pathway_name = pathway_col.split('_', 1)[1] if '_' in pathway_col else pathway_col
    pathway_name = pathway_name[:50]  # Truncate
    
    high_risk = adata.obs.loc[adata.obs['high_vulnerability']==1, pathway_col]
    low_risk = adata.obs.loc[adata.obs['high_vulnerability']==0, pathway_col]
    
    stat, pval_pathway = mannwhitneyu(high_risk, low_risk, alternative='two-sided')
    fc = high_risk.mean() / low_risk.mean() if low_risk.mean() != 0 else 0
    
    enrichment_results.append({
        'Pathway': pathway_name,
        'Full_Name': pathway_col,
        'High_Risk_Mean': high_risk.mean(),
        'Low_Risk_Mean': low_risk.mean(),
        'Fold_Change': fc,
        'P_Value': pval_pathway
    })

enrichment_df = pd.DataFrame(enrichment_results).sort_values('Fold_Change', ascending=False)

# FDR correction
from statsmodels.stats.multitest import multipletests
reject, qvals, _, _ = multipletests(enrichment_df['P_Value'], method='fdr_bh', alpha=0.05)
enrichment_df['Q_Value'] = qvals
enrichment_df['Significant_FDR'] = reject

print(f"\n  Top 5 enriched in high-risk:")
for i, (_, row) in enumerate(enrichment_df.head(5).iterrows(), 1):
    sig = "***" if row['Significant_FDR'] else ("*" if row['P_Value'] < 0.05 else "")
    print(f"  {i}. {row['Pathway']:45s} | FC={row['Fold_Change']:.2f}x, "
          f"p={row['P_Value']:.4f}, q={row['Q_Value']:.4f} {sig}")

print(f"\n  Legend: *** = FDR q < 0.05, * = p < 0.05")

# Save results
os.makedirs('results/tables', exist_ok=True)
enrichment_df.to_csv('results/tables/pathway_enrichment_vulnerability.csv', index=False)
print(f"\n Saved: results/tables/pathway_enrichment_vulnerability.csv")

# Summary statistics
print(f"\n[Enrichment Summary]")
print(f"  Total pathways tested: {len(enrichment_df)}")
print(f"  Nominally significant (p < 0.05): {(enrichment_df['P_Value'] < 0.05).sum()}")
print(f"  FDR significant (q < 0.05): {enrichment_df['Significant_FDR'].sum()}")
print(f"  Enriched in high-risk (FC > 1.0): {(enrichment_df['Fold_Change'] > 1.0).sum()}")
print(f"  Enriched in low-risk (FC < 1.0): {(enrichment_df['Fold_Change'] < 1.0).sum()}")

In [ ]:
# PERFORM CALIBRATION ANALYSIS
print("RUNNING CALIBRATION ANALYSIS")

y_true = y  # Your true labels
y_pred_proba = vulnerability_scores  

# Run calibration analysis
calibration_results = plot_calibration_analysis(
    y_true=y_true,
    y_pred_proba=y_pred_proba,
    model_name="Metabolic Vulnerability Model",
    save_path='figures/calibration_analysis.png'
)

# Save calibration metrics
calibration_metrics_df = pd.DataFrame([{
    'model': 'Neural Network',
    'brier_score': calibration_results['brier_score'],
    'ece': calibration_results['ece'],
    'pr_auc': calibration_results['pr_auc'],
    'calibration_slope': calibration_results['calibration_slope'],
    'interpretation': 'Good' if calibration_results['ece'] < 0.10 else 'Needs improvement'
}])

calibration_metrics_df.to_csv('results/tables/calibration_metrics.csv', index=False)
print(f"\n Calibration metrics saved: results/tables/calibration_metrics.csv")

In [ ]:
#Check Proliferation Columns

print("Checking Proliferation Columns")


# Find all proliferation-related columns
prolif_cols = [col for col in adata.obs.columns if 'prolif' in col.lower()]

print(f"\nProliferation columns found ({len(prolif_cols)}):")
for col in prolif_cols:
    print(f"  - {col}")
    print(f"    Dtype: {adata.obs[col].dtype}")
    print(f"    Unique values: {adata.obs[col].unique()}")
    if len(adata.obs[col].unique()) <= 10:  # Only show value counts if not too many
        print(f"    Value counts:")
        print(adata.obs[col].value_counts())
    print()

# Determine the correct column
if 'target_proliferation_binary' in adata.obs.columns:
    PROLIF_COL = 'target_proliferation_binary'
    print(f" Using column: {PROLIF_COL}")
elif 'proliferation_high' in adata.obs.columns:
    PROLIF_COL = 'proliferation_high'
    print(f" Using column: {PROLIF_COL}")
else:
    print(" WARNING: No proliferation column found!")
    PROLIF_COL = None

# Check global distribution
if PROLIF_COL:
    n_high = (adata.obs[PROLIF_COL].astype(int) == 1).sum()
    n_total = len(adata.obs)
    pct_high = 100.0 * n_high / n_total
    
    print(f"\nGlobal proliferation distribution:")
    print(f"  High proliferation: {n_high}/{n_total} ({pct_high:.1f}%)")
    print(f"  Low proliferation: {n_total - n_high}/{n_total} ({100-pct_high:.1f}%)")

In [ ]:
# METABOLIC NICHE CHARACTERIZATION & ANALYSIS

print(" " * 15 + "METABOLIC NICHE CHARACTERIZATION")

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.stats import mannwhitneyu, chi2_contingency
import seaborn as sns

# PREPARE DATA FOR CLUSTERING
print("\n[Preparing pathway data for clustering]")

pathway_cols = PRIMARY_PATHWAY_COLS.copy()
print(f"  Pathways available: {len(pathway_cols)}")

# Extract pathway features
X_pathways = adata.obs[pathway_cols].values
print(f"  Data shape: {X_pathways.shape}")

# Standardize features
scaler = StandardScaler()
X_pathways_scaled = scaler.fit_transform(X_pathways)
print(f"   Features standardized")

# DETERMINE OPTIMAL NUMBER OF CLUSTERS
print("\n[Determining optimal number of clusters]")

# Test different numbers of clusters
k_range = range(2, 8)
inertias = []
silhouette_scores = []

from sklearn.metrics import silhouette_score

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=CFG.RANDOM_SEED, n_init=20)
    labels = kmeans.fit_predict(X_pathways_scaled)
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_pathways_scaled, labels)
    silhouette_scores.append(sil_score)
    print(f"  k={k}: inertia={kmeans.inertia_:.2f}, silhouette={sil_score:.3f}")

# Plot elbow curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(k_range, inertias, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Inertia', fontsize=12, fontweight='bold')
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].grid(alpha=0.3)

# Silhouette plot
axes[1].plot(k_range, silhouette_scores, 'o-', linewidth=2, markersize=8, color=COLOR_TERTIARY)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_title('Silhouette Score', fontsize=13, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()

os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/niche_cluster_selection.png', dpi=300, bbox_inches='tight')
plt.show()

print("   Saved: results/figures/niche_cluster_selection.png")

# Select k=4 based on elbow and biological interpretability
optimal_k = 4
print(f"\n  Selected k = {optimal_k} clusters")

# PERFORM K-MEANS CLUSTERING
print("\n[Performing K-means clustering]")

# Final clustering
kmeans = KMeans(n_clusters=optimal_k, random_state=CFG.RANDOM_SEED, n_init=50)
niche_labels = kmeans.fit_predict(X_pathways_scaled)

# Add to adata
adata.obs['metabolic_niche'] = niche_labels
adata.obs['metabolic_niche'] = adata.obs['metabolic_niche'].astype('category')

print(f"   Clustering complete")
print(f"\n  Cluster sizes:")
for i in range(optimal_k):
    n_spots = (niche_labels == i).sum()
    pct = 100 * n_spots / len(niche_labels)
    print(f"    Niche {i}: {n_spots:4d} spots ({pct:5.1f}%)")

# CHARACTERIZE EACH NICHE
print("\n[Characterizing metabolic niches]")

# Compute mean pathway scores for each niche
niche_profiles = []

for niche_id in range(optimal_k):
    niche_mask = (niche_labels == niche_id)
    niche_data = adata.obs[niche_mask]
    
    profile = {
        'Niche': niche_id,
        'N_Spots': niche_mask.sum()
    }
    
    # Mean pathway scores
    for pathway_col in pathway_cols:
        # Clean pathway name
        pathway_name = pathway_col.split('_', 1)[1] if '_' in pathway_col else pathway_col
        profile[pathway_name] = niche_data[pathway_col].mean()
    
    # Proliferation enrichment
    if 'target_proliferation_binary' in adata.obs.columns:
        niche_prolif = niche_data['target_proliferation_binary'].astype(int)
        n_high = (niche_prolif == 1).sum()
        n_total = len(niche_data)
        pct_high_prolif = 100.0 * n_high / n_total
        profile['Pct_High_Proliferation'] = pct_high_prolif
    else:
        profile['Pct_High_Proliferation'] = np.nan
    
    niche_profiles.append(profile)

niche_df = pd.DataFrame(niche_profiles)

# Identify top pathways for each niche
print("\n  Niche characterisation:")

# Get pathway names (cleaned)
pathway_names = [col.split('_', 1)[1] if '_' in col else col for col in pathway_cols]

for idx, row in niche_df.iterrows():
    niche_id = int(row['Niche'])
    n_spots = int(row['N_Spots'])
    
    # Get pathway scores for this niche (use cleaned names)
    pathway_scores = {}
    for orig_col, clean_name in zip(pathway_cols, pathway_names):
        clean_key = orig_col.split('_', 1)[1] if '_' in orig_col else orig_col
        if clean_key in row:
            pathway_scores[clean_key] = row[clean_key]
    
    # Top 3 pathways
    top_pathways = sorted(pathway_scores.items(), key=lambda x: x[1], reverse=True)[:3]
    
    print(f"\n  Niche {niche_id}: {n_spots} spots")
    print(f"    Top pathways:")
    for pathway, score in top_pathways:
        pathway_short = pathway[:45] if len(pathway) > 45 else pathway
        print(f"       {pathway_short:<45s}: {score:6.3f}")
    
    if not np.isnan(row['Pct_High_Proliferation']):
        print(f"    High proliferation: {row['Pct_High_Proliferation']:.1f}%")

# STATISTICAL TESTS FOR NICHE DIFFERENCES
print("\n[Testing pathway differences between niches]")

# Test each pathway for differences across niches
pathway_tests = []

for pathway_col in pathway_cols:
    # Clean pathway name
    pathway_name = pathway_col.split('_', 1)[1] if '_' in pathway_col else pathway_col
    pathway_name = pathway_name[:50]  # Truncate
    
    # Get values for each niche
    niche_values = [adata.obs[adata.obs['metabolic_niche'] == i][pathway_col].values 
                    for i in range(optimal_k)]
    
    # Kruskal-Wallis test (non-parametric ANOVA)
    from scipy.stats import kruskal
    stat, p_value = kruskal(*niche_values)
    
    # Calculate means
    niche_means = [vals.mean() for vals in niche_values]
    
    pathway_tests.append({
        'Pathway': pathway_name,
        'Full_Name': pathway_col,
        'Kruskal_H': stat,
        'P_Value': p_value,
        'Significant': 'Yes' if p_value < 0.05 else 'No',
        'Max_Niche_Mean': max(niche_means),
        'Min_Niche_Mean': min(niche_means),
        'Range': max(niche_means) - min(niche_means)
    })

pathway_tests_df = pd.DataFrame(pathway_tests).sort_values('P_Value')

print(f"\n  Pathways with significant niche differences (p < 0.05):")
sig_pathways = pathway_tests_df[pathway_tests_df['P_Value'] < 0.05]
for idx, row in sig_pathways.head(10).iterrows():
    print(f"    {row['Pathway']:<45s}: p = {row['P_Value']:.4f}")

os.makedirs('results/tables', exist_ok=True)
pathway_tests_df.to_csv('results/tables/niche_pathway_differences.csv', index=False)
print(f"\n   Saved: results/tables/niche_pathway_differences.csv")

# VISUALISATION
print("\n[Creating niche visualisation]")

# Create comprehensive figure
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35)

# Panel A: Spatial map of niches
ax1 = fig.add_subplot(gs[0, :2])

colors_palette = PALETTE  # unified categorical palette
niche_colors = [colors_palette[i] for i in adata.obs['metabolic_niche']]

# Get spatial coordinates
if 'spatial' in adata.obsm.keys():
    x_coords = adata.obsm['spatial'][:, 0]
    y_coords = adata.obsm['spatial'][:, 1]
else:
    print("   Warning: No spatial coordinates found, using indices")
    x_coords = np.arange(adata.n_obs)
    y_coords = np.arange(adata.n_obs)

scatter = ax1.scatter(x_coords, y_coords, c=niche_colors, s=5, alpha=0.7, edgecolors='none')

# Create legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors_palette[i], label=f'Niche {i}') for i in range(optimal_k)]
ax1.legend(handles=legend_elements, loc='upper left', fontsize=10)
ax1.set_title('A. Spatial Distribution of Metabolic Niches', fontsize=14, fontweight='bold')
ax1.set_xlabel('X coordinate', fontsize=11)
ax1.set_ylabel('Y coordinate', fontsize=11)
ax1.invert_yaxis()

# Panel B: Niche sizes
ax2 = fig.add_subplot(gs[0, 2])

niche_sizes = [(niche_labels == i).sum() for i in range(optimal_k)]
bars = ax2.bar(range(optimal_k), niche_sizes, color=colors_palette, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Number of Spots', fontsize=11, fontweight='bold')
ax2.set_title('B. Niche Sizes', fontsize=13, fontweight='bold')
ax2.set_xticks(range(optimal_k))
ax2.set_xticklabels([f'N{i}' for i in range(optimal_k)])
ax2.grid(alpha=0.3, axis='y')

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Panel C: Pathway profiles heatmap
ax3 = fig.add_subplot(gs[1, :])

# Create matrix for heatmap
heatmap_data = []
for i in range(optimal_k):
    niche_mask = (adata.obs['metabolic_niche'] == i)
    pathway_means = [adata.obs[niche_mask][col].mean() for col in pathway_cols]
    heatmap_data.append(pathway_means)

heatmap_data = np.array(heatmap_data)

# Pathway names for heatmap (cleaned and truncated)
pathway_names_short = [col.split('_', 1)[1][:35] if '_' in col else col[:35] for col in pathway_cols]
niche_names = [f"Niche {i}" for i in range(optimal_k)]

sns.heatmap(heatmap_data, 
            xticklabels=pathway_names_short,
            yticklabels=niche_names,
            cmap=CMAP_DIVERGING,
            center=0,
            cbar_kws={'label': 'Mean Pathway Score'},
            ax=ax3,
            linewidths=0.5,
            linecolor=COLOR_REFERENCE)

ax3.set_title('C. Pathway Activity Profiles by Niche', fontsize=14, fontweight='bold')
ax3.set_xlabel('Metabolic Pathway', fontsize=11, fontweight='bold')
ax3.set_ylabel('Metabolic Niche', fontsize=11, fontweight='bold')
plt.setp(ax3.get_xticklabels(), rotation=45, ha='right', fontsize=8)

# Panel D: Proliferation by niche
ax4 = fig.add_subplot(gs[2, 0])

if 'target_proliferation_binary' in adata.obs.columns:
    prolif_pcts = []
    for i in range(optimal_k):
        niche_mask = (adata.obs['metabolic_niche'] == i)
        pct = (adata.obs[niche_mask]['target_proliferation_binary'] == 1).sum() / niche_mask.sum() * 100
        prolif_pcts.append(pct)
    
    bars = ax4.bar(range(optimal_k), prolif_pcts, color=colors_palette, edgecolor='black', linewidth=1.5)
    ax4.set_ylabel('% High Proliferation', fontsize=11, fontweight='bold')
    ax4.set_title('D. Proliferation Enrichment', fontsize=13, fontweight='bold')
    ax4.set_xticks(range(optimal_k))
    ax4.set_xticklabels([f'N{i}' for i in range(optimal_k)])
    ax4.set_ylim([0, 100])
    ax4.axhline(y=50, color=COLOR_REFERENCE, linestyle='--', linewidth=1.5, alpha=0.5)
    ax4.grid(alpha=0.3, axis='y')
    
    # Add value labels
    for bar, val in zip(bars, prolif_pcts):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
else:
    ax4.text(0.5, 0.5, 'No proliferation data found', 
             transform=ax4.transAxes, ha='center', va='center', fontsize=12)
    ax4.set_title('D. Proliferation Enrichment', fontsize=13, fontweight='bold')

# Panel E: Top pathways per niche
ax5 = fig.add_subplot(gs[2, 1:])

# For each niche, get top 3 pathways
top_pathways_text = []
for i in range(optimal_k):
    niche_mask = (adata.obs['metabolic_niche'] == i)
    
    pathway_means = []
    for col in pathway_cols:
        clean_name = col.split('_', 1)[1][:30] if '_' in col else col[:30]
        mean_val = adata.obs[niche_mask][col].mean()
        pathway_means.append((clean_name, mean_val))
    
    pathway_means.sort(key=lambda x: x[1], reverse=True)
    top_3 = pathway_means[:3]
    
    text = f"Niche {i}:\n"
    for j, (pathway, score) in enumerate(top_3, 1):
        text += f"  {j}. {pathway[:28]} ({score:.3f})\n"
    
    top_pathways_text.append(text)

ax5.text(0.05, 0.95, '\n'.join(top_pathways_text), 
         transform=ax5.transAxes,
         fontsize=9,
         verticalalignment='top',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

ax5.set_title('E. Top Pathways per Niche', fontsize=13, fontweight='bold')
ax5.axis('off')

plt.savefig('results/figures/metabolic_niches_comprehensive.png', dpi=300, bbox_inches='tight')
plt.show()

print("   Saved: results/figures/metabolic_niches_comprehensive.png")

In [ ]:
# STATISTICAL TESTING OF NICHE-PROLIFERATION ASSOCIATION
print("STATISTICAL TESTING: NICHE vs PROLIFERATION")

from scipy.stats import chi2_contingency

# Create contingency table
contingency = pd.crosstab(
    adata.obs['metabolic_niche'],
    adata.obs['target_proliferation_binary'],
    margins=True  
)

print("\nContingency Table:")
print(contingency)

# Chi-square test
chi2, p_value, dof, expected = chi2_contingency(contingency.iloc[:-1, :-1])  # Exclude margins

print(f"\nChi-square test:")
print(f"  χ² = {chi2:.2f}")
print(f"  df = {dof}")
print(f"  p-value = {p_value:.2e}")
print(f"  {' Significant' if p_value < 0.05 else '✗ Not significant'}")

# Effect size (Cramér's V)
n = contingency.iloc[:-1, :-1].sum().sum()
min_dim = min(contingency.shape[0] - 1, contingency.shape[1] - 1)
cramers_v = np.sqrt(chi2 / (n * min_dim))

print(f"\nEffect Size (Cramér's V): {cramers_v:.3f}")
if cramers_v < 0.1:
    print("  Interpretation: Negligible association")
elif cramers_v < 0.3:
    print("  Interpretation: Weak association")
elif cramers_v < 0.5:
    print("  Interpretation: Moderate association")
else:
    print("  Interpretation: Strong association")

# Pairwise comparisons (post-hoc)
print("\nPost-hoc pairwise comparisons (Mann-Whitney U):")
from scipy.stats import mannwhitneyu

for niche_id in range(optimal_k):
    niche_mask = adata.obs['metabolic_niche'] == niche_id
    niche_prolif = adata.obs.loc[niche_mask, 'target_proliferation_binary'].astype(int)
    other_prolif = adata.obs.loc[~niche_mask, 'target_proliferation_binary'].astype(int)
    
    u_stat, p_val = mannwhitneyu(niche_prolif, other_prolif, alternative='two-sided')
    
    niche_mean = niche_prolif.mean()
    other_mean = other_prolif.mean()
    diff = niche_mean - other_mean
    
    print(f"  Niche {niche_id} vs Others: "
          f"Δ = {diff:+.3f}, p = {p_val:.3e} {'*' if p_val < 0.05 else ''}")

In [ ]:
#  VISUALISATION: Niche Spatial Maps

print("\n Creating niche visualisation")

# Use the correct variable name from niche analysis
n_niches = optimal_k  

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Panel A: Spatial niches (MANUAL SCATTER PLOT)
if 'spatial' in adata.obsm.keys():
    coords = adata.obsm['spatial']
    niches = adata.obs['metabolic_niche'].astype(int)
    
    # Color palette
    colors_niche = np.array([cat_color(n) for n in niches])
    
    scatter = axes[0, 0].scatter(coords[:, 0], coords[:, 1], 
                                 c=niches, cmap=CMAP_CATEGORICAL, s=15, alpha=0.8)
    axes[0, 0].set_title('A. Metabolic Niches (Spatial)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Spatial X')
    axes[0, 0].set_ylabel('Spatial Y')
    axes[0, 0].axis('equal')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=cat_color(i), label=f'Niche {i}') 
                      for i in range(n_niches)]
    axes[0, 0].legend(handles=legend_elements, loc='best', fontsize=9)
else:
    axes[0, 0].text(0.5, 0.5, 'No spatial data', ha='center', va='center',
                   transform=axes[0, 0].transAxes, fontsize=14)

# Panel B: UMAP niches
if 'X_umap' in adata.obsm.keys():
    sc.pl.umap(adata, color='metabolic_niche', ax=axes[0, 1],
              show=False, title='B. Metabolic Niches (UMAP)', s=30,
              palette=CLUSTER_COLORS)
else:
    axes[0, 1].text(0.5, 0.5, 'No UMAP', ha='center', va='center',
                   transform=axes[0, 1].transAxes, fontsize=14)

# Panel C: Niche pathway heatmap
pathway_matrix = []
for niche_id in range(n_niches):
    # FIX: Compare with integer, not string
    niche_mask = adata.obs['metabolic_niche'] == niche_id
    pathway_means = [adata.obs.loc[niche_mask, col].mean() 
                    for col in pathway_cols]
    pathway_matrix.append(pathway_means)

pathway_names_short = [col.replace('pathway_', '').replace('_neighbor_mean', ' (nbr)') 
                      for col in pathway_cols]

im = axes[1, 0].imshow(np.array(pathway_matrix).T, aspect='auto', cmap=CMAP_DIVERGING)
axes[1, 0].set_xticks(range(n_niches))
axes[1, 0].set_xticklabels([f'Niche {i}' for i in range(n_niches)])
axes[1, 0].set_yticks(range(len(pathway_cols)))
axes[1, 0].set_yticklabels(pathway_names_short, fontsize=8)
axes[1, 0].set_title('C. Pathway Profiles by Niche', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[1, 0], label='Mean Pathway Score')

# Panel D: Niche proliferation enrichment
if 'target_proliferation_binary' in adata.obs.columns:
    prolif_by_niche = []
    for i in range(n_niches):
        # FIX: Compare with integer, not string
        niche_mask = adata.obs['metabolic_niche'] == i
        prolif_mean = adata.obs.loc[niche_mask, 'target_proliferation_binary'].mean()
        prolif_by_niche.append(prolif_mean)
    
    colors_prolif = [COLOR_HIGH if p > 0.6 else COLOR_SECONDARY if p > 0.5 else COLOR_LOW 
                     for p in prolif_by_niche]
    
    bars = axes[1, 1].bar(range(n_niches), prolif_by_niche, color=colors_prolif,
                          edgecolor='black', linewidth=1.5)
    axes[1, 1].axhline(y=0.5, color=COLOR_REFERENCE, linestyle='--', linewidth=1, label='Baseline')
    axes[1, 1].set_xlabel('Metabolic Niche', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Proportion High Proliferation', fontsize=12, fontweight='bold')
    axes[1, 1].set_title('D. Proliferation Enrichment by Niche', fontsize=12, fontweight='bold')
    axes[1, 1].set_xticks(range(n_niches))
    axes[1, 1].set_xticklabels([f'Niche {i}' for i in range(n_niches)])
    axes[1, 1].set_ylim([0, 1])
    axes[1, 1].grid(alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, prolif_by_niche)):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                       f'{val:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
else:
    axes[1, 1].text(0.5, 0.5, 'No proliferation data', ha='center', va='center',
                   transform=axes[1, 1].transAxes, fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(Paths.FIGURES, 'metabolic_niche_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"   Saved: figures/metabolic_niche_analysis.png")

In [ ]:
# ENRICHMENT FIGURE

print("CREATING PATHWAY ENRICHMENT FIGURE")

from scipy.stats import mannwhitneyu
import pandas as pd

# COMPUTE ENRICHMENT FOR ALL DATA-DRIVEN PATHWAYS
print("\n[Computing enrichment statistics]")

# Check if high_vulnerability exists
if 'high_vulnerability' not in adata.obs.columns:
    print("   Creating 'high_vulnerability' from vulnerability scores")
    threshold = np.percentile(adata.obs['dl_vulnerability_score'], 75)
    adata.obs['high_vulnerability'] = (adata.obs['dl_vulnerability_score'] > threshold).astype(int)
    print(f"   Created (top 25% = high vulnerability)")

enrichment_stats = []

for pathway_col in PRIMARY_PATHWAY_COLS:
    # Clean pathway name
    pathway_name = pathway_col.split('_', 1)[1] if '_' in pathway_col else pathway_col
    pathway_name_short = pathway_name[:40]
    
    # Get values for high vs low vulnerability
    high_vuln = adata.obs[adata.obs['high_vulnerability'] == 1][pathway_col].values
    low_vuln = adata.obs[adata.obs['high_vulnerability'] == 0][pathway_col].values
    
    # Statistical test
    stat, pval = mannwhitneyu(high_vuln, low_vuln, alternative='two-sided')
    
    # Effect size (Cohen's d)
    mean_high = np.mean(high_vuln)
    mean_low = np.mean(low_vuln)
    std_pooled = np.sqrt((np.var(high_vuln) + np.var(low_vuln)) / 2)
    cohens_d = (mean_high - mean_low) / std_pooled if std_pooled > 0 else 0
    
    # Absolute difference
    abs_diff = mean_high - mean_low
    
    enrichment_stats.append({
        'Pathway': pathway_name_short,
        'Full_Name': pathway_col,
        'Cohens_d': cohens_d,
        'Abs_Diff': abs_diff,
        'P_Value': pval,
        'Mean_High': mean_high,
        'Mean_Low': mean_low
    })

enrichment_df = pd.DataFrame(enrichment_stats)

# Sort by absolute effect size
enrichment_df['Abs_Cohens_d'] = np.abs(enrichment_df['Cohens_d'])
enrichment_df = enrichment_df.sort_values('Abs_Cohens_d', ascending=False)

print(f"   Computed enrichment for {len(enrichment_df)} pathways")

# SELECT TOP PATHWAYS FOR VISUALISATION
print("\n[Selecting top pathways for visualisation]")

# Get top 5 by absolute effect size
top_n = 5
top_pathways = enrichment_df.head(top_n)

print(f"\n  Top {top_n} pathways by effect size:")
for i, (_, row) in enumerate(top_pathways.iterrows(), 1):
    direction = "↑" if row['Cohens_d'] > 0 else "↓"
    print(f"  {i}. {direction} {row['Pathway']:40s} | d={row['Cohens_d']:+.3f}, p={row['P_Value']:.2e}")

# SAVE RESULTS
os.makedirs('results/tables', exist_ok=True)
enrichment_df.to_csv('results/tables/pathway_enrichment_vulnerability_stats.csv', index=False)
print(f"\n   Saved: results/tables/pathway_enrichment_vulnerability_stats.csv")

# CREATE VISUALISATION
print("\n[Creating enrichment figure]")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Prepare data for plotting
pathways_plot = [p[:30] for p in top_pathways['Pathway']]  # Truncate names
effect_sizes = top_pathways['Cohens_d'].values
abs_diffs = top_pathways['Abs_Diff'].values
p_values = top_pathways['P_Value'].values
log_p_values = [-np.log10(p) for p in p_values]

# Colors based on direction
colors_plot = [COLOR_HIGH if d > 0 else COLOR_LOW for d in effect_sizes]

# Panel A: Effect Sizes (Cohen's d)
y_pos = np.arange(len(pathways_plot))
axes[0].barh(y_pos, effect_sizes, color=colors_plot, edgecolor='black', linewidth=0.5)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(pathways_plot, fontsize=9)
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=1.5)
axes[0].set_xlabel("Cohen's d (Effect Size)", fontsize=12, fontweight='bold')
axes[0].set_title('A. Pathway Reprogramming\nin High-Vulnerability Regions', 
                  fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3, axis='x')
axes[0].invert_yaxis()

# Add interpretation labels
axes[0].text(0.98, 0.98, '↑ Enriched in\nHigh-Vulnerability', 
            transform=axes[0].transAxes, 
            fontsize=10, fontweight='bold', color=COLOR_HIGH, 
            ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[0].text(0.98, 0.02, '↓ Depleted in\nHigh-Vulnerability', 
            transform=axes[0].transAxes,
            fontsize=10, fontweight='bold', color=COLOR_LOW, 
            ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Panel B: Absolute Differences
axes[1].barh(y_pos, abs_diffs, color=colors_plot, edgecolor='black', linewidth=0.5)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(pathways_plot, fontsize=9)
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=1.5)
axes[1].set_xlabel('Absolute Difference\n(High-Vuln - Low-Vuln)', 
                  fontsize=12, fontweight='bold')
axes[1].set_title('B. Magnitude of Pathway Changes', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3, axis='x')
axes[1].invert_yaxis()

# Panel C: Statistical Significance
axes[2].barh(y_pos, log_p_values, color=colors_plot, edgecolor='black', linewidth=0.5)
axes[2].set_yticks(y_pos)
axes[2].set_yticklabels(pathways_plot, fontsize=9)
axes[2].set_xlabel('-log₁₀(P-value)', fontsize=12, fontweight='bold')
axes[2].set_title('C. Statistical Significance', fontsize=12, fontweight='bold')
axes[2].axvline(x=-np.log10(0.05), color=COLOR_REFERENCE, linestyle='--', 
               linewidth=2, alpha=0.7, label='p=0.05')
axes[2].axvline(x=-np.log10(0.05/len(PRIMARY_PATHWAY_COLS)), color=COLOR_SECONDARY, 
               linestyle=':', linewidth=2, alpha=0.7, label='Bonferroni')
axes[2].legend(fontsize=9, loc='lower right')
axes[2].grid(alpha=0.3, axis='x')
axes[2].invert_yaxis()

plt.tight_layout()

os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/pathway_enrichment_vulnerability.png', dpi=300, bbox_inches='tight')
plt.show()

print("  Saved: results/figures/pathway_enrichment_vulnerability.png")

# SUMMARY STATISTICS

print("ENRICHMENT SUMMARY")

print(f"\n[Overall Statistics]")
print(f"  Total pathways analyzed: {len(enrichment_df)}")
print(f"  Enriched in high-vulnerability (d > 0): {(enrichment_df['Cohens_d'] > 0).sum()}")
print(f"  Depleted in high-vulnerability (d < 0): {(enrichment_df['Cohens_d'] < 0).sum()}")
print(f"  Statistically significant (p < 0.05): {(enrichment_df['P_Value'] < 0.05).sum()}")

print(f"\n[Effect Size Categories]")
small_effect = (enrichment_df['Abs_Cohens_d'] > 0.2).sum()
medium_effect = (enrichment_df['Abs_Cohens_d'] > 0.5).sum()
large_effect = (enrichment_df['Abs_Cohens_d'] > 0.8).sum()

print(f"  Small effect (|d| > 0.2): {small_effect}/{len(enrichment_df)}")
print(f"  Medium effect (|d| > 0.5): {medium_effect}/{len(enrichment_df)}")
print(f"  Large effect (|d| > 0.8): {large_effect}/{len(enrichment_df)}")

print(f"\n[Top Enriched Pathway]")
top_enriched = enrichment_df.iloc[0]
print(f"  Pathway: {top_enriched['Pathway']}")
print(f"  Cohen's d: {top_enriched['Cohens_d']:+.3f}")
print(f"  Mean (High-Vuln): {top_enriched['Mean_High']:.3f}")
print(f"  Mean (Low-Vuln): {top_enriched['Mean_Low']:.3f}")
print(f"  P-value: {top_enriched['P_Value']:.2e}")

In [ ]:
# AUTOENCODER-BASED METABOLIC VULNERABILITY ANALYSIS
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

print("AUTOENCODER-BASED VULNERABILITY ANALYSIS")

# Get pathway scores (excluding technical covariates)
pathway_cols = [c for c in PRIMARY_FEATURES if not c.startswith(('n_counts', 'pct_counts'))]
S_path = adata.obs[pathway_cols].values
P = len(pathway_cols)  

print(f"Input: {S_path.shape[0]} spots × {P} pathways")

# Standardize
scaler = StandardScaler()
S_scaled = scaler.fit_transform(S_path)

# Train/val split
np.random.seed(42)
n_samples = len(S_scaled)
indices = np.random.permutation(n_samples)
split = int(0.8 * n_samples)
train_idx, val_idx = indices[:split], indices[split:]

X_train = torch.FloatTensor(S_scaled[train_idx])
X_val = torch.FloatTensor(S_scaled[val_idx])

# Define Autoencoder (20 → 32 → 8 → 32 → 20)
class MetabolicAutoencoder(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=32, latent_dim=8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)
    
    def encode(self, x):
        return self.encoder(x)

# Training
autoencoder = MetabolicAutoencoder(input_dim=P, hidden_dim=32, latent_dim=8)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()

train_loader = DataLoader(TensorDataset(X_train, X_train), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, X_val), batch_size=128)

# Training loop with early stopping
best_val_loss = float('inf')
patience_counter = 0
train_losses, val_losses = [], []

print("\nTraining autoencoder")
for epoch in range(100):
    # Train
    autoencoder.train()
    train_loss = 0
    for batch_x, _ in train_loader:
        optimizer.zero_grad()
        recon = autoencoder(batch_x)
        loss = criterion(recon, batch_x)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validate
    autoencoder.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, _ in val_loader:
            recon = autoencoder(batch_x)
            val_loss += criterion(recon, batch_x).item()
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = autoencoder.state_dict().copy()
    else:
        patience_counter += 1
    
    if patience_counter >= 15:
        print(f"  Early stopping at epoch {epoch+1}")
        break
    
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

autoencoder.load_state_dict(best_state)
print(f"\nFinal validation MSE: {best_val_loss:.4f}")

In [ ]:
# IN-SILICO PATHWAY KNOCKDOWN

print("Performing in-silico pathway knockdowns...")

autoencoder.eval()
X_all = torch.FloatTensor(S_scaled)

with torch.no_grad():
    # Baseline reconstruction error
    recon_baseline = autoencoder(X_all)
    RE_baseline = ((X_all - recon_baseline) ** 2).sum(dim=1).numpy()
    
    # Knockdown each pathway
    delta_RE = np.zeros((len(S_scaled), P))
    
    for p in range(P):
        X_knockdown = X_all.clone()
        X_knockdown[:, p] = 0  # Set pathway p to zero
        
        recon_knockdown = autoencoder(X_knockdown)
        RE_knockdown = ((X_all - recon_knockdown) ** 2).sum(dim=1).numpy()
        
        delta_RE[:, p] = RE_knockdown - RE_baseline

# Global pathway vulnerability ranking
V_AE = delta_RE.mean(axis=0)
pathway_vulnerability = pd.DataFrame({
    'pathway': pathway_cols,
    'V_AE': V_AE
}).sort_values('V_AE', ascending=False)

print("\nTop 10 most essential pathways (V^AE):")
print(pathway_vulnerability.head(10).to_string(index=False))

# Deep-learning vulnerability score per spot
V_DL = delta_RE.sum(axis=1)
adata.obs['vulnerability_score_DL'] = V_DL

# THE KEY RESULT - correlation with proliferation
from scipy.stats import pearsonr, spearmanr

r_pearson, p_pearson = pearsonr(V_DL, adata.obs['proliferation_score'].values)
r_spearman, p_spearman = spearmanr(V_DL, adata.obs['proliferation_score'].values)

print(f"\n{'='*50}")
print("VULNERABILITY-PROLIFERATION CORRELATION")
print(f"{'='*50}")
print(f"Pearson r  = {r_pearson:.3f} (p = {p_pearson:.2e})")
print(f"Spearman ρ = {r_spearman:.3f} (p = {p_spearman:.2e})")

In [ ]:
# Extracting gene lists from Patient 1 and using them for all patients
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support
from torch.utils.data import TensorDataset, DataLoader


print("LEAVE-ONE-PATIENT-OUT VALIDATION ")

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOPO_EPOCHS = 30
LOPO_BATCH_SIZE = 64
LOPO_LEARNING_RATE = 0.001

print(f"\nConfiguration:")
print(f"  Device: {DEVICE}")
print(f"  Epochs: {LOPO_EPOCHS}")
print(f"  Batch size: {LOPO_BATCH_SIZE}")
print(f"  Learning rate: {LOPO_LEARNING_RATE}")

# Add Patient 7 to config
if 7 not in CFG.PATIENT_CONFIG:
    CFG.PATIENT_CONFIG[7] = {
        'data_dir': 'Patient_7/',
        'h5_file': 'Visium_HD_16um_filtered_feature_bc_matrix.h5'
    }

# STEP 1: Extract pathway gene lists from Patient 1
print("\nExtracting pathway gene lists from Patient 1")

# Load pathway databases once
import gseapy as gp

all_pathway_genes = {}

databases_to_load = [
    ('MSigDB', 'MSigDB_Hallmark_2020'),
    ('KEGG', 'KEGG_2021_Human'),
    ('Reactome', 'Reactome_2022'),
    ('GO', 'GO_Biological_Process_2023'),
    ('BioCarta', 'BioCarta_2016')
]

pathway_dbs = {}
for prefix, db_name in databases_to_load:
    try:
        pathway_dbs[prefix] = gp.get_library(name=db_name, organism='Human')
        print(f"  Loaded {db_name}: {len(pathway_dbs[prefix])} pathways")
    except Exception as e:
        print(f"  Failed to load {db_name}: {e}")

# For each pathway in PRIMARY_FEATURES, find the gene list
# We need to search more flexibly
PATHWAY_COLS = [f for f in PRIMARY_FEATURES if not f.startswith(('n_counts', 'pct_counts'))]
TECHNICAL_COLS = [f for f in PRIMARY_FEATURES if f.startswith(('n_counts', 'pct_counts'))]

pathway_gene_lists = {}

for full_name in PATHWAY_COLS:
    parts = full_name.split('_')
    collection = parts[0]
    
    # Get the pathway name without collection prefix
    if collection == 'BioCarta':
        # Format: BioCarta_BIOCARTA_DNAFRAGMENT_PATHWAY
        # Search term: DNAFRAGMENT or similar
        search_terms = ['_'.join(parts[2:]), '_'.join(parts[1:])]
    elif collection == 'Reactome':
        # Format: Reactome_REACTOME_G2_M_DNA_REPLICATION_CHECKPOINT
        search_terms = ['_'.join(parts[2:]), '_'.join(parts[1:]), ' '.join(parts[2:])]
    elif collection == 'KEGG':
        # Format: KEGG_KEGG_MEDICUS_...
        search_terms = ['_'.join(parts[2:]), '_'.join(parts[1:])]
    else:
        search_terms = ['_'.join(parts[1:])]
    
    # Search in the appropriate database
    found = False
    if collection in pathway_dbs:
        db = pathway_dbs[collection]
        
        # Try exact matches first
        for term in search_terms:
            if term in db:
                pathway_gene_lists[full_name] = db[term]
                found = True
                break
        
        # If not found, try case-insensitive partial matching
        if not found:
            for db_pathway_name, genes in db.items():
                # Check if key parts match
                db_name_upper = db_pathway_name.upper().replace(' ', '_')
                for term in search_terms:
                    term_upper = term.upper()
                    if term_upper in db_name_upper or db_name_upper in term_upper:
                        pathway_gene_lists[full_name] = genes
                        found = True
                        break
                    # Also check last few words
                    term_words = term_upper.split('_')[-3:]
                    if all(w in db_name_upper for w in term_words if len(w) > 3):
                        pathway_gene_lists[full_name] = genes
                        found = True
                        break
                if found:
                    break
    
    if found:
        print(f"  Found: {full_name[:50]}... ({len(pathway_gene_lists[full_name])} genes)")
    else:
        print(f"  NOT FOUND: {full_name[:50]}...")
        pathway_gene_lists[full_name] = []

print(f"\nFound gene lists for {sum(1 for v in pathway_gene_lists.values() if len(v) > 0)}/{len(PATHWAY_COLS)} pathways")


def score_pathways_with_gene_lists(adata_patient, pathway_gene_lists, pathway_names):
    """Score pathways using pre-defined gene lists."""
    
    pathway_scores = pd.DataFrame(
        0.0,
        index=adata_patient.obs_names,
        columns=pathway_names
    )
    
    scored = 0
    for pathway_name in pathway_names:
        if pathway_name in pathway_gene_lists and len(pathway_gene_lists[pathway_name]) > 0:
            gene_list = pathway_gene_lists[pathway_name]
            genes_present = [g for g in gene_list if g in adata_patient.var_names]
            
            if len(genes_present) >= 3:
                pathway_expr = adata_patient[:, genes_present].X
                if hasattr(pathway_expr, 'toarray'):
                    pathway_expr = pathway_expr.toarray()
                pathway_scores[pathway_name] = np.mean(pathway_expr, axis=1)
                scored += 1
    
    print(f"    Scored {scored}/{len(pathway_names)} pathways")
    return pathway_scores


# PREPARE PATIENT 1 DATA
print("\nPreparing Patient 1 data")

patient_data = {}
patient_data['Patient_1'] = {
    'X': adata.obs[PRIMARY_FEATURES].values.copy(),
    'y': adata.obs['target_proliferation_binary'].values.copy(),
    'n_spots': len(adata)
}
print(f"Patient 1: {patient_data['Patient_1']['n_spots']} spots")
print(f"  Features: {len(PRIMARY_FEATURES)}")
print(f"  Class balance: {patient_data['Patient_1']['y'].mean():.1%} high proliferation")


# LOAD ADDITIONAL PATIENTS
print("\nLoading additional patients")

for patient_num in [2, 3, 4, 5, 6, 7]:
    patient_name = f'Patient_{patient_num}'
    
    print(f"\nLoading {patient_name}")
    
    try:
        if patient_num not in CFG.PATIENT_CONFIG:
            print(f"  No configuration found for {patient_name}")
            continue
        
        config = CFG.PATIENT_CONFIG[patient_num]
        data_dir = config['data_dir']
        h5_file = config['h5_file']
        
        print(f"  Path: {data_dir + h5_file}")
        
        is_visium_hd = (patient_num == 7) or ('HD' in h5_file) or ('16um' in h5_file)
        
        adata_temp = sc.read_10x_h5(data_dir + h5_file)
        adata_temp.var_names_make_unique()
        
        print(f"  Raw: {adata_temp.n_obs} spots x {adata_temp.n_vars} genes")
        
        if is_visium_hd and adata_temp.n_obs > 15000:
            print(f"  Downsampling Visium HD from {adata_temp.n_obs} to 15000 spots...")
            np.random.seed(42)
            idx = np.random.choice(adata_temp.n_obs, 15000, replace=False)
            adata_temp = adata_temp[idx, :].copy()
        
        sc.pp.filter_cells(adata_temp, min_counts=CFG.MIN_COUNTS)
        sc.pp.filter_genes(adata_temp, min_cells=CFG.MIN_CELLS)
        
        print(f"  After QC: {adata_temp.n_obs} spots x {adata_temp.n_vars} genes")
        
        sc.pp.normalize_total(adata_temp, target_sum=1e4)
        sc.pp.log1p(adata_temp)
        
        # Score pathways using gene lists
        print(f"  Scoring pathways...")
        pathway_scores = score_pathways_with_gene_lists(
            adata_temp, 
            pathway_gene_lists, 
            PATHWAY_COLS
        )
        
        # Add technical covariates
        if 'n_counts' not in adata_temp.obs.columns:
            if hasattr(adata_temp.X, 'A1'):
                adata_temp.obs['n_counts'] = adata_temp.X.sum(axis=1).A1
            else:
                adata_temp.obs['n_counts'] = np.array(adata_temp.X.sum(axis=1)).flatten()
        
        if 'n_counts' in TECHNICAL_COLS:
            pathway_scores['n_counts'] = adata_temp.obs['n_counts'].values
        
        adata_temp.var['mt'] = adata_temp.var_names.str.startswith('MT-')
        mt_counts = adata_temp[:, adata_temp.var['mt']].X.sum(axis=1)
        if hasattr(mt_counts, 'A1'):
            mt_counts = mt_counts.A1
        else:
            mt_counts = np.array(mt_counts).flatten()
        adata_temp.obs['pct_counts_mt'] = (mt_counts / adata_temp.obs['n_counts']) * 100
        
        if 'pct_counts_mt' in TECHNICAL_COLS:
            pathway_scores['pct_counts_mt'] = adata_temp.obs['pct_counts_mt'].values
        
        ribo_genes = [g for g in adata_temp.var_names if g.startswith(('RPS', 'RPL'))]
        if len(ribo_genes) > 0:
            ribo_counts = adata_temp[:, ribo_genes].X.sum(axis=1)
            if hasattr(ribo_counts, 'A1'):
                ribo_counts = ribo_counts.A1
            else:
                ribo_counts = np.array(ribo_counts).flatten()
            ribo_pct = (ribo_counts / adata_temp.obs['n_counts'].values) * 100
        else:
            ribo_pct = np.zeros(len(adata_temp))
        
        if 'pct_counts_ribo' in TECHNICAL_COLS:
            pathway_scores['pct_counts_ribo'] = ribo_pct
        
        # Create target
        prolif_genes_present = [g for g in CFG.PROLIFERATION_GENES if g in adata_temp.var_names]
        
        if len(prolif_genes_present) > 0:
            prolif_expr = adata_temp[:, prolif_genes_present].X
            if hasattr(prolif_expr, 'toarray'):
                prolif_expr = prolif_expr.toarray()
            prolif_score = np.mean(prolif_expr, axis=1)
            threshold = np.percentile(prolif_score, 75)
            y_temp = (prolif_score > threshold).astype(int)
            print(f"    Target: {len(prolif_genes_present)} proliferation genes (top 25%)")
        else:
            sc.pp.pca(adata_temp, n_comps=20)
            sc.pp.neighbors(adata_temp, n_neighbors=15)
            sc.tl.leiden(adata_temp, resolution=0.5)
            cluster_sizes = adata_temp.obs['leiden'].value_counts()
            tumor_cluster = cluster_sizes.index[0]
            y_temp = (adata_temp.obs['leiden'] == tumor_cluster).astype(int)
        
        # Ensure all columns exist
        for col in PRIMARY_FEATURES:
            if col not in pathway_scores.columns:
                pathway_scores[col] = 0.0
        
        X_temp = pathway_scores[PRIMARY_FEATURES].values
        X_temp = np.nan_to_num(X_temp, nan=0.0)
        
        patient_data[patient_name] = {
            'X': X_temp,
            'y': y_temp,
            'n_spots': len(X_temp)
        }
        
        print(f"  {patient_name}: {patient_data[patient_name]['n_spots']} spots, {y_temp.mean():.1%} high prolif")
        
        del adata_temp, pathway_scores, X_temp, y_temp
        gc.collect()
        
    except Exception as e:
        import traceback
        print(f"  Error: {str(e)}")
        traceback.print_exc()
        continue

print(f"\nTotal patients loaded: {len(patient_data)}")

# Check feature variance across patients
print("\nFeature variance check:")
for feat_idx, feat_name in enumerate(PRIMARY_FEATURES[:5]):
    variances = []
    for p_name, p_data in patient_data.items():
        var = np.var(p_data['X'][:, feat_idx])
        variances.append((p_name, var))
    print(f"  {feat_name[:40]}:")
    for p_name, var in variances:
        print(f"    {p_name}: var={var:.4f}")


# RUN LOPO
print("\nRunning LOPO Cross-Validation")

if len(patient_data) < 2:
    print(f"WARNING: Need at least 2 patients for LOPO")
    lopo_df = pd.DataFrame()
else:
    print(f"Running LOPO with {len(patient_data)} patients\n")
    
    lopo_results = []
    available_patients = list(patient_data.keys())
    
    for fold_idx, held_out_patient in enumerate(available_patients):
        print(f"\nFOLD {fold_idx + 1}/{len(available_patients)}: Holding out {held_out_patient}")
        
        train_patients = [p for p in available_patients if p != held_out_patient]
        
        X_train = np.vstack([patient_data[p]['X'] for p in train_patients])
        y_train = np.concatenate([patient_data[p]['y'] for p in train_patients])
        
        X_test = patient_data[held_out_patient]['X']
        y_test = patient_data[held_out_patient]['y']
        
        print(f"  Train: {len(X_train)} spots ({y_train.mean():.1%} pos)")
        print(f"  Test: {len(X_test)} spots ({y_test.mean():.1%} pos)")
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        X_train_tensor = torch.FloatTensor(X_train_scaled).to(DEVICE)
        y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1).to(DEVICE)
        X_test_tensor = torch.FloatTensor(X_test_scaled).to(DEVICE)
        
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_dataset, batch_size=LOPO_BATCH_SIZE, 
                                  shuffle=True, drop_last=True)
        
        model_lopo = MetabolicVulnerabilityNet(
            input_dim=X_train.shape[1],
            hidden_dim1=CFG.HIDDEN_DIM1,
            hidden_dim2=CFG.HIDDEN_DIM2,
            hidden_dim3=16,
            dropout=CFG.DROPOUT
        ).to(DEVICE)
        
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model_lopo.parameters(), lr=LOPO_LEARNING_RATE, 
                                     weight_decay=CFG.WEIGHT_DECAY)
        
        model_lopo.train()
        for epoch in range(LOPO_EPOCHS):
            epoch_loss = 0
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = model_lopo(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            if (epoch + 1) % 10 == 0:
                print(f"    Epoch {epoch+1}/{LOPO_EPOCHS}, Loss: {epoch_loss/len(train_loader):.4f}")
        
        model_lopo.eval()
        with torch.no_grad():
            y_pred_proba = model_lopo(X_test_tensor).cpu().numpy().flatten()
            y_pred = (y_pred_proba > 0.5).astype(int)
        
        try:
            auc = roc_auc_score(y_test, y_pred_proba)
        except:
            auc = np.nan
        
        accuracy = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average='binary', zero_division=0
        )
        
        print(f"  Results: AUC={auc:.4f}, Acc={accuracy:.4f}, F1={f1:.4f}")
        
        lopo_results.append({
            'held_out_patient': held_out_patient,
            'n_train_patients': len(train_patients),
            'n_train_spots': len(X_train),
            'n_test_spots': len(X_test),
            'train_balance': y_train.mean(),
            'test_balance': y_test.mean(),
            'auc': auc,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        })
        
        del model_lopo, X_train_tensor, y_train_tensor, X_test_tensor
        gc.collect()
    
    # Results
    lopo_df = pd.DataFrame(lopo_results)
    
    print("\n\nLOPO RESULTS SUMMARY")
    print("\nIndividual Patient Performance:")
    print(f"{'Patient':<15} | {'AUC':>6} | {'Accuracy':>8} | {'F1':>6} | {'Test Spots':>10}")
    print("-" * 60)
    for _, row in lopo_df.iterrows():
        print(f"{row['held_out_patient']:<15} | {row['auc']:>6.4f} | {row['accuracy']:>8.4f} | "
              f"{row['f1']:>6.4f} | {row['n_test_spots']:>10d}")
    
    print("\nAggregate Performance:")
    print(f"  Mean AUC:       {lopo_df['auc'].mean():.4f} +/- {lopo_df['auc'].std():.4f}")
    print(f"  Mean Accuracy:  {lopo_df['accuracy'].mean():.4f} +/- {lopo_df['accuracy'].std():.4f}")
    print(f"  Mean F1:        {lopo_df['f1'].mean():.4f} +/- {lopo_df['f1'].std():.4f}")
    
    lopo_df.to_csv('results/tables/lopo_validation_results.csv', index=False)
    print("\nSaved: results/tables/lopo_validation_results.csv")

In [ ]:
# LOPO CONFUSION MATRICES 
print("LOPO CONFUSION MATRIX ANALYSIS")


from sklearn.metrics import confusion_matrix, classification_report

# Check if LOPO was run
if 'lopo_df' not in dir() or len(lopo_df) == 0:
    print(" LOPO validation not run yet. Run LOPO cell first.\n")
else:
    print(f"Analyzing predictions for {len(patient_data)} patients\n")
    
    print("[Re-running LOPO predictions to generate confusion matrices]\n")
    
    confusion_results = []
    available_patients = list(patient_data.keys())
    
    # Create figure for confusion matrices
    n_patients = len(available_patients)
    
    # Determine grid size
    if n_patients <= 4:
        n_rows, n_cols = 2, 2
    elif n_patients <= 6:
        n_rows, n_cols = 2, 3
    elif n_patients <= 9:
        n_rows, n_cols = 3, 3
    else:
        n_rows, n_cols = 4, 3
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
    
    # Always flatten to 1D array for consistent indexing
    if n_patients == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
    
    for idx, held_out_patient in enumerate(available_patients):
        print(f"Processing {held_out_patient}...")
        
        # Get training patients
        train_patients = [p for p in available_patients if p != held_out_patient]
        
        # Combine training data
        X_train = np.vstack([patient_data[p]['X'] for p in train_patients])
        y_train = np.concatenate([patient_data[p]['y'] for p in train_patients])
        
        # Test data
        X_test = patient_data[held_out_patient]['X']
        y_test = patient_data[held_out_patient]['y']
        
        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Convert to PyTorch
        X_train_tensor = torch.FloatTensor(X_train_scaled).to(DEVICE)
        y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1).to(DEVICE)
        X_test_tensor = torch.FloatTensor(X_test_scaled).to(DEVICE)
        
        # Create data loader - with drop_last=True
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_dataset, batch_size=LOPO_BATCH_SIZE, 
                                  shuffle=True, drop_last=True)
        
        # Initialise model
        model_lopo = MetabolicVulnerabilityNet(
            input_dim=X_train.shape[1],
            hidden_dim1=CFG.HIDDEN_DIM1,
            hidden_dim2=CFG.HIDDEN_DIM2,
            hidden_dim3=16,
            dropout=CFG.DROPOUT
        ).to(DEVICE)
        
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model_lopo.parameters(), lr=LOPO_LEARNING_RATE,
                                     weight_decay=CFG.WEIGHT_DECAY)
        
        # Training (silent)
        model_lopo.train()
        for epoch in range(LOPO_EPOCHS):
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = model_lopo(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
        
        # Evaluate
        model_lopo.eval()
        with torch.no_grad():
            y_pred_proba = model_lopo(X_test_tensor).cpu().numpy().flatten()
            y_pred = (y_pred_proba > 0.5).astype(int)
        
        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        
        # Store results
        tn, fp, fn, tp = cm.ravel()
        
        confusion_results.append({
            'Patient': held_out_patient,
            'TN': tn,
            'FP': fp,
            'FN': fn,
            'TP': tp,
            'Total': len(y_test),
            'True_Pos_Rate': tp / (tp + fn) if (tp + fn) > 0 else 0,
            'True_Neg_Rate': tn / (tn + fp) if (tn + fp) > 0 else 0,
            'Positive_Pred_Value': tp / (tp + fp) if (tp + fp) > 0 else 0,
            'Negative_Pred_Value': tn / (tn + fn) if (tn + fn) > 0 else 0
        })
        
        # Plot confusion matrix
        ax = axes[idx]
        
        # Normalise by row (true labels) for better interpretation
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        # Plot
        sns.heatmap(cm, annot=True, fmt='d', cmap=CMAP_SEQUENTIAL, ax=ax,
                   cbar=False, square=True, linewidths=2, linecolor='black',
                   annot_kws={'size': 14, 'weight': 'bold'})
        
        # Add percentages
        for i in range(2):
            for j in range(2):
                percentage = cm_normalized[i, j] * 100
                ax.text(j + 0.5, i + 0.7, f'({percentage:.1f}%)',
                       ha='center', va='center', fontsize=10, color='gray')
        
        ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
        ax.set_title(f'{held_out_patient}\n(n={len(y_test)}, AUC={roc_auc_score(y_test, y_pred_proba):.3f})',
                    fontsize=12, fontweight='bold')
        ax.set_xticklabels(['Low Prolif (0)', 'High Prolif (1)'], fontsize=10)
        ax.set_yticklabels(['Low Prolif (0)', 'High Prolif (1)'], fontsize=10, rotation=0)
        
        # Add accuracy
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        ax.text(0.02, 0.98, f'Accuracy: {accuracy:.3f}',
               transform=ax.transAxes, fontsize=10, fontweight='bold',
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
        
        print(f"   {held_out_patient}: TN={tn}, FP={fp}, FN={fn}, TP={tp}\n")
        
        # Clean up
        del model_lopo, X_train_tensor, y_train_tensor, X_test_tensor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    
    # Hide unused subplots
    for idx in range(n_patients, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    
    os.makedirs('results/figures', exist_ok=True)
    plt.savefig('results/figures/lopo_confusion_matrices.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(" Saved: results/figures/lopo_confusion_matrices.png\n")
    
    # Create summary table
    confusion_df = pd.DataFrame(confusion_results)
    
    print("CONFUSION MATRIX SUMMARY")

    
    print("Raw Counts:")
    print(f"{'Patient':<15} | {'TN':>6} | {'FP':>6} | {'FN':>6} | {'TP':>6} | {'Total':>6}")
    for _, row in confusion_df.iterrows():
        print(f"{row['Patient']:<15} | {int(row['TN']):>6d} | {int(row['FP']):>6d} | "
              f"{int(row['FN']):>6d} | {int(row['TP']):>6d} | {int(row['Total']):>6d}")
    
    print("Performance Metrics:")
    
    print(f"{'Patient':<15} | {'Sensitivity':>11} | {'Specificity':>11} | {'PPV':>11} | {'NPV':>11}")
    for _, row in confusion_df.iterrows():
        print(f"{row['Patient']:<15} | {row['True_Pos_Rate']:>11.3f} | {row['True_Neg_Rate']:>11.3f} | "
              f"{row['Positive_Pred_Value']:>11.3f} | {row['Negative_Pred_Value']:>11.3f}")
    
    print("Interpretation:")
    
    # Aggregate statistics
    print(f"\nAggregate Performance:")
    print(f"  Mean Sensitivity (TPR): {confusion_df['True_Pos_Rate'].mean():.3f} ± {confusion_df['True_Pos_Rate'].std():.3f}")
    print(f"  Mean Specificity (TNR): {confusion_df['True_Neg_Rate'].mean():.3f} ± {confusion_df['True_Neg_Rate'].std():.3f}")
    print(f"  Mean PPV (Precision):   {confusion_df['Positive_Pred_Value'].mean():.3f} ± {confusion_df['Positive_Pred_Value'].std():.3f}")
    print(f"  Mean NPV:               {confusion_df['Negative_Pred_Value'].mean():.3f} ± {confusion_df['Negative_Pred_Value'].std():.3f}")
    
    # Check for bias
    print(f"\n[Bias Analysis]")
    
    # Total predictions
    total_tn = confusion_df['TN'].sum()
    total_fp = confusion_df['FP'].sum()
    total_fn = confusion_df['FN'].sum()
    total_tp = confusion_df['TP'].sum()
    
    print(f"  Across all patients:")
    print(f"    True Negatives:  {int(total_tn):>6d}")
    print(f"    False Positives: {int(total_fp):>6d}")
    print(f"    False Negatives: {int(total_fn):>6d}")
    print(f"    True Positives:  {int(total_tp):>6d}")
    
    # Bias towards positive or negative
    if total_fp > total_fn:
        bias_ratio = total_fp / total_fn if total_fn > 0 else float('inf')
        print(f"\n   Model tends to OVER-PREDICT high proliferation")
        print(f"    FP/FN ratio: {bias_ratio:.2f}")
        print(f"    (More false positives than false negatives)")
    elif total_fn > total_fp:
        bias_ratio = total_fn / total_fp if total_fp > 0 else float('inf')
        print(f"\n   Model tends to UNDER-PREDICT high proliferation")
        print(f"    FN/FP ratio: {bias_ratio:.2f}")
        print(f"    (More false negatives than false positives)")
    else:
        print(f"\n   Model is balanced (FP ≈ FN)")
    
    # Save results
    os.makedirs('results/tables', exist_ok=True)
    confusion_df.to_csv('results/tables/lopo_confusion_matrices.csv', index=False)
    print(f"\n Saved: results/tables/lopo_confusion_matrices.csv")

In [ ]:
# LOPO Result 

print("EXPORTING LOPO RESULTS FOR FLUX INTERPRETATION")

# Save LOPO performance metrics
lopo_df.to_csv('lopo_results.csv', index=False)
print(" Saved: lopo_results.csv")

# Save discovered pathways from each fold
if 'fold_pathways' in dir():  
    for fold_id, pathways_df in fold_pathways.items():
        pathways_df.to_csv(f'pathways_fold_{fold_id}.csv', index=False)
    print(f" Saved pathway discoveries for {len(fold_pathways)} folds")

#  Save all predictions
if 'all_lopo_predictions' in dir():
    all_lopo_predictions.to_csv('all_lopo_predictions.csv', index=False)
    print(" Saved: all_lopo_predictions.csv")

# Export pathway scores for Patient 1 (for validation)
pathway_scores = adata.obs[PRIMARY_PATHWAY_COLS].copy()
pathway_scores.to_csv('Patient_1_pathway_scores.csv')
print(" Saved: Patient_1_pathway_scores.csv")

# Export target variable
target_df = pd.DataFrame({
    'target': adata.obs['target_proliferation_binary']
}, index=adata.obs_names)
target_df.to_csv('Patient_1_target.csv')
print(" Saved: Patient_1_target.csv")

In [ ]:
# SCFEA 

import shutil

scfea_file = './scFEA/src/scFEA.py'

if not Path(scfea_file).exists():
    print(f"\n ERROR: {scfea_file} not found!")
else:
    print(f"\n Found: {scfea_file}")
    
    # Backup original
    backup_file = './scFEA/src/scFEA.py.backup'
    if not Path(backup_file).exists():
        shutil.copy(scfea_file, backup_file)
        print(f" Backup created: {backup_file}")
    else:
        print(f" Backup exists: {backup_file}")
    
    # Read file
    with open(scfea_file, 'r') as f:
        content = f.read()
    
    # Check if already patched
    if 'pd.concat([geneExprDf' in content:
        print("\n Already patched! No changes needed.")
    else:
        print("\nPatching file")
        
        original_line = "geneExprDf = geneExprDf.append(temp, ignore_index = True, sort=False)"
        fixed_line = "geneExprDf = pd.concat([geneExprDf, temp], ignore_index=True, sort=False)"
        
        if original_line in content:
            content = content.replace(original_line, fixed_line)
            print("   Fixed line 172: DataFrame.append() → pd.concat()")
        
        # Also fix any other .append() variations
        content = content.replace(
            "geneExprDf.append(temp, ignore_index=True, sort=False)",
            "pd.concat([geneExprDf, temp], ignore_index=True, sort=False)"
        )
        
        # Write back
        with open(scfea_file, 'w') as f:
            f.write(content)
    
        print(f"  - Original backed up to: {backup_file}")

In [ ]:
print("RUNNING scFEA")

# Verify input
input_file = './scFEA/input/Patient_1_expression.csv'
expr_check = pd.read_csv(input_file, index_col=0)
print(f"\n Input: {expr_check.shape[0]:,} genes x {expr_check.shape[1]:,} spots")

# Create output directory
Path('./scFEA/results/Patient_1').mkdir(parents=True, exist_ok=True)

# Run scFEA

cmd = """
python ./scFEA/src/scFEA.py \
    --data_dir ./scFEA/data \
    --input_dir ./scFEA/input \
    --res_dir ./scFEA/results/Patient_1 \
    --test_file Patient_1_expression.csv \
    --moduleGene_file module_gene_m168.csv \
    --stoichiometry_matrix cmMat_c70_m168.csv \
    --output_flux_file Patient_1_flux.csv \
    --output_balance_file Patient_1_balance.csv \
    --sc_imputation True
"""

exit_code = os.system(cmd)

if exit_code == 0:
    print(" scFEA COMPLETED SUCCESSFULLY!")
    
    # Verify output
    flux_file = './scFEA/results/Patient_1/Patient_1_flux.csv'
    if Path(flux_file).exists():
        flux_df = pd.read_csv(flux_file, index_col=0)
        print(f"\n Flux output:")
        print(f"  Modules: {flux_df.shape[0]}")
        print(f"  Spots: {flux_df.shape[1]}")
        
        if flux_df.shape[0] > 100:
            print(f"\n SUCCESS! {flux_df.shape[0]} modules!")
        else:
            print(f"\n  Warning: Only {flux_df.shape[0]} modules")
    else:
        print(f"\n  Output not found at {flux_file}")
else:
    print(f"\n scFEA failed with exit code: {exit_code}")
   

In [ ]:
# Load the transposed file
flux_file = './scFEA/results/Patient_1/Patient_1_flux.csv'
flux_df = pd.read_csv(flux_file, index_col=0)

print(f"\nOriginal (transposed):")
print(f"  Shape: {flux_df.shape}")
print(f"  Rows: spots ({flux_df.shape[0]})")
print(f"  Cols: modules ({flux_df.shape[1]})")

# Transpose it!
flux_df_correct = flux_df.T

print(f"\nTransposed (correct):")
print(f"  Shape: {flux_df_correct.shape}")
print(f"  Rows: modules ({flux_df_correct.shape[0]})")
print(f"  Cols: spots ({flux_df_correct.shape[1]})")

# Verify
print(f"\nVerifying:")
print(f"  Index (first 5): {list(flux_df_correct.index[:5])}")
print(f"  Columns (first 5): {list(flux_df_correct.columns[:5])}")

if all(str(idx).startswith('M_') for idx in flux_df_correct.index[:5]):
    print("\n   Index is modules (M_1, M_2, etc.)")
if all(str(col).startswith('AAACAA') or '-' in str(col) for col in flux_df_correct.columns[:5]):
    print("   Columns are spot barcodes")

# Save corrected version
output_file = './scFEA/results/Patient_1/Patient_1_flux_corrected.csv'
flux_df_correct.to_csv(output_file)

print(f"\n Saved corrected flux to: {output_file}")

In [ ]:
# EXPORT RESULTS FOR FLUX INTERPRETATION

print("EXPORTING DATA FOR FLUX INTERPRETATION")

# Export pathway scores
pathway_cols = [col for col in adata.obs.columns 
                if any(col.startswith(p) for p in ['Reactome_', 'GO_', 'KEGG_', 'MSigDB_'])]

pathway_scores = adata.obs[pathway_cols].copy()
pathway_scores.to_csv('Patient_1_pathway_scores.csv')
print(f" Saved pathway scores: {pathway_scores.shape}")

#  Export target
target_df = pd.DataFrame({
    'target': adata.obs['target_proliferation_binary']
}, index=adata.obs_names)
target_df.to_csv('Patient_1_target.csv')
print(f" Saved target: {target_df.shape}")

# Export LOPO results
if 'lopo_df' in dir():
    lopo_df.to_csv('lopo_results.csv', index=False)
    print(f" Saved LOPO results: {lopo_df.shape}")

print("\n All data exported! Ready for flux interpretation.")

In [ ]:
print("FLUX PROCESSING - PATIENT 1")

# Load CORRECTED flux data
flux_file = './scFEA/results/Patient_1/Patient_1_flux_corrected.csv'
flux_df = pd.read_csv(flux_file, index_col=0)

print(f"\nLoaded flux data:")
print(f"  Modules: {flux_df.shape[0]}")
print(f"  Spots: {flux_df.shape[1]}")

# Transpose to spots x modules (for processing)
flux_spots = flux_df.T

print(f"\nTransposed for processing:")
print(f"  Spots: {flux_spots.shape[0]}")
print(f"  Modules: {flux_spots.shape[1]}")

# Check variance
print("\nChecking module variance:")
variances = flux_spots.var(axis=0)
n_with_variance = (variances > 1e-10).sum()

print(f"  Total modules: {len(variances)}")
print(f"  With variance: {n_with_variance}")
print(f"  Zero variance: {len(variances) - n_with_variance}")
print(f"  Mean variance: {variances.mean():.6e}")

# Show top modules by variance
top_variance = variances.sort_values(ascending=False).head(10)
print(f"\nTop 10 modules by variance:")
for i, (module, var) in enumerate(top_variance.items(), 1):
    print(f"  {i:2d}. {module}: {var:.6e}")

# Filter to modules with variance
modules_with_variance = variances[variances > 1e-10].index.tolist()
flux_filtered = flux_spots[modules_with_variance]

print(f"\n Filtered to {len(modules_with_variance)} modules with variance")

# Save as RAW modules (spots x modules)
output_file = 'Patient_1_flux_RAW_modules.csv'
flux_filtered.to_csv(output_file)

print(f"\n Saved: {output_file}")
print(f"  Shape: {flux_filtered.shape[0]} spots x {flux_filtered.shape[1]} modules")

# Summary
print("FLUX PROCESSING COMPLETE!")
print(f"\nResults:")
print(f"  Input: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")
print(f"  Filtered: {flux_filtered.shape[1]} modules with variance")
print(f"  Output: {output_file}")
print(f"\n Ready for flux interpretation!")

In [ ]:
# COMPREHENSIVE FLUX INTERPRETATION ANALYSIS

from scipy.stats import spearmanr, mannwhitneyu
from statsmodels.stats.multitest import multipletests

print("FLUX INTERPRETATION - METABOLIC VULNERABILITIES")


# CONFIGURATION

CONFIG = {
    # Patient to analyse
    'PATIENT_ID': 1,  
    
    # Input files (auto-detected)
    'lopo_results_file': 'lopo_results.csv',
    'pathway_scores_file': None,  # Will be auto-set
    'target_file': None,  # Will be auto-set
    'flux_file': None,  # Will be auto-detected
    
    # Output directory
    'output_dir': './flux_interpretation_results',
    
    # Analysis parameters
    'FDR_THRESHOLD': 0.05,
    'MIN_CORRELATION': 0.1,
    'TOP_N_PATHWAYS': 20,
    'TOP_N_FLUX': 20,  # Number of top flux modules to report
}

patient_id = CONFIG['PATIENT_ID']
CONFIG['pathway_scores_file'] = f'Patient_{patient_id}_pathway_scores.csv'
CONFIG['target_file'] = f'Patient_{patient_id}_target.csv'

Path(CONFIG['output_dir']).mkdir(exist_ok=True)

print(f"\nAnalyzing Patient {patient_id}")
print(f"Output directory: {CONFIG['output_dir']}")

# AUTO-DETECT FLUX DATA FORMAT


print(" AUTO-DETECTING FLUX DATA FORMAT")


# Check for raw modules
raw_modules_file = f'Patient_{patient_id}_flux_RAW_modules.csv'
aggregated_file = f'Patient_{patient_id}_flux_metabolic_matched.csv'

if Path(raw_modules_file).exists():
    CONFIG['flux_file'] = raw_modules_file
    flux_type = 'raw_modules'
    print(f"\nDetected: RAW MODULES")
    print(f"  File: {raw_modules_file}")
elif Path(aggregated_file).exists():
    CONFIG['flux_file'] = aggregated_file
    flux_type = 'aggregated'
    print(f"\nDetected: AGGREGATED PATHWAYS")
    print(f"  File: {aggregated_file}")
else:
    print("\nERROR: No flux data found!")
    print(f"  Looking for: {raw_modules_file}")
    print(f"           or: {aggregated_file}")
    raise FileNotFoundError("Flux data not found")


#  LOAD LOPO RESULTS

print(" LOADING LOPO RESULTS")


try:
    lopo_results = pd.read_csv(CONFIG['lopo_results_file'])
    print(f"Loaded LOPO results: {len(lopo_results)} rows")
    if 'auc' in lopo_results.columns:
        print(f"  Mean AUC: {lopo_results['auc'].mean():.4f}")
    elif 'AUC' in lopo_results.columns:
        print(f"  Mean AUC: {lopo_results['AUC'].mean():.4f}")
except FileNotFoundError:
    print("LOPO results not found (optional)")
    lopo_results = None

# LOAD PATHWAY SCORES


print(" LOADING EXPRESSION-BASED PATHWAY SCORES")


try:
    pathway_scores = pd.read_csv(CONFIG['pathway_scores_file'], index_col=0)
    print(f"Loaded: {pathway_scores.shape[0]} spots x {pathway_scores.shape[1]} pathways")
    print(f"\nExample pathways:")
    for i, col in enumerate(pathway_scores.columns[:5], 1):
        print(f"  {i}. {col}")
except FileNotFoundError:
    print(f"ERROR: Pathway scores not found: {CONFIG['pathway_scores_file']}")
    print("Please export from your main notebook first!")
    raise

# LOAD TARGET VARIABLE

print(" LOADING TARGET VARIABLE")

try:
    target_df = pd.read_csv(CONFIG['target_file'], index_col=0)
    print(f"Loaded: {len(target_df)} spots")
    
    n_high = target_df['target'].sum()
    n_low = len(target_df) - n_high
    print(f"  High proliferation: {n_high} ({100*n_high/len(target_df):.1f}%)")
    print(f"  Low proliferation:  {n_low} ({100*n_low/len(target_df):.1f}%)")
except FileNotFoundError:
    print(f"ERROR: Target not found: {CONFIG['target_file']}")
    print("Please export from your main notebook first!")
    raise

#  LOAD FLUX DATA

print(" LOADING FLUX DATA")


flux_df = pd.read_csv(CONFIG['flux_file'], index_col=0)
print(f"Loaded flux: {flux_df.shape[0]} spots x {flux_df.shape[1]} features")
print(f"Format: {flux_type}")

print(f"\nFlux features:")
n_show = min(10, len(flux_df.columns))
for i, col in enumerate(flux_df.columns[:n_show], 1):
    print(f"  {i}. {col}")
if len(flux_df.columns) > n_show:
    print(f"  ... and {len(flux_df.columns) - n_show} more")

# Check variance
variances = flux_df.var()
n_variance = (variances > 1e-10).sum()
print(f"\nVariance check:")
print(f"  Features with variance: {n_variance} / {len(variances)}")
print(f"  Mean variance: {variances.mean():.6e}")

if n_variance < len(variances) * 0.5:
    print("\n  WARNING: Many features have no variance!")

#  MATCH SPOTS ACROSS DATASETS


print(" MATCHING SPOTS ACROSS DATASETS")


common_spots = (
    pathway_scores.index
    .intersection(target_df.index)
    .intersection(flux_df.index)
)

print(f"Pathway scores: {len(pathway_scores)} spots")
print(f"Target:         {len(target_df)} spots")
print(f"Flux:           {len(flux_df)} spots")
print(f"Common:         {len(common_spots)} spots")

if len(common_spots) == 0:
    raise ValueError("No matching spots found!")

# Filter to common spots
pathway_scores_matched = pathway_scores.loc[common_spots]
target_matched = target_df.loc[common_spots]
flux_matched = flux_df.loc[common_spots]


# IDENTIFY METABOLIC PATHWAYS

print("IDENTIFYING METABOLIC PATHWAYS")


metabolic_keywords = [
    'glycolysis', 'gluconeogenesis',
    'tca', 'citrate', 'citric',
    'oxidative phosphorylation', 'electron transport', 'oxphos',
    'fatty acid', 'lipid', 'beta-oxidation',
    'pentose phosphate',
    'amino acid', 'glutamine', 'glutamate',
    'pyruvate', 'lactate',
    'metabolic', 'metabolism'
]

def is_metabolic(pathway_name):
    return any(kw in pathway_name.lower() for kw in metabolic_keywords)

metabolic_pathways = [col for col in pathway_scores.columns if is_metabolic(col)]

print(f"Total pathways:     {len(pathway_scores.columns)}")
print(f"Metabolic pathways: {len(metabolic_pathways)}")

print(f"\nTop metabolic pathways:")
for i, pathway in enumerate(metabolic_pathways[:10], 1):
    print(f"  {i:2d}. {pathway}")
if len(metabolic_pathways) > 10:
    print(f"  ... and {len(metabolic_pathways) - 10} more")

#  EXPRESSION-BASED PATHWAY ASSOCIATIONS

print(" EXPRESSION-BASED PATHWAY ASSOCIATIONS")


expr_associations = []

for pathway in metabolic_pathways:
    pathway_expr = pathway_scores_matched[pathway].values
    target_values = target_matched['target'].values
    
    if np.var(pathway_expr) < 1e-10:
        continue
    
    # Correlation
    corr, pval = spearmanr(pathway_expr, target_values)
    
    # Mann-Whitney U
    high_expr = pathway_expr[target_values == 1]
    low_expr = pathway_expr[target_values == 0]
    
    if len(high_expr) > 0 and len(low_expr) > 0:
        _, pval_mw = mannwhitneyu(high_expr, low_expr, alternative='two-sided')
        mean_high = np.mean(high_expr)
        mean_low = np.mean(low_expr)
        fold_change = mean_high / mean_low if mean_low > 0 else np.inf
    else:
        pval_mw = 1.0
        mean_high = 0
        mean_low = 0
        fold_change = 1.0
    
    expr_associations.append({
        'pathway': pathway,
        'correlation': corr,
        'pval_corr': pval,
        'pval_mw': pval_mw,
        'mean_high': mean_high,
        'mean_low': mean_low,
        'fold_change': fold_change
    })

expr_assoc_df = pd.DataFrame(expr_associations)

# FDR correction
_, qvals, _, _ = multipletests(expr_assoc_df['pval_corr'], method='fdr_bh')
expr_assoc_df['qval'] = qvals
expr_assoc_df['significant'] = qvals < CONFIG['FDR_THRESHOLD']
expr_assoc_df['abs_corr'] = np.abs(expr_assoc_df['correlation'])

expr_assoc_df = expr_assoc_df.sort_values('abs_corr', ascending=False)

print(f"Tested:      {len(expr_assoc_df)} metabolic pathways")
print(f"Significant: {expr_assoc_df['significant'].sum()} (FDR < {CONFIG['FDR_THRESHOLD']})")

print(f"\nTop {CONFIG['TOP_N_PATHWAYS']} expression-based associations:")
print(f"{'Rank':<6} {'Pathway':<50} {'Corr':<8} {'Sig':<5}")
print("-" * 70)
for i, (_, row) in enumerate(expr_assoc_df.head(CONFIG['TOP_N_PATHWAYS']).iterrows(), 1):
    sig = "***" if row['significant'] else ""
    pathway_short = row['pathway'][:48]
    print(f"{i:<6} {pathway_short:<50} {row['correlation']:+.3f}  {sig}")

expr_assoc_df.to_csv(f"{CONFIG['output_dir']}/expression_pathway_associations.csv", index=False)
print(f"\nSaved: {CONFIG['output_dir']}/expression_pathway_associations.csv")

# FLUX-BASED ASSOCIATIONS

print("FLUX-BASED ASSOCIATIONS")


flux_associations = []

for flux_col in flux_matched.columns:
    flux_values = flux_matched[flux_col].values
    target_values = target_matched['target'].values
    
    if np.var(flux_values) < 1e-10:
        continue
    
    # Correlation
    corr, pval = spearmanr(flux_values, target_values)
    
    # Mann-Whitney U
    high_flux = flux_values[target_values == 1]
    low_flux = flux_values[target_values == 0]
    
    if len(high_flux) > 0 and len(low_flux) > 0:
        _, pval_mw = mannwhitneyu(high_flux, low_flux, alternative='two-sided')
        mean_high = np.mean(high_flux)
        mean_low = np.mean(low_flux)
        fold_change = mean_high / mean_low if mean_low > 0 else np.inf
    else:
        pval_mw = 1.0
        mean_high = 0
        mean_low = 0
        fold_change = 1.0
    
    flux_associations.append({
        'flux_feature': flux_col,
        'correlation': corr,
        'pval_corr': pval,
        'pval_mw': pval_mw,
        'mean_high': mean_high,
        'mean_low': mean_low,
        'fold_change': fold_change
    })

flux_assoc_df = pd.DataFrame(flux_associations)

if len(flux_assoc_df) > 0:
    # FDR correction
    _, qvals, _, _ = multipletests(flux_assoc_df['pval_corr'], method='fdr_bh')
    flux_assoc_df['qval'] = qvals
    flux_assoc_df['significant'] = qvals < CONFIG['FDR_THRESHOLD']
    flux_assoc_df['abs_corr'] = np.abs(flux_assoc_df['correlation'])
    
    flux_assoc_df = flux_assoc_df.sort_values('abs_corr', ascending=False)
    
    print(f"Tested:      {len(flux_assoc_df)} flux features")
    print(f"Significant: {flux_assoc_df['significant'].sum()} (FDR < {CONFIG['FDR_THRESHOLD']})")
    
    n_show = min(20, len(flux_assoc_df))
    print(f"\nTop {n_show} flux associations:")
    print(f"{'Rank':<6} {'Feature':<35} {'Dir':<5} {'FC':<7} {'Corr':<8} {'Q-val':<10} {'Sig':<5}")
    print("-" * 80)
    for i, (_, row) in enumerate(flux_assoc_df.head(n_show).iterrows(), 1):
        sig = "***" if row['significant'] else ""
        direction = "UP" if row['fold_change'] > 1 else "DOWN"
        feature_short = row['flux_feature'][:33]
        print(f"{i:<6} {feature_short:<35} {direction:<5} {row['fold_change']:<7.2f} {row['correlation']:+.3f}  {row['qval']:<10.2e} {sig}")
    
    flux_assoc_df.to_csv(f"{CONFIG['output_dir']}/flux_associations.csv", index=False)
    print(f"\nSaved: {CONFIG['output_dir']}/flux_associations.csv")
else:
    print("No flux features with variance found!")

# TOP FLUX MODULES (INDEPENDENT ANALYSIS - NO INTEGRATION)


print("TOP FLUX MODULES BY CORRELATION")


print("\nNote: Flux modules (M_X) analyzed independently")
print("      Integration with expression pathways requires module-to-pathway mapping")
print("      (See Option B for integrated analysis)")

if len(flux_assoc_df) > 0:
    # Use top flux features by correlation as "vulnerabilities"
    vuln_df = flux_assoc_df.head(CONFIG['TOP_N_FLUX']).copy()
    vuln_df['vulnerability_score'] = vuln_df['abs_corr']  # Use absolute correlation as score
    
    print(f"\nIdentified {len(vuln_df)} top flux modules (by |correlation|)")
    print(f"  Significant (FDR < 0.05): {vuln_df['significant'].sum()}")
    
    print(f"\nTop 10 Flux Modules:")
    print(f"{'Rank':<6} {'Module':<15} {'Score':<8} {'Corr':<8} {'Q-val':<10} {'FC':<8} {'Sig':<5}")
    print("-" * 75)
    
    for i, (_, row) in enumerate(vuln_df.head(10).iterrows(), 1):
        sig = "***" if row['significant'] else ""
        print(f"{i:<6} {row['flux_feature']:<15} {row['vulnerability_score']:<8.3f} {row['correlation']:+.3f}  {row['qval']:<10.2e} {row['fold_change']:<8.2f} {sig}")
    
    # Save
    vuln_df.to_csv(f"{CONFIG['output_dir']}/top_flux_modules.csv", index=False)
    print(f"\nSaved: {CONFIG['output_dir']}/top_flux_modules.csv")
    
    # Add interpretation
    print("INTERPRETATION GUIDE")
    print("\nHow to interpret these results:")
    print("  1. Each module (M_X) represents a metabolic reaction or pathway")
    print("  2. Positive correlation: Higher flux in high-proliferation regions")
    print("  3. Negative correlation: Lower flux in high-proliferation regions")
    print("  4. Fold change: Flux ratio (high vs low proliferation)")
    print("  5. Q-value < 0.05: Statistically significant after multiple testing")
    print("\nNext steps:")
    print("  - Map significant modules to known metabolic pathways (scFEA documentation)")
    print("  - Validate top modules with metabolic assays")
    print("  - Consider pharmacological inhibition of upregulated pathways")
    
else:
    print("No flux features to analyse")
    vuln_df = None

# VISUALISATIONS


print(" GENERATING VISUALISATIONS")

fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Plot 1: Top expression associations (horizontal bar)
ax1 = fig.add_subplot(gs[0, :2])
top_expr = expr_assoc_df.head(15).sort_values('correlation')
colors = [COLOR_HIGH if x < 0 else COLOR_TERTIARY for x in top_expr['correlation']]
ax1.barh(range(len(top_expr)), top_expr['correlation'], color=colors, alpha=0.7)
ax1.set_yticks(range(len(top_expr)))
ax1.set_yticklabels([p[:40] for p in top_expr['pathway']], fontsize=8)
ax1.set_xlabel('Correlation with Proliferation', fontweight='bold')
ax1.set_title('Top Expression-Based Pathways', fontweight='bold', fontsize=12)
ax1.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax1.grid(alpha=0.3, axis='x')

# Plot 2: Flux associations
if len(flux_assoc_df) > 0:
    ax2 = fig.add_subplot(gs[0, 2])
    top_flux = flux_assoc_df.head(15).sort_values('correlation')
    colors = [COLOR_HIGH if x < 0 else COLOR_TERTIARY for x in top_flux['correlation']]
    ax2.barh(range(len(top_flux)), top_flux['correlation'], color=colors, alpha=0.7)
    ax2.set_yticks(range(len(top_flux)))
    labels = [f.replace('Flux_', '').replace('M_', 'M')[:18] for f in top_flux['flux_feature']]
    ax2.set_yticklabels(labels, fontsize=7)
    ax2.set_xlabel('Correlation', fontweight='bold')
    ax2.set_title('Flux Module Associations', fontweight='bold', fontsize=12)
    ax2.axvline(x=0, color='black', linestyle='--', alpha=0.5)
    ax2.grid(alpha=0.3, axis='x')

# Plot 3: Top flux modules by score
if vuln_df is not None and len(vuln_df) > 0:
    ax3 = fig.add_subplot(gs[1, :2])
    top_vuln = vuln_df.head(12).sort_values('vulnerability_score')
    colors = [COLOR_HIGH if c < 0 else COLOR_TERTIARY for c in [vuln_df.loc[top_vuln.index, 'correlation'].values[i] for i in range(len(top_vuln))]]
    ax3.barh(range(len(top_vuln)), top_vuln['vulnerability_score'], color=colors, alpha=0.7)
    ax3.set_yticks(range(len(top_vuln)))
    labels = [f.replace('Flux_', '').replace('M_', 'M')[:30] for f in top_vuln['flux_feature']]
    ax3.set_yticklabels(labels, fontsize=8)
    ax3.set_xlabel('Vulnerability Score (|Correlation|)', fontweight='bold')
    ax3.set_title('Top Flux Modules', fontweight='bold', fontsize=12)
    ax3.grid(alpha=0.3, axis='x')

# Plot 4: Correlation vs Q-value
if vuln_df is not None and len(vuln_df) > 0:
    ax4 = fig.add_subplot(gs[1, 2])
    scatter_colors = [COLOR_SIG if sig else COLOR_NOT_SIG for sig in vuln_df['significant']]
    ax4.scatter(vuln_df['correlation'], -np.log10(vuln_df['qval']), 
               s=100, alpha=0.6, c=scatter_colors)
    ax4.axhline(y=-np.log10(0.05), color=COLOR_REFERENCE, linestyle='--', alpha=0.5, label='FDR 0.05')
    ax4.axvline(x=0, color='black', linestyle='--', alpha=0.3)
    ax4.set_xlabel('Correlation with Proliferation', fontweight='bold')
    ax4.set_ylabel('-log10(Q-value)', fontweight='bold')
    ax4.set_title('Volcano Plot: Flux Modules', fontweight='bold', fontsize=12)
    ax4.legend()
    ax4.grid(alpha=0.3)

# Plot 5: Flux fold changes
if len(flux_assoc_df) > 0:
    ax5 = fig.add_subplot(gs[2, :2])
    top_fc = flux_assoc_df.head(15).sort_values('fold_change')
    colors = [COLOR_HIGH if x < 1 else COLOR_TERTIARY for x in top_fc['fold_change']]
    y_pos = range(len(top_fc))
    ax5.barh(y_pos, top_fc['fold_change'], color=colors, alpha=0.7)
    ax5.set_yticks(y_pos)
    labels = [f.replace('Flux_', '').replace('M_', 'M')[:25] for f in top_fc['flux_feature']]
    ax5.set_yticklabels(labels, fontsize=7)
    ax5.set_xlabel('Fold Change (High vs Low Proliferation)', fontweight='bold')
    ax5.set_title('Flux Fold Changes', fontweight='bold', fontsize=12)
    ax5.axvline(x=1, color='black', linestyle='--', alpha=0.5)
    ax5.grid(alpha=0.3, axis='x')

# Plot 6: Distribution comparison
ax6 = fig.add_subplot(gs[2, 2])
if len(flux_assoc_df) > 0:
    top_flux_feature = flux_assoc_df.iloc[0]['flux_feature']
    high_values = flux_matched.loc[target_matched['target'] == 1, top_flux_feature]
    low_values = flux_matched.loc[target_matched['target'] == 0, top_flux_feature]
    
    ax6.hist(low_values, bins=30, alpha=0.5, label='Low Prolif', color=COLOR_LOW, density=True)
    ax6.hist(high_values, bins=30, alpha=0.5, label='High Prolif', color=COLOR_HIGH, density=True)
    ax6.set_xlabel('Flux Value', fontweight='bold')
    ax6.set_ylabel('Density', fontweight='bold')
    ax6.set_title(f'Distribution: {top_flux_feature.replace("Flux_", "")[:20]}', 
                  fontweight='bold', fontsize=10)
    ax6.legend()
    ax6.grid(alpha=0.3)

plt.suptitle(f'Flux Interpretation Analysis - Patient {patient_id}', 
             fontsize=16, fontweight='bold', y=0.995)

output_plot = f"{CONFIG['output_dir']}/flux_interpretation_summary_patient_{patient_id}.png"
plt.savefig(output_plot, dpi=300, bbox_inches='tight')
plt.close()

print(f"Saved: {output_plot}")

#  FINAL SUMMARY


print("FLUX INTERPRETATION COMPLETE!")


print(f"\nKey Findings for Patient {patient_id}:")
print(f"  - Expression pathways tested: {len(expr_assoc_df)}")
print(f"  - Expression significant:     {expr_assoc_df['significant'].sum()}")
print(f"  - Flux features tested:       {len(flux_assoc_df)}")
print(f"  - Flux significant:           {flux_assoc_df['significant'].sum()}")

if vuln_df is not None and len(vuln_df) > 0:
    print(f"  - Top flux modules analyzed:  {len(vuln_df)}")
    print(f"  - Significant modules:        {vuln_df['significant'].sum()}")
    
    top_module = vuln_df.iloc[0]
    print(f"\n  TOP FLUX MODULE:")
    print(f"    Module:        {top_module['flux_feature']}")
    print(f"    Score:         {top_module['vulnerability_score']:.3f}")
    print(f"    Correlation:   {top_module['correlation']:+.3f}")
    print(f"    Q-value:       {top_module['qval']:.2e}")
    print(f"    Fold change:   {top_module['fold_change']:.2f}x")
    print(f"    Significant:   {'Yes' if top_module['significant'] else 'No'}")

print(f"\nOutput Files:")
print(f"  All results saved to: {CONFIG['output_dir']}/")
print(f"    - expression_pathway_associations.csv")
print(f"    - flux_associations.csv")
print(f"    - top_flux_modules.csv")
print(f"    - flux_interpretation_summary_patient_{patient_id}.png")

print("\nNext Steps:")
print("  1. Map significant modules to metabolic pathways (see scFEA documentation)")
print("  2. Validate top modules with metabolic assays")
print("  3. Consider therapeutic targeting of upregulated modules")
print("  4. Repeat analysis for other patients (change PATIENT_ID in CONFIG)")
print("  5. For integrated analysis, use Option B (requires module-to-pathway mapping)")


In [ ]:
print("MAPPING scFEA MODULES TO METABOLIC PATHWAYS")

# LOAD MODULE-GENE MAPPING
print("\nLoading module-gene mappings")

module_gene_file = './scFEA/data/module_gene_m168.csv'
module_genes = pd.read_csv(module_gene_file)

print(f"Loaded: {module_gene_file}")
print(f"  Shape: {module_genes.shape}")

# The file has module names in column '1' and genes in 'A', 'A.1', 'A.2', etc.
print("\nModule file structure:")
print("  Module names in column: '1'")
print("  Gene names in columns: 'A', 'A.1', 'A.2', etc.")

print("\nFirst 5 modules:")
print(module_genes[['1', 'A', 'A.1', 'A.2', 'A.3']].head())


# MAP YOUR TOP MODULES TO GENES
print("\nExtracting genes for top flux modules")

top_modules = [157, 135, 2, 92, 123, 8, 57, 47, 75, 132, 
               19, 6, 151, 150, 67, 161, 100, 107, 153, 159]

flux_results = pd.read_csv('./flux_interpretation_results/flux_associations.csv')

print(f"\nProcessing {len(top_modules)} top modules")

module_annotations = []

for module_num in top_modules:
    module_name = f'M_{module_num}'
    
    # Find row for this module
    module_row = module_genes[module_genes['1'] == module_name]
    
    if len(module_row) > 0:
        # Get all gene columns (columns starting with 'A')
        gene_columns = [col for col in module_genes.columns if col.startswith('A')]
        
        # Extract genes, removing NaN values
        genes = []
        for col in gene_columns:
            gene = module_row[col].values[0]
            if pd.notna(gene):
                genes.append(gene)
        
        # Get flux stats
        flux_stats = flux_results[flux_results['flux_feature'] == module_name]
        
        if len(flux_stats) > 0:
            module_annotations.append({
                'Module': module_name,
                'Module_Num': module_num,
                'N_Genes': len(genes),
                'Genes': ', '.join(genes[:5]) + ('...' if len(genes) > 5 else ''),
                'All_Genes': genes,
                'Correlation': flux_stats.iloc[0]['correlation'],
                'Q_value': flux_stats.iloc[0]['qval'],
                'Fold_Change': flux_stats.iloc[0]['fold_change'],
                'Direction': 'UP' if flux_stats.iloc[0]['fold_change'] > 1 else 'DOWN'
            })
            print(f"  {module_name}: {len(genes)} genes")
        else:
            print(f"  {module_name}: Found genes but no flux stats")
    else:
        print(f"  {module_name}: Not found in module file")

module_anno_df = pd.DataFrame(module_annotations)

print(f"\nSuccessfully mapped {len(module_anno_df)} modules")


print("MODULE-GENE MAPPINGS")


for i, row in module_anno_df.head(10).iterrows():
    print(f"\n{row['Module']} ({row['Direction']}, r={row['Correlation']:.3f})")
    print(f"  Genes ({row['N_Genes']}): {row['Genes']}")


# INFER METABOLIC PATHWAYS FROM GENES
print("\nInferring metabolic pathways from gene content")

pathway_markers = {
    'Glycolysis': ['HK1', 'HK2', 'GPI', 'PFKL', 'PFKM', 'PFKP', 'ALDOA', 'ALDOB', 'ALDOC', 
                   'TPI1', 'GAPDH', 'PGK1', 'PGAM1', 'ENO1', 'ENO2', 'PKM', 'LDHA', 'LDHB'],
    'TCA_Cycle': ['CS', 'ACO1', 'ACO2', 'IDH1', 'IDH2', 'IDH3A', 'OGDH', 'SUCLA2', 'SUCLG1', 
                  'SDHA', 'SDHB', 'FH', 'MDH1', 'MDH2'],
    'OXPHOS': ['NDUFA', 'NDUFB', 'NDUFS', 'NDUFV', 'COX', 'CYC1', 'UQCR', 'ATP5', 'MT-'],
    'Pentose_Phosphate': ['G6PD', 'PGD', 'RPIA', 'RPE', 'TKT', 'TALDO1'],
    'Fatty_Acid_Synthesis': ['ACACA', 'ACACB', 'FASN', 'ACLY', 'SCD', 'ELOVL'],
    'Fatty_Acid_Oxidation': ['CPT1A', 'CPT1B', 'CPT2', 'ACAD', 'HADH', 'ECHS1'],
    'Amino_Acid_Metabolism': ['GLS', 'GLS2', 'GLUL', 'GOT1', 'GOT2', 'GPT', 'GPT2', 'ASS1', 
                               'ASL', 'PHGDH', 'PSAT1', 'PSPH'],
    'Purine_Metabolism': ['PPAT', 'GART', 'PAICS', 'ADSL', 'ATIC', 'IMPDH1', 'IMPDH2'],
    'Pyrimidine_Metabolism': ['CAD', 'DHODH', 'UMPS', 'CTPS1', 'CTPS2'],
    'One_Carbon': ['MTHFD1', 'MTHFD2', 'SHMT1', 'SHMT2', 'TYMS', 'DHFR'],
}

def infer_pathway(genes):
    pathway_scores = {}
    
    for pathway, markers in pathway_markers.items():
        overlap = sum(1 for gene in genes if any(marker in gene for marker in markers))
        if overlap > 0:
            pathway_scores[pathway] = overlap
    
    if pathway_scores:
        best_pathway = max(pathway_scores, key=pathway_scores.get)
        score = pathway_scores[best_pathway]
        return best_pathway, score
    else:
        return 'Unknown', 0

for idx, row in module_anno_df.iterrows():
    pathway, score = infer_pathway(row['All_Genes'])
    module_anno_df.loc[idx, 'Inferred_Pathway'] = pathway
    module_anno_df.loc[idx, 'Pathway_Confidence'] = score

print("TOP MODULES WITH PATHWAY ANNOTATIONS")

print(f"\n{'Rank':<6} {'Module':<10} {'Pathway':<25} {'Conf':<6} {'Corr':<8} {'Dir':<6} {'Genes':<10}")

for i, (_, row) in enumerate(module_anno_df.head(20).iterrows(), 1):
    print(f"{i:<6} {row['Module']:<10} {row['Inferred_Pathway']:<25} {row['Pathway_Confidence']:<6.0f} "
          f"{row['Correlation']:>7.3f} {row['Direction']:<6} {row['N_Genes']:<10}")

module_anno_df.to_csv('./flux_interpretation_results/module_pathway_annotations.csv', index=False)
print(f"\nSaved: ./flux_interpretation_results/module_pathway_annotations.csv")


# DETAILED REPORT FOR TOP 10 MODULES
print("DETAILED ANALYSIS: TOP 10 FLUX MODULES")


for i, (_, row) in enumerate(module_anno_df.head(10).iterrows(), 1):
    print(f"RANK {i}: {row['Module']}")
    
    
    print(f"\nStatistics:")
    print(f"  Correlation:    {row['Correlation']:+.3f}")
    print(f"  Q-value:        {row['Q_value']:.2e}")
    print(f"  Fold Change:    {row['Fold_Change']:.2f}x")
    print(f"  Direction:      {row['Direction']}")
    
    print(f"\nGene Content:")
    print(f"  Number of genes: {row['N_Genes']}")
    print(f"  Gene list: {', '.join(row['All_Genes'][:10])}")
    if row['N_Genes'] > 10:
        print(f"  ... and {row['N_Genes'] - 10} more")
    
    print(f"\nPathway Annotation:")
    print(f"  Inferred pathway: {row['Inferred_Pathway']}")
    print(f"  Confidence:       {row['Pathway_Confidence']} matching genes")
    
    print(f"\nBiological Interpretation:")
    if row['Direction'] == 'UP':
        print(f"  This pathway is UPREGULATED in high-proliferation regions")
        print(f"  Potential therapeutic target")
    else:
        print(f"  This pathway is DOWNREGULATED in high-proliferation regions")
        print(f"  May represent metabolic compensation")

In [ ]:
# BIOLOGICAL INTERPRETATION OF TOP FLUX MODULES

print("COMPREHENSIVE BIOLOGICAL INTERPRETATION")

# Manually annotate modules that were marked as "Unknown"
manual_annotations = {
    'M_157': {
        'pathway': 'Nucleotide_Biosynthesis',
        'genes': ['RRM1', 'RRM2', 'RRM2B'],
        'function': 'Ribonucleotide reductase - converts ribonucleotides to deoxyribonucleotides for DNA synthesis',
        'therapeutic': ['Hydroxyurea', 'Gemcitabine', 'Triapine', 'Clofarabine'],
        'biology': 'Essential for DNA replication and cell division. Highly upregulated in proliferating cancer cells.'
    },
    'M_135': {
        'pathway': 'Purine_Metabolism',
        'genes': ['ATIC', 'NME1', 'ENTPD family'],
        'function': 'Purine nucleotide synthesis and salvage',
        'therapeutic': ['Methotrexate', 'Pemetrexed', '6-Mercaptopurine'],
        'biology': 'Provides purines for RNA/DNA synthesis. Critical for rapidly dividing cells.'
    },
    'M_2': {
        'pathway': 'Glycolysis',
        'genes': ['ALDOA', 'PFKL', 'PFKM', 'PFKP', 'GPI', 'TPI1'],
        'function': 'Glucose breakdown to pyruvate',
        'therapeutic': ['2-Deoxyglucose', 'Lonidamine', '3-Bromopyruvate', 'Dichloroacetate'],
        'biology': 'Warburg effect - cancer cells preferentially use glycolysis even with oxygen present.'
    },
    'M_92': {
        'pathway': 'Glutamine_Transport',
        'genes': ['SLC1A5'],
        'function': 'Glutamine transporter (ASCT2) - imports glutamine into cells',
        'therapeutic': ['V-9302', 'GPNA', 'Glutaminase inhibitors'],
        'biology': 'DOWNREGULATED - suggests reduced glutamine dependence or alternative uptake mechanisms.'
    },
    'M_123': {
        'pathway': 'N-Glycan_Biosynthesis',
        'genes': ['MGAT4A', 'MGAT4B', 'MGAT4C'],
        'function': 'N-glycosylation of proteins',
        'therapeutic': ['Tunicamycin', 'Swainsonine'],
        'biology': 'Altered glycosylation patterns in cancer affect metastasis and immune evasion.'
    },
    'M_8': {
        'pathway': 'TCA_Cycle',
        'genes': ['ACO1', 'ACO2', 'IDH1', 'IDH2', 'IDH3A'],
        'function': 'Citric acid cycle',
        'therapeutic': ['CPI-613', 'AG-221 (Enasidenib)', 'AG-120 (Ivosidenib)'],
        'biology': 'Energy production and biosynthetic precursors. IDH mutations common in cancer.'
    },
    'M_57': {
        'pathway': 'Tyrosine_Metabolism',
        'genes': ['GOT1', 'GOT2', 'TAT', 'HPD', 'FAH'],
        'function': 'Tyrosine degradation and transamination',
        'therapeutic': ['Aminotransferase inhibitors'],
        'biology': 'Tyrosine metabolism linked to melanin synthesis and neurotransmitter production.'
    },
    'M_47': {
        'pathway': 'Fatty_Acid_Oxidation',
        'genes': ['ACACA', 'ACACB', 'HADHA', 'ECHS1'],
        'function': 'Beta-oxidation of fatty acids',
        'therapeutic': ['Etomoxir', 'Perhexiline', 'Ranolazine'],
        'biology': 'DOWNREGULATED - cancer cells shift from oxidation to synthesis of fatty acids.'
    },
    'M_75': {
        'pathway': 'Metabolite_Transport',
        'genes': ['SLC13A3', 'SLC25A10'],
        'function': 'Dicarboxylate and citrate transporters',
        'therapeutic': ['Transport inhibitors'],
        'biology': 'Maintains metabolite exchange between mitochondria and cytoplasm.'
    },
    'M_132': {
        'pathway': 'Heparan_Sulfate_Biosynthesis',
        'genes': ['EXT1', 'EXT2', 'EXTL1', 'EXTL2', 'EXTL3'],
        'function': 'Glycosaminoglycan synthesis',
        'therapeutic': ['Heparanase inhibitors'],
        'biology': 'Extracellular matrix remodeling. Important for growth factor signaling.'
    }
}

print("\nTOP 10 METABOLIC VULNERABILITIES - DETAILED INTERPRETATION")


module_order = ['M_157', 'M_135', 'M_2', 'M_92', 'M_123', 
                'M_8', 'M_57', 'M_47', 'M_75', 'M_132']

for rank, module in enumerate(module_order, 1):
    if module in manual_annotations:
        info = manual_annotations[module]
        
        print(f"\n{rank}. {module}: {info['pathway']}")
        print(f"   Genes: {', '.join(info['genes'])}")
        print(f"   Function: {info['function']}")
        print(f"   Biology: {info['biology']}")
        print(f"   Therapeutic targets: {', '.join(info['therapeutic'][:3])}")

print("\nKEY BIOLOGICAL THEMES")


print("\n1. ANABOLIC METABOLISM (Upregulated)")
print("   - Nucleotide biosynthesis (M_157, M_135)")
print("   - Glycolysis (M_2)")
print("   - TCA cycle (M_8)")
print("   - N-glycan synthesis (M_123)")
print("   Interpretation: Cancer cells building blocks for growth and division")

print("\n2. WARBURG EFFECT")
print("   - High glycolysis (M_2: r=+0.259)")
print("   - Low fatty acid oxidation (M_47: r=-0.236)")
print("   Interpretation: Classic metabolic reprogramming in cancer")

print("\n3. AMINO ACID METABOLISM")
print("   - Tyrosine metabolism upregulated (M_57)")
print("   - Glutamine transport downregulated (M_92)")
print("   Interpretation: Altered nitrogen metabolism and anaplerosis")

print("\n4. PROLIFERATION SIGNATURE")
print("   - DNA synthesis enzymes (M_157: RRM1/RRM2)")
print("   - Purine metabolism (M_135)")
print("   Interpretation: Markers of rapid cell division")


print("\nTOP THERAPEUTIC OPPORTUNITIES")

therapeutic_targets = [
    {
        'rank': 1,
        'module': 'M_157',
        'target': 'Ribonucleotide Reductase',
        'drug': 'Hydroxyurea or Gemcitabine',
        'rationale': 'Strongest correlation (r=0.284), essential for DNA synthesis',
        'status': 'FDA approved (both drugs)'
    },
    {
        'rank': 2,
        'module': 'M_135',
        'target': 'Purine Metabolism',
        'drug': 'Methotrexate or Pemetrexed',
        'rationale': 'Strong correlation (r=0.262), blocks nucleotide synthesis',
        'status': 'FDA approved'
    },
    {
        'rank': 3,
        'module': 'M_2',
        'target': 'Glycolysis',
        'drug': '2-Deoxyglucose or Lonidamine',
        'rationale': 'High correlation (r=0.259), 21% flux increase',
        'status': 'Clinical trials'
    },
    {
        'rank': 4,
        'module': 'M_8',
        'target': 'TCA Cycle / IDH',
        'drug': 'CPI-613 or Ivosidenib',
        'rationale': 'Strong correlation (r=0.241), energy metabolism',
        'status': 'FDA approved for IDH mutations'
    },
    {
        'rank': 5,
        'module': 'M_47',
        'target': 'Fatty Acid Oxidation',
        'drug': 'Etomoxir',
        'rationale': 'Downregulated (r=-0.236), metabolic vulnerability',
        'status': 'Experimental'
    }
]

for target in therapeutic_targets:
    print(f"\n{target['rank']}. {target['module']}: {target['target']}")
    print(f"   Recommended drug: {target['drug']}")
    print(f"   Rationale: {target['rationale']}")
    print(f"   Status: {target['status']}")

print("\nCOMBINATION THERAPY STRATEGIES")


print("\n1. Dual Nucleotide Blockade")
print("   Combine: M_157 inhibitor + M_135 inhibitor")
print("   Example: Hydroxyurea + Methotrexate")
print("   Rationale: Block both DNA synthesis and purine metabolism")

print("\n2. Glycolysis + TCA Inhibition")
print("   Combine: M_2 inhibitor + M_8 inhibitor")
print("   Example: 2-DG + CPI-613")
print("   Rationale: Complete metabolic shutdown")

print("\n3. Metabolic + Standard Therapy")
print("   Combine: M_157 inhibitor + Chemotherapy")
print("   Example: Gemcitabine + Paclitaxel")
print("   Rationale: Exploit metabolic vulnerability during treatment")

print("\nVALIDATION EXPERIMENTS")


print("\n1. IN VITRO VALIDATION")
print("   a) Drug screening on breast cancer cell lines")
print("      - Test hydroxyurea on high vs low proliferation cells")
print("      - Measure IC50 values")
print("   b) Metabolic assays")
print("      - Seahorse analyzer for glycolysis/OXPHOS")
print("      - LC-MS metabolomics for nucleotides")
print("   c) Gene knockdown")
print("      - siRNA against RRM1/RRM2")
print("      - Measure proliferation and viability")

print("\n2. SPATIAL VALIDATION")
print("   a) Immunohistochemistry")
print("      - Stain for RRM2, PFKP, IDH1 in tissue")
print("      - Confirm spatial patterns match flux data")
print("   b) Spatial metabolomics")
print("      - MALDI imaging of metabolites")
print("      - Confirm nucleotide/glucose gradients")

print("\n3. IN VIVO VALIDATION")
print("   a) Patient-derived xenografts (PDX)")
print("      - Test drug combinations in mice")
print("      - Measure tumor growth and survival")
print("   b) Clinical correlation")
print("      - Check if RRM2 expression predicts response")
print("      - Retrospective analysis of gemcitabine-treated patients")

print("SUMMARY")

print("\nYou have identified a clear metabolic signature of proliferative")
print("breast cancer regions characterized by:")
print("  1. Upregulated nucleotide biosynthesis (M_157, M_135)")
print("  2. Enhanced glycolysis (M_2, M_6)")
print("  3. Active TCA cycle (M_8)")
print("  4. Suppressed fatty acid oxidation (M_47)")
print("  5. Reduced glutamine uptake (M_92)")

print("\nThis represents the ANABOLIC PHENOTYPE of cancer:")
print("  - Building blocks for growth")
print("  - Energy production through aerobic glycolysis")
print("  - Reduced catabolism")

print("\nTOP THERAPEUTIC TARGET:")
print("  M_157 (Ribonucleotide Reductase)")
print("  - Strongest association (r=0.284, q=1.99e-89)")
print("  - Druggable target (hydroxyurea, gemcitabine)")
print("  - FDA-approved drugs available")
print("  - Can be combined with standard chemotherapy")

In [ ]:
# CHECK AND FIX PATIENT 7 scFEA INPUT

print("CHECKING PATIENT 7 scFEA INPUT")

input_file = './scFEA/input/Patient_7_expression.csv'

# Check current size
if Path(input_file).exists():
    # Read just the header to get number of columns (spots)
    expr_header = pd.read_csv(input_file, index_col=0, nrows=0)
    n_spots = len(expr_header.columns)
    
    # Read just the index to get number of genes
    expr_index = pd.read_csv(input_file, usecols=[0])
    n_genes = len(expr_index)
    
    print(f"Current input file:")
    print(f"  Genes: {n_genes}")
    print(f"  Spots: {n_spots}")
    
    # scFEA is slow with >5000 spots
    MAX_SPOTS = 5000
    
    if n_spots > MAX_SPOTS:
        print(f"\nWARNING: {n_spots} spots is too many for scFEA")
        print(f"Downsampling to {MAX_SPOTS} spots...")
        
        # Load full file
        print("  Loading expression matrix...")
        expr_df = pd.read_csv(input_file, index_col=0)
        
        # Randomly select spots
        np.random.seed(42)
        selected_spots = np.random.choice(expr_df.columns, MAX_SPOTS, replace=False)
        expr_downsampled = expr_df[selected_spots]
        
        print(f"  Downsampled: {expr_downsampled.shape[0]} genes x {expr_downsampled.shape[1]} spots")
        
        # Backup original
        backup_file = './scFEA/input/Patient_7_expression_FULL.csv'
        if not Path(backup_file).exists():
            shutil.copy(input_file, backup_file)
            print(f"  Backed up original to: {backup_file}")
        
        # Save downsampled
        expr_downsampled.to_csv(input_file)
        print(f"  Saved downsampled file: {input_file}")
        
        del expr_df, expr_downsampled
        import gc
        gc.collect()
    else:
        print(f"\nInput file size is OK for scFEA")
else:
    print(f"ERROR: Input file not found: {input_file}")

In [ ]:
# scFEA FOR PATIENT 7 

print("RUNNING scFEA FOR PATIENT 7")

# Create output directory
Path('./scFEA/results/Patient_7').mkdir(parents=True, exist_ok=True)

# Verify input file size
input_file = './scFEA/input/Patient_7_expression.csv'
if Path(input_file).exists():
    expr_header = pd.read_csv(input_file, index_col=0, nrows=0)
    n_spots = len(expr_header.columns)
    print(f"Input file: {n_spots} spots")
    print(f"Estimated runtime: 20-40 minutes")
else:
    print(f"ERROR: Input file not found")

# Run scFEA with correct parameter name
print("\nRunning scFEA...")

cmd = """
python ./scFEA/src/scFEA.py \
    --data_dir ./scFEA/data \
    --input_dir ./scFEA/input \
    --res_dir ./scFEA/results/Patient_7 \
    --test_file Patient_7_expression.csv \
    --moduleGene_file module_gene_m168.csv \
    --stoichiometry_matrix cmMat_c70_m168.csv \
    --output_flux_file Patient_7_flux.csv \
    --output_balance_file Patient_7_balance.csv \
    --sc_imputation True \
    --train_epoch 50
"""

exit_code = os.system(cmd)

if exit_code == 0:
    print("\nscFEA completed successfully")
else:
    print(f"\nscFEA finished with exit code: {exit_code}")

# Check for output files
flux_file = './scFEA/results/Patient_7/Patient_7_flux.csv'
balance_file = './scFEA/results/Patient_7/Patient_7_balance.csv'

# Move files if in current directory
if not Path(flux_file).exists() and Path('./Patient_7_flux.csv').exists():
    shutil.move('./Patient_7_flux.csv', flux_file)
    print(f"Moved flux file to: {flux_file}")

if not Path(balance_file).exists() and Path('./Patient_7_balance.csv').exists():
    shutil.move('./Patient_7_balance.csv', balance_file)

# Verify
if Path(flux_file).exists():
    flux_df = pd.read_csv(flux_file, index_col=0)
    print(f"\nFlux file created: {flux_file}")
    print(f"  Shape: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")
else:
    print(f"\nERROR: Flux file not created")

print("\nDONE")

In [ ]:
import gseapy as gp


patient_id = 7

# STEP 1: Process flux output
print("\nStep 1: Processing flux output")

flux_file = f'./scFEA/results/Patient_{patient_id}/Patient_{patient_id}_flux.csv'

if not Path(flux_file).exists():
    print(f"  ERROR: Flux file not found: {flux_file}")
else:
    flux_df = pd.read_csv(flux_file, index_col=0)
    print(f"  Original shape: {flux_df.shape}")
    
    # Check if transposed
    if flux_df.shape[0] > flux_df.shape[1]:
        print(f"  Transposing")
        flux_df = flux_df.T
    
    print(f"  Corrected shape: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")
    
    # Save corrected flux
    corrected_file = f'./scFEA/results/Patient_{patient_id}/Patient_{patient_id}_flux_corrected.csv'
    flux_df.to_csv(corrected_file)
    
    # Transpose for processing (spots x modules)
    flux_spots = flux_df.T
    print(f"  For processing: {flux_spots.shape[0]} spots x {flux_spots.shape[1]} modules")
    
    # Check variance
    variances = flux_spots.var(axis=0)
    n_with_variance = (variances > 1e-10).sum()
    print(f"  Modules with variance: {n_with_variance}/{len(variances)}")
    
    # Filter to modules with variance
    modules_with_variance = variances[variances > 1e-10].index.tolist()
    flux_filtered = flux_spots[modules_with_variance]
    
    # Save RAW modules
    output_file = f'Patient_{patient_id}_flux_RAW_modules.csv'
    flux_filtered.to_csv(output_file)
    print(f"  Saved: {output_file}")
    print(f"  Final: {flux_filtered.shape[0]} spots x {flux_filtered.shape[1]} modules")

# STEP 2: Export pathway scores and target
print("\nStep 2: Exporting pathway scores and target")

h5_path = 'Patient_7/Visium_HD_16um_filtered_feature_bc_matrix.h5'

# Load expression data
print("  Loading expression data...")
adata = sc.read_10x_h5(h5_path)
adata.var_names_make_unique()
print(f"  Loaded: {adata.n_obs} spots x {adata.n_vars} genes")

# Get spot names from flux data to match
flux_spots_list = flux_filtered.index.tolist()
print(f"  scFEA spots: {len(flux_spots_list)}")

# Try to match spots
common_spots = [s for s in flux_spots_list if s in adata.obs_names]
print(f"  Common spots: {len(common_spots)}")

if len(common_spots) > 0:
    adata = adata[common_spots, :].copy()
elif len(flux_spots_list) <= adata.n_obs:
    # Use first N spots if names don't match
    print("  Warning: Spot names don't match, using positional matching")
    adata = adata[:len(flux_spots_list), :].copy()
    adata.obs_names = flux_spots_list

# QC
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
print(f"  After QC: {adata.n_obs} spots x {adata.n_vars} genes")

# Normalize
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Score pathways
print("  Scoring pathways...")

pathway_dbs = ['MSigDB_Hallmark_2020', 'KEGG_2021_Human', 'Reactome_2022']
pathway_scores_dict = {}

for db_name in pathway_dbs:
    try:
        pathway_db = gp.get_library(name=db_name, organism='Human')
        prefix = db_name.split('_')[0]
        print(f"    {db_name}: {len(pathway_db)} pathways")
        
        for pathway_name, gene_list in pathway_db.items():
            genes_present = [g for g in gene_list if g in adata.var_names]
            
            if len(genes_present) >= 5:
                pathway_expr = adata[:, genes_present].X
                if hasattr(pathway_expr, 'toarray'):
                    pathway_expr = pathway_expr.toarray()
                
                col_name = f'{prefix}_{pathway_name}'
                pathway_scores_dict[col_name] = np.mean(pathway_expr, axis=1)
    except Exception as e:
        print(f"    Error with {db_name}: {e}")

pathway_scores = pd.DataFrame(pathway_scores_dict, index=adata.obs_names)
print(f"  Scored {len(pathway_scores.columns)} pathways")

# Save pathway scores
pathway_file = f'Patient_{patient_id}_pathway_scores.csv'
pathway_scores.to_csv(pathway_file)
print(f"  Saved: {pathway_file}")

# Create target variable
print("  Creating target variable...")

PROLIFERATION_GENES = [
    'MKI67', 'PCNA', 'TOP2A', 'CDK1', 'CCNB1', 'CCNB2',
    'BIRC5', 'UBE2C', 'CENPF', 'AURKB', 'CDC20', 'BUB1',
    'PLK1', 'CCNA2', 'MCM2', 'MCM3', 'MCM4', 'MCM5', 'MCM6', 'MCM7'
]

prolif_genes_present = [g for g in PROLIFERATION_GENES if g in adata.var_names]
print(f"  Found {len(prolif_genes_present)}/{len(PROLIFERATION_GENES)} proliferation genes")

if len(prolif_genes_present) > 0:
    prolif_expr = adata[:, prolif_genes_present].X
    if hasattr(prolif_expr, 'toarray'):
        prolif_expr = prolif_expr.toarray()
    prolif_score = np.mean(prolif_expr, axis=1)
    threshold = np.percentile(prolif_score, 75)
    target = (prolif_score > threshold).astype(int)
    
    n_high = target.sum()
    n_low = len(target) - n_high
    print(f"  High proliferation: {n_high} ({100*n_high/len(target):.1f}%)")
    print(f"  Low proliferation: {n_low} ({100*n_low/len(target):.1f}%)")
else:
    print("  ERROR: No proliferation genes found")
    target = np.zeros(len(adata))

# Save target
target_df = pd.DataFrame({
    'target': target
}, index=adata.obs_names)
target_file = f'Patient_{patient_id}_target.csv'
target_df.to_csv(target_file)
print(f"  Saved: {target_file}")

# Verify all files exist
print("\nVerification:")
files_to_check = [
    f'Patient_{patient_id}_flux_RAW_modules.csv',
    f'Patient_{patient_id}_pathway_scores.csv',
    f'Patient_{patient_id}_target.csv'
]

all_exist = True
for f in files_to_check:
    exists = Path(f).exists()
    status = "OK" if exists else "MISSING"
    print(f"  {f}: {status}")
    if not exists:
        all_exist = False

if all_exist:
    print("\nPATIENT 7 PROCESSING COMPLETE")
else:
    print("\nWARNING: Some files missing")

In [ ]:
# MULTI-PATIENT FLUX ANALYSIS PIPELINE

print("MULTI-PATIENT FLUX ANALYSIS PIPELINE")

PATIENTS = [2, 3, 4, 5, 6, 7]
OUTPUT_DIR = './multi_patient_results'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# Patient 7 config (Visium HD)
PATIENT_7_CONFIG = {
    'data_dir': 'Patient_7/',
    'h5_file': 'Visium_HD_16um_filtered_feature_bc_matrix.h5',
    'is_visium_hd': True
}

# Check which steps are needed for each patient
print("\nChecking patient data status")

patient_status = {}
for patient_id in [1] + PATIENTS:
    status = {
        'scfea_input': Path(f'./scFEA/input/Patient_{patient_id}_expression.csv').exists(),
        'scfea_output': Path(f'./scFEA/results/Patient_{patient_id}/Patient_{patient_id}_flux.csv').exists(),
        'flux_processed': Path(f'Patient_{patient_id}_flux_RAW_modules.csv').exists(),
        'pathway_scores': Path(f'Patient_{patient_id}_pathway_scores.csv').exists(),
        'target': Path(f'Patient_{patient_id}_target.csv').exists(),
    }
    patient_status[patient_id] = status
    
    # Mark Patient 7 as Visium HD
    patient_type = " (Visium HD)" if patient_id == 7 else ""
    print(f"\nPatient {patient_id}{patient_type}:")
    for step, exists in status.items():
        symbol = "Done" if exists else "TODO"
        print(f"  {step}: {symbol}")

print("\nPIPELINE RECOMMENDATION")

# Determine what needs to be done
patients_need_scfea_input = [p for p in PATIENTS if not patient_status[p]['scfea_input']]
patients_need_scfea_run = [p for p in PATIENTS if not patient_status[p]['scfea_output']]
patients_need_flux_processing = [p for p in PATIENTS if not patient_status[p]['flux_processed']]
patients_need_data_export = [p for p in PATIENTS if not patient_status[p]['pathway_scores']]

if patients_need_scfea_input:
    print(f"\nPatients needing scFEA input preparation: {patients_need_scfea_input}")
    print("  Action: Run Cell 9 from ScFEA notebook for each patient")
    if 7 in patients_need_scfea_input:
        print("  Note: Patient 7 is Visium HD - will be downsampled to ~10000 spots")

if patients_need_scfea_run:
    print(f"\nPatients needing scFEA execution: {patients_need_scfea_run}")
    print("  Action: Run scFEA for each patient (10-30 min each)")
    if 7 in patients_need_scfea_run:
        print("  Note: Patient 7 may take longer due to larger spot count")

if patients_need_flux_processing:
    print(f"\nPatients needing flux processing: {patients_need_flux_processing}")
    print("  Action: Run flux processing for each patient")

if patients_need_data_export:
    print(f"\nPatients needing data export: {patients_need_data_export}")
    print("  Action: Export pathway scores and target for each patient")

if not any([patients_need_scfea_input, patients_need_scfea_run, 
            patients_need_flux_processing, patients_need_data_export]):
    print("\nAll patients have complete data. Ready for analysis.")

# Summary table
print("\n\nSUMMARY TABLE")
print(f"{'Patient':<12} | {'Input':<6} | {'scFEA':<6} | {'Flux':<6} | {'Pathway':<8} | {'Target':<6}")
print("-" * 60)
for patient_id in [1] + PATIENTS:
    status = patient_status[patient_id]
    row = f"Patient_{patient_id:<4}"
    if patient_id == 7:
        row += " (HD)"
    else:
        row += "     "
    row += f" | {'Yes':<6}" if status['scfea_input'] else f" | {'No':<6}"
    row += f" | {'Yes':<6}" if status['scfea_output'] else f" | {'No':<6}"
    row += f" | {'Yes':<6}" if status['flux_processed'] else f" | {'No':<6}"
    row += f" | {'Yes':<8}" if status['pathway_scores'] else f" | {'No':<8}"
    row += f" | {'Yes':<6}" if status['target'] else f" | {'No':<6}"
    print(row)

print(f"\nTotal patients: {len([1] + PATIENTS)} (including Patient 1)")
print(f"  Standard Visium: {len([p for p in [1] + PATIENTS if p != 7])}")
print(f"  Visium HD: 1 (Patient 7)")

In [ ]:
# AUTOMATED FLUX PROCESSING FOR PATIENTS 2-7

print("AUTOMATED FLUX PROCESSING FOR PATIENTS 2-7")

PATIENTS = [2, 3, 4, 5, 6, 7]

# Process flux outputs (transpose and filter)
print("\nStep 1: Processing flux outputs")

for patient_id in PATIENTS:
    patient_type = " (Visium HD)" if patient_id == 7 else ""
    print(f"\nProcessing Patient {patient_id}{patient_type}...")
    
    # Load scFEA output
    flux_file = f'./scFEA/results/Patient_{patient_id}/Patient_{patient_id}_flux.csv'
    
    if not Path(flux_file).exists():
        print(f"  ERROR: Flux file not found: {flux_file}")
        continue
    
    flux_df = pd.read_csv(flux_file, index_col=0)
    print(f"  Original shape: {flux_df.shape}")
    
    # Check if transposed
    if flux_df.shape[0] > flux_df.shape[1]:
        print(f"  Transposing")
        flux_df = flux_df.T
    
    print(f"  Corrected shape: {flux_df.shape[0]} modules x {flux_df.shape[1]} spots")
    
    # Save corrected flux
    corrected_file = f'./scFEA/results/Patient_{patient_id}/Patient_{patient_id}_flux_corrected.csv'
    flux_df.to_csv(corrected_file)
    
    # Transpose for processing (spots x modules)
    flux_spots = flux_df.T
    print(f"  For processing: {flux_spots.shape[0]} spots x {flux_spots.shape[1]} modules")
    
    # Check variance
    variances = flux_spots.var(axis=0)
    n_with_variance = (variances > 1e-10).sum()
    
    print(f"  Modules with variance: {n_with_variance}/{len(variances)}")
    print(f"  Mean variance: {variances.mean():.6e}")
    
    # Filter to modules with variance
    modules_with_variance = variances[variances > 1e-10].index.tolist()
    flux_filtered = flux_spots[modules_with_variance]
    
    # Save RAW modules
    output_file = f'Patient_{patient_id}_flux_RAW_modules.csv'
    flux_filtered.to_csv(output_file)
    
    print(f"  Saved: {output_file}")
    print(f"  Final: {flux_filtered.shape[0]} spots x {flux_filtered.shape[1]} modules")

# Summary
print("\n\nSUMMARY")
print(f"{'Patient':<15} | {'Spots':>8} | {'Modules':>8} | {'Status':<10}")
print("-" * 50)

for patient_id in PATIENTS:
    output_file = f'Patient_{patient_id}_flux_RAW_modules.csv'
    if Path(output_file).exists():
        df = pd.read_csv(output_file, index_col=0)
        patient_label = f"Patient_{patient_id}"
        if patient_id == 7:
            patient_label += " (HD)"
        print(f"{patient_label:<15} | {df.shape[0]:>8} | {df.shape[1]:>8} | {'Done':<10}")
    else:
        print(f"Patient_{patient_id:<8} | {'N/A':>8} | {'N/A':>8} | {'Missing':<10}")

print("\nFlux processing complete for all patients")

In [ ]:
# EXPORT PATHWAY SCORES AND TARGETS FOR PATIENTS 2-7


print("EXPORTING PATHWAY SCORES AND TARGETS FOR PATIENTS 2-7")

PATIENTS = [2, 3, 4, 5, 6, 7]

PROLIFERATION_GENES = [
    'MKI67', 'PCNA', 'TOP2A', 'CDK1', 'CCNB1', 'CCNB2',
    'BIRC5', 'UBE2C', 'CENPF', 'AURKB', 'CDC20', 'BUB1',
    'PLK1', 'CCNA2', 'MCM2', 'MCM3', 'MCM4', 'MCM5', 'MCM6', 'MCM7'
]

def process_patient(patient_id, h5_file_path, is_visium_hd=False):
    
    patient_type = " (Visium HD)" if is_visium_hd else ""
    print(f"\nProcessing Patient {patient_id}{patient_type}")
    
    # Load data
    print("  Loading data")
    adata = sc.read_10x_h5(h5_file_path)
    print(f"    Loaded: {adata.n_obs} spots x {adata.n_vars} genes")
    
    # Make gene names unique
    adata.var_names_make_unique()
    print(f"    Made gene names unique")
    
    # For Visium HD, check if we need to subset to match scFEA spots
    if is_visium_hd:
        # Try to load scFEA spots to match
        flux_file = f'Patient_{patient_id}_flux_RAW_modules.csv'
        if Path(flux_file).exists():
            flux_df = pd.read_csv(flux_file, index_col=0)
            scfea_spots = flux_df.index.tolist()
            
            # Find matching spots
            common_spots = [s for s in scfea_spots if s in adata.obs_names]
            
            if len(common_spots) > 0:
                print(f"    Matching to scFEA spots: {len(common_spots)} spots")
                adata = adata[common_spots, :].copy()
            else:
                print(f"    Warning: No matching spots found with scFEA data")
                print(f"    Using first {len(scfea_spots)} spots")
                adata = adata[:len(scfea_spots), :].copy()
        else:
            # Downsample if too large
            if adata.n_obs > 15000:
                print(f"    Downsampling from {adata.n_obs} to 15000 spots")
                np.random.seed(42)
                idx = np.random.choice(adata.n_obs, 15000, replace=False)
                adata = adata[idx, :].copy()
    
    # Save raw
    adata.raw = adata.copy()
    
    # QC
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    print(f"    After QC: {adata.n_obs} spots x {adata.n_vars} genes")
    
    # Normalize
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    
    # Calculate proliferation score
    print("  Creating proliferation target")
    prolif_genes_present = [g for g in PROLIFERATION_GENES if g in adata.var_names]
    print(f"    Found {len(prolif_genes_present)}/{len(PROLIFERATION_GENES)} proliferation genes")
    
    if len(prolif_genes_present) > 0:
        prolif_expr = adata[:, prolif_genes_present].X
        if hasattr(prolif_expr, 'toarray'):
            prolif_expr = prolif_expr.toarray()
        prolif_score = np.mean(prolif_expr, axis=1)
        threshold = np.percentile(prolif_score, 75)
        adata.obs['target_proliferation_binary'] = (prolif_score > threshold).astype(int)
        
        n_high = adata.obs['target_proliferation_binary'].sum()
        n_low = len(adata.obs) - n_high
        print(f"    High proliferation: {n_high} ({100*n_high/len(adata.obs):.1f}%)")
        print(f"    Low proliferation: {n_low} ({100*n_low/len(adata.obs):.1f}%)")
    else:
        print("    ERROR: No proliferation genes found!")
        return None
    
    # Check for pathway scores
    pathway_cols = [col for col in adata.obs.columns 
                    if any(col.startswith(p) for p in ['Reactome_', 'GO_', 'KEGG_', 'MSigDB_'])]
    
    if len(pathway_cols) > 0:
        print(f"  Found {len(pathway_cols)} pathway scores in adata.obs")
        
        # Export pathway scores
        print("  Exporting pathway scores")
        pathway_scores = adata.obs[pathway_cols].copy()
        pathway_file = f'Patient_{patient_id}_pathway_scores.csv'
        pathway_scores.to_csv(pathway_file)
        print(f"    Saved: {pathway_file}")
        print(f"    Shape: {pathway_scores.shape}")
    else:
        print("  WARNING: No pathway scores found")
        print("  Running pathway discovery now")
        
        # Run simplified pathway discovery
        adata = run_pathway_discovery(adata, patient_id)
        
        # Export pathway scores
        pathway_cols = [col for col in adata.obs.columns 
                        if any(col.startswith(p) for p in ['Reactome_', 'GO_', 'KEGG_', 'MSigDB_'])]
        
        if len(pathway_cols) > 0:
            pathway_scores = adata.obs[pathway_cols].copy()
            pathway_file = f'Patient_{patient_id}_pathway_scores.csv'
            pathway_scores.to_csv(pathway_file)
            print(f"    Saved: {pathway_file}")
            print(f"    Shape: {pathway_scores.shape}")
    
    # Export target
    print("  Exporting target...")
    target_df = pd.DataFrame({
        'target': adata.obs['target_proliferation_binary']
    }, index=adata.obs_names)
    target_file = f'Patient_{patient_id}_target.csv'
    target_df.to_csv(target_file)
    print(f"    Saved: {target_file}")
    print(f"    Shape: {target_df.shape}")
    
    return adata


def run_pathway_discovery(adata, patient_id):
    print("    Running pathway discovery")
    
    import gseapy as gp
    
    # Load pathway databases
    pathway_dbs = ['MSigDB_Hallmark_2020', 'KEGG_2021_Human', 'Reactome_2022']
    
    pathway_collections = []
    
    for db_name in pathway_dbs:
        try:
            pathway_db = gp.get_library(name=db_name, organism='Human')
            if pathway_db and len(pathway_db) > 0:
                pathway_collections.append((db_name.split('_')[0], pathway_db))
                print(f"      Loaded {db_name}: {len(pathway_db)} pathways")
        except:
            print(f"      Skipped {db_name}")
            continue
    
    if len(pathway_collections) == 0:
        print("      ERROR: No pathways loaded")
        return adata
    
    # Score pathways
    print("    Scoring pathways")
    pathway_scores_dict = {}
    
    for collection_name, pathway_dict in pathway_collections:
        for pathway_name, gene_list in pathway_dict.items():
            genes_present = [g for g in gene_list if g in adata.var_names]
            
            if len(genes_present) >= 5:
                pathway_expr = adata[:, genes_present].X
                if hasattr(pathway_expr, 'toarray'):
                    pathway_expr = pathway_expr.toarray()
                
                pathway_score = np.mean(pathway_expr, axis=1)
                col_name = f'{collection_name}_{pathway_name}'
                pathway_scores_dict[col_name] = pathway_score
    
    # Add to adata.obs
    pathway_scores_df = pd.DataFrame(pathway_scores_dict, index=adata.obs_names)
    adata.obs = pd.concat([adata.obs, pathway_scores_df], axis=1)
    
    print(f"      Scored {len(pathway_scores_dict)} pathways")
    
    return adata


# H5 file paths for all patients
h5_files = {
    2: "Patient_2/Visium_FFPE_Human_Breast_Cancer_filtered_feature_bc_matrix.h5",
    3: "Patient_3/Parent_Visium_Human_BreastCancer_filtered_feature_bc_matrix.h5",
    4: "Patient_4/CytAssist_Fresh_Frozen_Human_Breast_Cancer_filtered_feature_bc_matrix.h5",
    5: "Patient_5/CytAssist_FFPE_Protein_Expression_Human_Breast_Cancer_filtered_feature_bc_matrix.h5",
    6: "Patient_6/CytAssist_Fresh_Frozen_Human_Breast_Cancer_filtered_feature_bc_matrix.h5",
    7: "Patient_7/Visium_HD_16um_filtered_feature_bc_matrix.h5",
}

# Track which patients are Visium HD
visium_hd_patients = [7]

for patient_id in PATIENTS:
    h5_path = h5_files[patient_id]
    is_visium_hd = patient_id in visium_hd_patients
    
    if Path(h5_path).exists():
        adata = process_patient(patient_id, h5_path, is_visium_hd=is_visium_hd)
    else:
        print(f"\nPatient {patient_id}: H5 file not found at {h5_path}")

# Summary
print("\n\nSUMMARY")
print(f"{'Patient':<15} | {'Pathway Scores':<15} | {'Target':<10}")
print("-" * 45)

for patient_id in PATIENTS:
    pathway_file = f'Patient_{patient_id}_pathway_scores.csv'
    target_file = f'Patient_{patient_id}_target.csv'
    
    patient_label = f"Patient_{patient_id}"
    if patient_id in visium_hd_patients:
        patient_label += " (HD)"
    
    pathway_status = "Done" if Path(pathway_file).exists() else "Missing"
    target_status = "Done" if Path(target_file).exists() else "Missing"
    
    print(f"{patient_label:<15} | {pathway_status:<15} | {target_status:<10}")

print("\nExport complete")

In [ ]:
# RUN FLUX INTERPRETATION FOR ALL PATIENTS

from scipy.stats import spearmanr, mannwhitneyu
from statsmodels.stats.multitest import multipletests

print("RUNNING FLUX INTERPRETATION FOR ALL PATIENTS")

PATIENTS = [2, 3, 4, 5, 6, 7]
BASE_OUTPUT_DIR = './flux_interpretation_results'
VISIUM_HD_PATIENTS = [7]

def run_flux_interpretation(patient_id):
    
    patient_type = " (Visium HD)" if patient_id in VISIUM_HD_PATIENTS else ""
    print(f"\nPatient {patient_id}{patient_type}")
    
    # Setup
    output_dir = f'{BASE_OUTPUT_DIR}/Patient_{patient_id}'
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Load data
    print("  Loading data")
    try:
        pathway_scores = pd.read_csv(f'Patient_{patient_id}_pathway_scores.csv', index_col=0)
        target_df = pd.read_csv(f'Patient_{patient_id}_target.csv', index_col=0)
        flux_df = pd.read_csv(f'Patient_{patient_id}_flux_RAW_modules.csv', index_col=0)
        
        print(f"    Pathways: {pathway_scores.shape}")
        print(f"    Target: {target_df.shape}")
        print(f"    Flux: {flux_df.shape}")
    except FileNotFoundError as e:
        print(f"    ERROR: Missing file - {e}")
        return None
    
    # Match spots
    common_spots = (pathway_scores.index
                   .intersection(target_df.index)
                   .intersection(flux_df.index))
    
    print(f"    Common spots: {len(common_spots)}")
    
    if len(common_spots) == 0:
        print("    ERROR: No matching spots")
        return None
    
    pathway_scores_matched = pathway_scores.loc[common_spots]
    target_matched = target_df.loc[common_spots]
    flux_matched = flux_df.loc[common_spots]
    
    # Analyze flux associations
    print("  Analyzing flux associations")
    flux_associations = []
    
    for flux_col in flux_matched.columns:
        flux_values = flux_matched[flux_col].values
        target_values = target_matched['target'].values
        
        if np.var(flux_values) < 1e-10:
            continue
        
        corr, pval = spearmanr(flux_values, target_values)
        
        high_flux = flux_values[target_values == 1]
        low_flux = flux_values[target_values == 0]
        
        if len(high_flux) > 0 and len(low_flux) > 0:
            _, pval_mw = mannwhitneyu(high_flux, low_flux, alternative='two-sided')
            mean_high = np.mean(high_flux)
            mean_low = np.mean(low_flux)
            fold_change = mean_high / mean_low if mean_low > 0 else np.inf
        else:
            pval_mw = 1.0
            mean_high = 0
            mean_low = 0
            fold_change = 1.0
        
        flux_associations.append({
            'flux_feature': flux_col,
            'correlation': corr,
            'pval_corr': pval,
            'pval_mw': pval_mw,
            'mean_high': mean_high,
            'mean_low': mean_low,
            'fold_change': fold_change
        })
    
    flux_assoc_df = pd.DataFrame(flux_associations)
    
    if len(flux_assoc_df) > 0:
        _, qvals, _, _ = multipletests(flux_assoc_df['pval_corr'], method='fdr_bh')
        flux_assoc_df['qval'] = qvals
        flux_assoc_df['significant'] = qvals < 0.05
        flux_assoc_df['abs_corr'] = np.abs(flux_assoc_df['correlation'])
        flux_assoc_df = flux_assoc_df.sort_values('abs_corr', ascending=False)
        
        flux_assoc_df.to_csv(f'{output_dir}/flux_associations.csv', index=False)
        
        print(f"    Tested: {len(flux_assoc_df)} modules")
        print(f"    Significant: {flux_assoc_df['significant'].sum()} (FDR < 0.05)")
        
        if flux_assoc_df['significant'].sum() > 0:
            top_module = flux_assoc_df.iloc[0]
            print(f"    Top module: {top_module['flux_feature']}")
            print(f"      r = {top_module['correlation']:.3f}")
            print(f"      q = {top_module['qval']:.2e}")
        
        return flux_assoc_df
    else:
        print("    ERROR: No modules to analyze")
        return None

# Run for all patients
print("\nProcessing all patients")

all_results = {}

for patient_id in PATIENTS:
    result = run_flux_interpretation(patient_id)
    if result is not None:
        all_results[patient_id] = result

# Summary
print("\n\nSUMMARY")
print(f"{'Patient':<15} | {'Modules':<10} | {'Significant':<12} | {'Top Module':<20} | {'Corr':<8}")
print("-" * 75)

for patient_id in PATIENTS:
    patient_label = f"Patient_{patient_id}"
    if patient_id in VISIUM_HD_PATIENTS:
        patient_label += " (HD)"
    
    if patient_id in all_results:
        df = all_results[patient_id]
        n_modules = len(df)
        n_sig = df['significant'].sum()
        top_module = df.iloc[0]['flux_feature'] if len(df) > 0 else "N/A"
        top_corr = df.iloc[0]['correlation'] if len(df) > 0 else 0
        
        # Truncate module name if too long
        if len(top_module) > 18:
            top_module = top_module[:15] + "..."
        
        print(f"{patient_label:<15} | {n_modules:<10} | {n_sig:<12} | {top_module:<20} | {top_corr:<8.3f}")
    else:
        print(f"{patient_label:<15} | {'Failed':<10} | {'-':<12} | {'-':<20} | {'-':<8}")

print(f"\nCOMPLETED: {len(all_results)}/{len(PATIENTS)} patients")
print(f"  Standard Visium: {len([p for p in all_results.keys() if p not in VISIUM_HD_PATIENTS])}")
print(f"  Visium HD: {len([p for p in all_results.keys() if p in VISIUM_HD_PATIENTS])}")

In [ ]:
# CROSS-PATIENT COMPARISON ANALYSIS

print("CROSS-PATIENT COMPARISON ANALYSIS")

# Include all patients
all_patients = [1, 2, 3, 4, 5, 6, 7]
BASE_OUTPUT_DIR = './flux_interpretation_results'
VISIUM_HD_PATIENTS = [7]

# Load results for all patients
print("\nLoading results")
all_flux_results = {}

for patient_id in all_patients:
    flux_file = f'{BASE_OUTPUT_DIR}/Patient_{patient_id}/flux_associations.csv'
    patient_type = " (Visium HD)" if patient_id in VISIUM_HD_PATIENTS else ""
    
    if Path(flux_file).exists():
        flux_df = pd.read_csv(flux_file)
        all_flux_results[patient_id] = flux_df
        print(f"  Patient {patient_id}{patient_type}: {len(flux_df)} modules, {flux_df['significant'].sum()} significant")
    else:
        print(f"  Patient {patient_id}{patient_type}: File not found")

if len(all_flux_results) < 2:
    print("\nERROR: Need at least 2 patients")
else:
    print(f"\nComparing {len(all_flux_results)} patients")
    print(f"  Standard Visium: {len([p for p in all_flux_results.keys() if p not in VISIUM_HD_PATIENTS])}")
    print(f"  Visium HD: {len([p for p in all_flux_results.keys() if p in VISIUM_HD_PATIENTS])}")
    
    # Get all modules
    all_modules = set()
    for patient_id, flux_df in all_flux_results.items():
        all_modules.update(flux_df['flux_feature'].tolist())
    
    all_modules = sorted(list(all_modules))
    print(f"Total unique modules: {len(all_modules)}")
    
    # Create comparison matrix
    print("\nBuilding comparison matrix")
    comparison_data = []
    
    for module in all_modules:
        module_data = {'Module': module}
        
        for patient_id in all_flux_results.keys():
            flux_df = all_flux_results[patient_id]
            module_row = flux_df[flux_df['flux_feature'] == module]
            
            if len(module_row) > 0:
                module_data[f'P{patient_id}_corr'] = module_row.iloc[0]['correlation']
                module_data[f'P{patient_id}_qval'] = module_row.iloc[0]['qval']
                module_data[f'P{patient_id}_sig'] = module_row.iloc[0]['qval'] < 0.05
            else:
                module_data[f'P{patient_id}_corr'] = np.nan
                module_data[f'P{patient_id}_qval'] = np.nan
                module_data[f'P{patient_id}_sig'] = False
        
        # Calculate statistics
        corrs = [module_data[f'P{p}_corr'] for p in all_flux_results.keys() 
                if not np.isnan(module_data.get(f'P{p}_corr', np.nan))]
        
        if len(corrs) > 0:
            module_data['mean_corr'] = np.mean(corrs)
            module_data['std_corr'] = np.std(corrs)
            module_data['n_patients'] = len(corrs)
            module_data['n_significant'] = sum([module_data[f'P{p}_sig'] for p in all_flux_results.keys()])
            module_data['conserved'] = module_data['n_significant'] >= len(all_flux_results) * 0.5
        else:
            module_data['mean_corr'] = np.nan
            module_data['std_corr'] = np.nan
            module_data['n_patients'] = 0
            module_data['n_significant'] = 0
            module_data['conserved'] = False
        
        comparison_data.append(module_data)
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('mean_corr', key=abs, ascending=False)
    
    # Save
    output_dir = './multi_patient_results'
    Path(output_dir).mkdir(exist_ok=True)
    comparison_df.to_csv(f'{output_dir}/cross_patient_comparison.csv', index=False)
    print(f"\nSaved: {output_dir}/cross_patient_comparison.csv")
    
    # Conserved modules
    print("\nCONSERVED METABOLIC VULNERABILITIES")
    print("(Significant in 50 percent or more of patients)")
    
    conserved = comparison_df[comparison_df['conserved'] == True].copy()
    print(f"\nFound {len(conserved)} conserved modules")
    
    print(f"\n{'Rank':<6} {'Module':<10} {'Mean r':<10} {'Std':<10} {'Patients':<12} {'Direction':<12}")
    
    for i, (_, row) in enumerate(conserved.head(20).iterrows(), 1):
        direction = "UP" if row['mean_corr'] > 0 else "DOWN"
        print(f"{i:<6} {row['Module']:<10} {row['mean_corr']:>9.3f} {row['std_corr']:>9.3f} "
              f"{row['n_significant']}/{row['n_patients']:<9} {direction:<12}")
    
    # Universal targets
    print("\nUNIVERSAL TARGETS (All patients)")
    
    universal = conserved[conserved['n_significant'] == len(all_flux_results)]
    
    if len(universal) > 0:
        print(f"\nFound {len(universal)} universal modules")
        for i, (_, row) in enumerate(universal.head(10).iterrows(), 1):
            direction = "UP" if row['mean_corr'] > 0 else "DOWN"
            print(f"{i}. {row['Module']}: r={row['mean_corr']:.3f} ({direction})")
    else:
        print(f"\nNo modules significant in all {len(all_flux_results)} patients")
        print("Showing modules significant in most patients:")
        
        most_common = conserved.head(10)
        for i, (_, row) in enumerate(most_common.iterrows(), 1):
            direction = "UP" if row['mean_corr'] > 0 else "DOWN"
            print(f"{i}. {row['Module']}: r={row['mean_corr']:.3f} ({row['n_significant']}/{row['n_patients']} patients, {direction})")
    
    # Check cross-platform consistency (Standard Visium vs Visium HD)
    print("\nCROSS-PLATFORM CONSISTENCY")
    print("(Comparing Standard Visium vs Visium HD)")
    
    standard_patients = [p for p in all_flux_results.keys() if p not in VISIUM_HD_PATIENTS]
    hd_patients = [p for p in all_flux_results.keys() if p in VISIUM_HD_PATIENTS]
    
    if len(hd_patients) > 0 and len(standard_patients) > 0:
        # Find modules significant in both platforms
        cross_platform_consistent = []
        
        for _, row in comparison_df.iterrows():
            # Check if significant in at least one standard Visium
            sig_standard = any(row.get(f'P{p}_sig', False) for p in standard_patients)
            # Check if significant in Visium HD
            sig_hd = any(row.get(f'P{p}_sig', False) for p in hd_patients)
            
            if sig_standard and sig_hd:
                # Check direction consistency
                standard_corrs = [row[f'P{p}_corr'] for p in standard_patients 
                                 if not np.isnan(row.get(f'P{p}_corr', np.nan))]
                hd_corrs = [row[f'P{p}_corr'] for p in hd_patients 
                           if not np.isnan(row.get(f'P{p}_corr', np.nan))]
                
                if len(standard_corrs) > 0 and len(hd_corrs) > 0:
                    standard_mean = np.mean(standard_corrs)
                    hd_mean = np.mean(hd_corrs)
                    
                    # Same direction?
                    same_direction = (standard_mean > 0) == (hd_mean > 0)
                    
                    cross_platform_consistent.append({
                        'Module': row['Module'],
                        'Standard_mean_corr': standard_mean,
                        'HD_mean_corr': hd_mean,
                        'Same_direction': same_direction,
                        'Overall_mean': row['mean_corr']
                    })
        
        if len(cross_platform_consistent) > 0:
            cross_df = pd.DataFrame(cross_platform_consistent)
            cross_df = cross_df.sort_values('Overall_mean', key=abs, ascending=False)
            
            print(f"\nModules significant in both platforms: {len(cross_df)}")
            print(f"Consistent direction: {cross_df['Same_direction'].sum()}")
            
            print(f"\n{'Module':<10} {'Standard r':<12} {'HD r':<10} {'Consistent':<12}")
            print("-" * 50)
            for _, row in cross_df.head(10).iterrows():
                consistent = "Yes" if row['Same_direction'] else "No"
                print(f"{row['Module']:<10} {row['Standard_mean_corr']:>11.3f} {row['HD_mean_corr']:>9.3f} {consistent:<12}")
            
            # Save cross-platform comparison
            cross_df.to_csv(f'{output_dir}/cross_platform_comparison.csv', index=False)
            print(f"\nSaved: {output_dir}/cross_platform_comparison.csv")
        else:
            print("\nNo modules significant in both platforms")
    else:
        print("\nCannot compare - need both Standard Visium and Visium HD patients")
    
    # Patient-specific
    print("\nPATIENT-SPECIFIC VULNERABILITIES")
    
    patient_specific = comparison_df[comparison_df['n_significant'] == 1]
    print(f"\nModules significant in only 1 patient: {len(patient_specific)}")
    
    for patient_id in all_flux_results.keys():
        patient_type = " (HD)" if patient_id in VISIUM_HD_PATIENTS else ""
        patient_unique = patient_specific[patient_specific[f'P{patient_id}_sig'] == True]
        if len(patient_unique) > 0:
            print(f"\nPatient {patient_id}{patient_type} unique: {len(patient_unique)} modules")
            for i, (_, row) in enumerate(patient_unique.head(3).iterrows(), 1):
                print(f"  {i}. {row['Module']}: r={row[f'P{patient_id}_corr']:.3f}")
    
    # Summary statistics
    print("\nSUMMARY STATISTICS")
    
    print(f"\nTotal patients analyzed: {len(all_flux_results)}")
    print(f"  Standard Visium: {len(standard_patients)}")
    print(f"  Visium HD: {len(hd_patients)}")
    print(f"Total unique modules: {len(all_modules)}")
    print(f"Conserved modules (50 percent+ patients): {len(conserved)}")
    print(f"Patient-specific modules: {len(patient_specific)}")
    
    if len(universal) > 0:
        print(f"Universal modules (all patients): {len(universal)}")

In [ ]:
# BIOLOGICAL INTERPRETATION OF CONSERVED MODULES


print("BIOLOGICAL INTERPRETATION OF UNIVERSAL TARGETS")

# Configuration
ALL_PATIENTS = [1, 2, 3, 4, 5, 6, 7]
VISIUM_HD_PATIENTS = [7]
OUTPUT_DIR = './multi_patient_results'

# Load module annotations from Patient 1
module_gene_file = './scFEA/data/module_gene_m168.csv'
module_genes = pd.read_csv(module_gene_file)

# Load cross-patient comparison results (dynamically)
comparison_file = f'{OUTPUT_DIR}/cross_patient_comparison.csv'

if Path(comparison_file).exists():
    comparison_df = pd.read_csv(comparison_file)
    print(f"Loaded cross-patient comparison: {len(comparison_df)} modules")
    
    # Get number of patients analyzed
    patient_cols = [col for col in comparison_df.columns if col.endswith('_sig')]
    n_patients = len(patient_cols)
    print(f"Patients in analysis: {n_patients}")
    print(f"  Standard Visium: {n_patients - len(VISIUM_HD_PATIENTS)}")
    print(f"  Visium HD: {len(VISIUM_HD_PATIENTS)}")
    
    # Find universal modules (significant in all patients)
    universal_df = comparison_df[comparison_df['n_significant'] == n_patients].copy()
    universal_df = universal_df.sort_values('mean_corr', key=abs, ascending=False)
    
    # Build universal_modules dictionary dynamically
    universal_modules = {}
    for _, row in universal_df.head(15).iterrows():
        module = row['Module']
        direction = 'UP' if row['mean_corr'] > 0 else 'DOWN'
        universal_modules[module] = {
            'r': row['mean_corr'],
            'dir': direction,
            'patients': f"{int(row['n_significant'])}/{int(row['n_patients'])}"
        }
    
    print(f"\nFound {len(universal_modules)} universal modules")
    
else:
    print(f"WARNING: {comparison_file} not found")
    print("Using fallback hardcoded values (may be outdated)")
    
    # Fallback hardcoded values (update these after running cross-patient analysis)
    universal_modules = {
        'M_148': {'r': 0.341, 'dir': 'UP', 'patients': '7/7'},
        'M_155': {'r': 0.326, 'dir': 'UP', 'patients': '7/7'},
        'M_120': {'r': 0.303, 'dir': 'UP', 'patients': '7/7'},
        'M_140': {'r': 0.264, 'dir': 'UP', 'patients': '7/7'},
        'M_30': {'r': 0.257, 'dir': 'UP', 'patients': '7/7'},
        'M_121': {'r': -0.252, 'dir': 'DOWN', 'patients': '7/7'},
        'M_130': {'r': -0.244, 'dir': 'DOWN', 'patients': '7/7'},
        'M_149': {'r': 0.240, 'dir': 'UP', 'patients': '7/7'},
        'M_152': {'r': -0.210, 'dir': 'DOWN', 'patients': '7/7'},
        'M_42': {'r': -0.201, 'dir': 'DOWN', 'patients': '7/7'},
    }

# Display results
print(f"\nTop {len(universal_modules)} Universal Metabolic Vulnerabilities:")
print(f"(Significant across ALL {n_patients if 'n_patients' in dir() else 7} patients, including Visium HD)")

for rank, (module, stats) in enumerate(universal_modules.items(), 1):
    # Extract genes for this module
    module_num = int(module.split('_')[1])
    module_row = module_genes[module_genes['1'] == module]
    
    if len(module_row) > 0:
        gene_columns = [col for col in module_genes.columns if col.startswith('A')]
        genes = []
        for col in gene_columns:
            gene = module_row[col].values[0]
            if pd.notna(gene):
                genes.append(gene)
        
        print(f"\n{rank}. {module} ({stats['dir']}, r={stats['r']:.3f}, {stats['patients']} patients)")
        print(f"   Genes: {', '.join(genes[:5])}")
        if len(genes) > 5:
            print(f"   ... and {len(genes)-5} more")
    else:
        print(f"\n{rank}. {module} ({stats['dir']}, r={stats['r']:.3f}, {stats['patients']} patients)")
        print(f"   Genes: (not found in module annotations)")

# Additional analysis: Cross-platform validation
print("\n\nCROSS-PLATFORM VALIDATION")
print("(Modules significant in both Standard Visium AND Visium HD)")

cross_platform_file = f'{OUTPUT_DIR}/cross_platform_comparison.csv'
if Path(cross_platform_file).exists():
    cross_df = pd.read_csv(cross_platform_file)
    consistent_df = cross_df[cross_df['Same_direction'] == True].copy()
    consistent_df = consistent_df.sort_values('Overall_mean', key=abs, ascending=False)
    
    print(f"\nModules validated across both platforms: {len(consistent_df)}")
    
    print(f"\n{'Rank':<6} {'Module':<10} {'Standard r':<12} {'HD r':<10} {'Direction':<10}")
    print("-" * 55)
    
    for rank, (_, row) in enumerate(consistent_df.head(10).iterrows(), 1):
        direction = 'UP' if row['Overall_mean'] > 0 else 'DOWN'
        print(f"{rank:<6} {row['Module']:<10} {row['Standard_mean_corr']:>11.3f} {row['HD_mean_corr']:>9.3f} {direction:<10}")
    
    # Highlight modules that are both universal AND cross-platform validated
    print("\n\nHIGHEST CONFIDENCE TARGETS")
    print("(Universal across all patients AND validated in Visium HD)")
    
    universal_set = set(universal_modules.keys())
    crossplatform_set = set(consistent_df['Module'].tolist())
    highest_confidence = universal_set.intersection(crossplatform_set)
    
    if len(highest_confidence) > 0:
        print(f"\nFound {len(highest_confidence)} highest-confidence modules:")
        
        for module in highest_confidence:
            stats = universal_modules[module]
            cross_row = consistent_df[consistent_df['Module'] == module].iloc[0]
            
            # Get genes
            module_row = module_genes[module_genes['1'] == module]
            genes = []
            if len(module_row) > 0:
                gene_columns = [col for col in module_genes.columns if col.startswith('A')]
                for col in gene_columns:
                    gene = module_row[col].values[0]
                    if pd.notna(gene):
                        genes.append(gene)
            
            print(f"\n  {module} ({stats['dir']})")
            print(f"    Mean correlation: r={stats['r']:.3f}")
            print(f"    Standard Visium: r={cross_row['Standard_mean_corr']:.3f}")
            print(f"    Visium HD: r={cross_row['HD_mean_corr']:.3f}")
            print(f"    Genes: {', '.join(genes[:5])}")
    else:
        print("\nNo modules meet both criteria")
else:
    print(f"\nCross-platform file not found: {cross_platform_file}")
    print("Run cross-patient comparison first")

# Summary
print("\n\nSUMMARY")
print(f"Total patients: {len(ALL_PATIENTS)}")
print(f"  Standard Visium: {len([p for p in ALL_PATIENTS if p not in VISIUM_HD_PATIENTS])}")
print(f"  Visium HD: {len(VISIUM_HD_PATIENTS)} (Patient 7)")
print(f"Universal modules: {len(universal_modules)}")
if 'highest_confidence' in dir() and len(highest_confidence) > 0:
    print(f"Highest confidence targets: {len(highest_confidence)}")

In [ ]:
# DETAILED BIOLOGICAL INTERPRETATION OF UNIVERSAL TARGETS

print("\nDETAILED PATHWAY ANALYSIS OF UNIVERSAL TARGETS")

universal_pathways = {
    'M_148': {
        'genes': ['APRT', 'GMPS', 'HPRT1', 'NT5C', 'NT5C1A'],
        'pathway': 'Purine Salvage & Biosynthesis',
        'function': 'Recycles purines and synthesizes GMP from IMP',
        'biology': 'Essential for DNA/RNA synthesis in proliferating cells',
        'drugs': ['Methotrexate', '6-Mercaptopurine', 'Azathioprine', 'Mycophenolate'],
        'status': 'FDA approved',
        'rationale': 'Block purine metabolism, starve rapidly dividing cancer cells'
    },
    'M_155': {
        'genes': ['CTPS1', 'CTPS2', 'ENTPD1', 'ENTPD3'],
        'pathway': 'Pyrimidine Biosynthesis',
        'function': 'CTP synthase converts UTP to CTP for DNA/RNA',
        'biology': 'Required for nucleotide pools in S-phase',
        'drugs': ['5-Fluorouracil', 'Gemcitabine', 'Capecitabine'],
        'status': 'FDA approved',
        'rationale': 'Standard chemotherapy targets pyrimidine metabolism'
    },
    'M_120': {
        'genes': ['STT3A', 'STT3B'],
        'pathway': 'N-Glycosylation (ER)',
        'function': 'Oligosaccharyltransferase - adds glycans to proteins',
        'biology': 'Protein folding, trafficking, cell surface receptor function',
        'drugs': ['Tunicamycin', 'NGI-1 (experimental)'],
        'status': 'Experimental',
        'rationale': 'Disrupt protein glycosylation, impair metastasis and growth factor signaling'
    },
    'M_140': {
        'genes': ['AK1', 'AK2', 'AK4', 'AK5', 'AK7'],
        'pathway': 'Adenylate Kinase',
        'function': 'ATP + AMP ↔ 2 ADP (energy homeostasis)',
        'biology': 'Maintains cellular energy balance and nucleotide pools',
        'drugs': ['Experimental AK inhibitors'],
        'status': 'Preclinical',
        'rationale': 'Disrupt energy metabolism in cancer cells'
    },
    'M_30': {
        'genes': ['AHCY', 'CBS', 'CTH', 'MAT2A'],
        'pathway': 'Methionine/Homocysteine Metabolism',
        'function': 'SAM synthesis and one-carbon metabolism',
        'biology': 'Provides methyl groups for DNA/histone methylation',
        'drugs': ['MAT2A inhibitors (AG-270)', 'Homocysteine modulators'],
        'status': 'Clinical trials',
        'rationale': 'Block methylation, affect epigenetics and nucleotide synthesis'
    },
    'M_121': {
        'genes': ['GANAB', 'MAN1A1', 'MAN1A2', 'MAN1B1'],
        'pathway': 'Mannosidase (Glycan Processing)',
        'function': 'Trims mannose from N-glycans in ER/Golgi',
        'biology': 'DOWNREGULATED - altered glycosylation patterns',
        'drugs': ['Not a drug target (downregulated)'],
        'status': 'Biomarker',
        'rationale': 'Indicates altered glycan maturation, possible immune evasion'
    },
    'M_130': {
        'genes': ['CHPF', 'CHPF2', 'CHSY1', 'CHSY3'],
        'pathway': 'Chondroitin Sulfate Biosynthesis',
        'function': 'Synthesizes glycosaminoglycans for ECM',
        'biology': 'DOWNREGULATED - reduced ECM production',
        'drugs': ['Not a drug target (downregulated)'],
        'status': 'Biomarker',
        'rationale': 'ECM remodeling for invasion/metastasis'
    },
    'M_149': {
        'genes': ['APRT', 'HPRT1', 'NT5C', 'NT5C1A'],
        'pathway': 'Purine Salvage',
        'function': 'Recycles purines from nucleotide breakdown',
        'biology': 'Similar to M_148, energy-efficient purine recycling',
        'drugs': ['Same as M_148'],
        'status': 'FDA approved',
        'rationale': 'Complementary to M_148, block all purine sources'
    },
    'M_152': {
        'genes': ['DPYD', 'DPYS', 'NT5C'],
        'pathway': 'Pyrimidine Degradation',
        'function': 'Breaks down uracil and thymine',
        'biology': 'DOWNREGULATED - reduced pyrimidine catabolism',
        'drugs': ['Not a drug target (downregulated)'],
        'status': 'Biomarker',
        'rationale': 'Conservation of pyrimidines for DNA synthesis'
    },
    'M_42': {
        'genes': ['ALDH1A3', 'ALDH1B1', 'ALDH2', 'ALDH3A1'],
        'pathway': 'Aldehyde Dehydrogenase',
        'function': 'Detoxifies aldehydes, retinoic acid synthesis',
        'biology': 'DOWNREGULATED - reduced detoxification',
        'drugs': ['Disulfiram (ALDH inhibitor)'],
        'status': 'FDA approved (for alcoholism)',
        'rationale': 'ALDH1A3 associated with cancer stem cells'
    }
}

print("\nRANK  MODULE  PATHWAY                          DIRECTION  DRUGGABILITY")

for i, (module, info) in enumerate(universal_pathways.items(), 1):
    direction = "UP" if "DOWNREGULATED" not in info['biology'] else "DOWN"
    druggable = "HIGH" if info['status'] in ['FDA approved', 'Clinical trials'] else "MEDIUM" if info['status'] == 'Preclinical' else "LOW"
    
    print(f"{i:<6} {module:<8} {info['pathway']:<32} {direction:<11} {druggable}")

In [ ]:
# METABOLIC PATHWAY VISUALISATION 
import json


print("METABOLIC PATHWAY VISUALISATION")

# Create output directories
os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/tables', exist_ok=True)

# STEP 1: Prepare pathway fold change data

print("\n[Step 1] Preparing pathway data\n")

# Find pathway columns
pathway_cols = [col for col in adata.obs.columns 
                if any(col.startswith(p) for p in ['Hallmark_', 'KEGG_', 'Reactome_', 'BioCarta_'])]

print(f"Found {len(pathway_cols)} pathway scores")

# Calculate fold change (high prolif / low prolif)
high_mask = adata.obs['target_proliferation_binary'] == 1
low_mask = adata.obs['target_proliferation_binary'] == 0

pathway_fc = {}
for col in pathway_cols:
    high_mean = float(adata.obs.loc[high_mask, col].mean())
    low_mean = float(adata.obs.loc[low_mask, col].mean())
    if low_mean > 0:
        fc = np.log2(high_mean / low_mean)
    else:
        fc = 0.0
    pathway_fc[col] = float(fc)  # Convert to Python float

# Sort by absolute fold change
pathway_fc_sorted = dict(sorted(pathway_fc.items(), key=lambda x: abs(x[1]), reverse=True))

print("\nTop 10 pathways by fold change (high vs low proliferation):")
for i, (pathway, fc) in enumerate(list(pathway_fc_sorted.items())[:10], 1):
    direction = "UP" if fc > 0 else "DOWN"
    clean_name = pathway.split('_', 1)[1] if '_' in pathway else pathway
    print(f"  {i:2d}. {clean_name[:45]:<45s} {fc:>+6.2f} ({direction})")

# STEP 2: Map genes to reactions

print("\n[Step 2] Mapping genes to reactions\n")

GENE_TO_REACTION = {
    # Glycolysis
    'HK1': 'HEX1', 'HK2': 'HEX1',
    'GPI': 'PGI',
    'PFKL': 'PFK', 'PFKM': 'PFK', 'PFKP': 'PFK',
    'ALDOA': 'FBA', 'ALDOB': 'FBA', 'ALDOC': 'FBA',
    'TPI1': 'TPI',
    'GAPDH': 'GAPD',
    'PGK1': 'PGK',
    'PGAM1': 'PGM',
    'ENO1': 'ENO', 'ENO2': 'ENO',
    'PKM': 'PYK',
    'LDHA': 'LDH_L', 'LDHB': 'LDH_L',
    # TCA Cycle
    'CS': 'CS',
    'ACO1': 'ACONT', 'ACO2': 'ACONT',
    'IDH1': 'ICDHy', 'IDH2': 'ICDHy',
    'OGDH': 'AKGD',
    'SUCLA2': 'SUCOAS',
    'SDHA': 'SUCD', 'SDHB': 'SUCD',
    'FH': 'FUM',
    'MDH1': 'MDH', 'MDH2': 'MDH',
    # Pentose Phosphate
    'G6PD': 'G6PDH',
    'PGD': 'PGD',
    'TKT': 'TKT',
    'TALDO1': 'TALA',
}

reaction_data = {}

for gene, reaction in GENE_TO_REACTION.items():
    if gene in adata.var_names:
        gene_expr = adata[:, gene].X
        if hasattr(gene_expr, 'toarray'):
            gene_expr = gene_expr.toarray()
        
        high_mean = float(gene_expr[high_mask].mean())
        low_mean = float(gene_expr[low_mask].mean())
        
        if low_mean > 0:
            fc = float(np.log2(high_mean / low_mean))
        else:
            fc = 0.0
        
        if reaction in reaction_data:
            reaction_data[reaction].append(fc)
        else:
            reaction_data[reaction] = [fc]

# Average multiple genes per reaction and convert to Python float
reaction_fc = {rxn: float(np.mean(fcs)) for rxn, fcs in reaction_data.items()}

print(f"Mapped {len(reaction_fc)} reactions")
print("\nReaction fold changes:")
for rxn, fc in sorted(reaction_fc.items(), key=lambda x: abs(x[1]), reverse=True)[:10]:
    print(f"  {rxn:<10s}: {fc:>+.3f}")

# STEP 3: Export data for Escher web interface

print("\n[Step 3] Exporting data for Escher web interface\n")

# Export reaction data as JSON
with open('results/tables/escher_reaction_data.json', 'w') as f:
    json.dump(reaction_fc, f, indent=2)

print("Saved: results/tables/escher_reaction_data.json")
print("\nTo use with Escher web interface:")
print("  1. Go to https://escher.github.io")
print("  2. Map > Load map > e4e5 map: Core metabolism")
print("  3. Data > Load reaction data > Upload the JSON file")

# STEP 4: Try Escher Python visualization

print("\n[Step 4] Creating Escher visualization\n")

try:
    from escher import Builder
    
    # Try different map names (they change between versions)
    map_names_to_try = [
        'e_coli_core.Core metabolism',  # Works in most versions
        'iJO1366.Central metabolism',
    ]
    
    builder = None
    for map_name in map_names_to_try:
        try:
            builder = Builder(
                map_name=map_name,
                reaction_data=reaction_fc,
                reaction_scale=[
                    {'type': 'min', 'color': '#3498db', 'size': 12},
                    {'type': 'value', 'value': 0, 'color': '#ecf0f1', 'size': 8},
                    {'type': 'max', 'color': '#e74c3c', 'size': 12}
                ]
            )
            print(f"Loaded map: {map_name}")
            break
        except:
            continue
    
    if builder:
        output_file = 'results/figures/escher_metabolism.html'
        builder.save_html(output_file)
        print(f"Saved: {output_file}")
    else:
        print("Could not load Escher maps from server.")
        print("Use the web interface instead: https://escher.github.io")
        
except Exception as e:
    print(f"Escher visualization skipped: {e}")
    print("Use the web interface instead: https://escher.github.io")

# STEP 5: Create static pathway visualization (always works)

print("\n[Step 5] Creating static pathway visualization\n")

import matplotlib.pyplot as plt

# Get top 20 pathways
top_n = 20
top_pathways_list = list(pathway_fc_sorted.keys())[:top_n]

fc_values = [pathway_fc[p] for p in top_pathways_list]
pathway_names = []
for p in top_pathways_list:
    name = p.split('_', 1)[1] if '_' in p else p
    name = name[:45] + '...' if len(name) > 45 else name
    pathway_names.append(name)

# Sort by value for better visualization
sorted_idx = np.argsort(fc_values)
fc_values_sorted = [fc_values[i] for i in sorted_idx]
names_sorted = [pathway_names[i] for i in sorted_idx]

# Plot
fig, ax = plt.subplots(figsize=(10, 10))

colors = [COLOR_HIGH if x > 0 else COLOR_LOW for x in fc_values_sorted]
bars = ax.barh(range(len(names_sorted)), fc_values_sorted, color=colors, edgecolor='black', linewidth=0.5)

ax.set_yticks(range(len(names_sorted)))
ax.set_yticklabels(names_sorted, fontsize=9)
ax.set_xlabel('Log2 Fold Change (High / Low Proliferation)', fontsize=11, fontweight='bold')
ax.set_title('Top 20 Pathways by Differential Activity\n(Red = Upregulated in High Proliferation)', fontsize=12, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('results/figures/pathway_foldchange_barplot.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: results/figures/pathway_foldchange_barplot.png")

# STEP 6: Create heatmap of top pathways across clusters

print("\n[Step 6] Creating pathway heatmap by cluster\n")

if 'leiden' in adata.obs.columns:
    import seaborn as sns
    
    # Get top 15 pathways
    top_15 = list(pathway_fc_sorted.keys())[:15]
    
    # Calculate mean pathway score per cluster
    clusters = sorted(adata.obs['leiden'].unique())
    heatmap_data = []
    
    for cluster in clusters:
        cluster_mask = adata.obs['leiden'] == cluster
        row = {'Cluster': f'Cluster {cluster}'}
        for pathway in top_15:
            clean_name = pathway.split('_', 1)[1][:30] if '_' in pathway else pathway[:30]
            row[clean_name] = float(adata.obs.loc[cluster_mask, pathway].mean())
        heatmap_data.append(row)
    
    heatmap_df = pd.DataFrame(heatmap_data)
    heatmap_df = heatmap_df.set_index('Cluster')
    
    # Z-score normalize
    heatmap_zscore = (heatmap_df - heatmap_df.mean()) / heatmap_df.std()
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(heatmap_zscore.T, cmap=CMAP_DIVERGING, center=0, 
                linewidths=0.5, linecolor=COLOR_REFERENCE,
                cbar_kws={'label': 'Z-score'},
                ax=ax)
    ax.set_title('Top 15 Pathway Activity Across Clusters', fontsize=13, fontweight='bold')
    ax.set_xlabel('Cluster', fontsize=11, fontweight='bold')
    ax.set_ylabel('Pathway', fontsize=11, fontweight='bold')
    plt.xticks(rotation=0)
    plt.yticks(rotation=0, fontsize=9)
    
    plt.tight_layout()
    plt.savefig('results/figures/pathway_cluster_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Saved: results/figures/pathway_cluster_heatmap.png")
else:
    print("No leiden clustering found, skipping heatmap")

In [ ]:
# IMPORT METABOLIC TRANSFORMER MODULE

import sys
sys.path.insert(0, '.')

from metabolic_transformer import (
    MetabolicTransformer,
    FeatureTokenizer,
    MultiHeadSelfAttention,
    TransformerBlock,
    EarlyStopping,
    prepare_multimodal_features,
    spatial_cv_transformer,
    analyse_attention,
    compare_models
)

from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score,
    precision_recall_fscore_support
)
from torch.utils.data import DataLoader, TensorDataset
import gc
import os

print("MetabolicTransformer v2.0 loaded")
print("  Architecture: Feature Tokenization Transformer")
print("  Innovation: Expression + Flux concatenation with cross-modal attention")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Device: {DEVICE}")

In [ ]:
print("LOADING MULTI-PATIENT DATA")

COVARIATE_COLS = ['log1p_total_counts', 'pct_counts_mt', 'pct_counts_ribo']
PROLIF_GENES = ['MKI67', 'TOP2A', 'PCNA', 'CCNB1', 'CCNB2', 'CDK1', 'BIRC5']

patient_data = {}

for patient_id in [1, 2, 3, 4, 5, 6, 7]:
    print(f"Patient {patient_id}")
    
    # Load flux data FIRST (skip if missing)
    flux_file = f'./scFEA/results/Patient_{patient_id}/Patient_{patient_id}_flux_corrected.csv'
    if not Path(flux_file).exists():
        print(f"   No flux data")
        continue
    
    flux = pd.read_csv(flux_file, index_col=0).T  # TRANSPOSE: spots × modules
    print(f"  Flux: {flux.shape}")
    
    # Load AnnData 
    if patient_id == 1:
        # Check if current adata is Patient 1 or Patient 7 (Visium HD)
        if 'adata' in dir() and not adata.obs_names[0].startswith('s_016um'):
            patient_adata = adata.copy()
            print(f"  Using existing adata: {patient_adata.n_obs} spots")
        else:
            # Load Patient 1 fresh
            try:
                patient_adata = sc.read_visium(
                    path='Patient_1',
                    count_file='Visium_Human_Breast_Cancer_filtered_feature_bc_matrix.h5',
                    load_images=False
                )
                patient_adata.var_names_make_unique()
                sc.pp.filter_genes(patient_adata, min_cells=3)
                sc.pp.normalize_total(patient_adata, target_sum=1e4)
                sc.pp.log1p(patient_adata)
                print(f"  Loaded fresh: {patient_adata.n_obs} spots")
            except Exception as e:
                print(f"   Could not load Patient 1: {e}")
                continue
    else:
        config = CFG.PATIENT_CONFIG.get(patient_id, {})
        if not config:
            print(f"   No config")
            continue
        
        try:
            patient_adata = sc.read_visium(
                path=config['data_dir'],
                count_file=config['h5_file'],
                load_images=False
            )
            patient_adata.var_names_make_unique()
            sc.pp.filter_genes(patient_adata, min_cells=3)
            sc.pp.normalize_total(patient_adata, target_sum=1e4)
            sc.pp.log1p(patient_adata)
            print(f"  Loaded: {patient_adata.n_obs} spots × {patient_adata.n_vars} genes")
        except Exception as e:
            print(f"   Load error: {e}")
            continue
    
    # Score pathways from GMT files
    gmt_dir = Path("msigdb/")
    if gmt_dir.exists():
        all_gene_sets = {}
        for gmt_file in gmt_dir.glob("*.gmt"):
            db_name = gmt_file.stem.split('.')[-1] if '.' in gmt_file.stem else gmt_file.stem
            with open(gmt_file) as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) >= 3:
                        gs_name = f"{db_name}_{parts[0]}"
                        genes = parts[2:]
                        all_gene_sets[gs_name] = genes
        
        print(f"  Scoring {len(all_gene_sets)} pathways...")
        for gs_name, genes in all_gene_sets.items():
            if gs_name in patient_adata.obs.columns:
                continue  # Already scored
            genes_present = [g for g in genes if g in patient_adata.var_names]
            if len(genes_present) >= 3:
                gene_idx = [list(patient_adata.var_names).index(g) for g in genes_present]
                if hasattr(patient_adata.X, 'toarray'):
                    scores = np.array(patient_adata.X[:, gene_idx].mean(axis=1)).flatten()
                else:
                    scores = patient_adata.X[:, gene_idx].mean(axis=1)
                patient_adata.obs[gs_name] = scores
            else:
                patient_adata.obs[gs_name] = 0.0
    
    #  Create covariates
    if 'total_counts' not in patient_adata.obs.columns:
        sc.pp.calculate_qc_metrics(patient_adata, percent_top=None, log1p=False, inplace=True)
    
    if 'log1p_total_counts' not in patient_adata.obs.columns:
        patient_adata.obs['log1p_total_counts'] = np.log1p(patient_adata.obs['total_counts'])
    
    if 'pct_counts_mt' not in patient_adata.obs.columns:
        patient_adata.var['mt'] = patient_adata.var_names.str.startswith('MT-')
        sc.pp.calculate_qc_metrics(patient_adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    if 'pct_counts_ribo' not in patient_adata.obs.columns:
        patient_adata.var['ribo'] = patient_adata.var_names.str.match('^RP[SL]')
        if patient_adata.var['ribo'].sum() > 0:
            sc.pp.calculate_qc_metrics(patient_adata, qc_vars=['ribo'], percent_top=None, log1p=False, inplace=True)
        else:
            patient_adata.obs['pct_counts_ribo'] = 0.0
    
    #  Create proliferation target 
    if 'target_proliferation_binary' not in patient_adata.obs.columns:
        present = [g for g in PROLIF_GENES if g in patient_adata.var_names]
        if len(present) >= 3:
            sc.tl.score_genes(patient_adata, gene_list=present, score_name='proliferation_score')
            threshold = np.percentile(patient_adata.obs['proliferation_score'], 75)
            patient_adata.obs['target_proliferation_binary'] = (
                patient_adata.obs['proliferation_score'] >= threshold
            ).astype(float)
        else:
            print(f"   Insufficient proliferation genes")
            continue
    
    # Align barcodes with flux 
    common = patient_adata.obs_names.intersection(flux.index)
    if len(common) == 0:
        flux.index = [b + '-1' if '-1' not in b else b for b in flux.index]
        common = patient_adata.obs_names.intersection(flux.index)
    if len(common) == 0:
        flux.index = [b.replace('-1', '') for b in flux.index]
        common = patient_adata.obs_names.intersection(flux.index)
    
    print(f"  Barcode overlap: {len(common)} / {patient_adata.n_obs}")
    
    if len(common) < 100:
        print(f"   Too few matching spots")
        continue
    
    patient_adata = patient_adata[common].copy()
    flux = flux.loc[common]
    
    patient_data[patient_id] = {'adata': patient_adata, 'flux': flux}
    print(f"   Ready: {patient_adata.n_obs} spots")


print(f"PATIENTS LOADED: {list(patient_data.keys())}")

In [ ]:
# SELECT CONSENSUS PROLIFERATION PATHWAYS

print("SELECTING CONSENSUS PATHWAYS")


#  Score pathways per patient 
patient_pathway_corrs = {}

for pid in sorted(patient_data.keys()):
    adata_p = patient_data[pid]['adata']
    y_p = adata_p.obs['target_proliferation_binary'].values
    
    pw_cols = [c for c in adata_p.obs.columns 
               if any(db in c for db in ['KEGG', 'Reactome', 'BioCarta', 'Hallmark', 'GO_', 'GOBP'])]
    
    corrs = {}
    for col in pw_cols:
        vals = adata_p.obs[col].values
        if vals.std() > 0:
            rho, pval = spearmanr(vals, y_p)
            if not np.isnan(rho):
                corrs[col] = {'rho': rho, 'abs_rho': abs(rho), 'pval': pval}
    
    patient_pathway_corrs[pid] = pd.DataFrame(corrs).T
    print(f"  Patient {pid}: {len(corrs)} pathways with variance")

#  Find consensus pathways 
all_pathways = set()
for pid, df in patient_pathway_corrs.items():
    all_pathways.update(df.index)

pathway_consensus = []
for pw in all_pathways:
    n_sig = 0
    rhos = []
    for pid, df in patient_pathway_corrs.items():
        if pw in df.index:
            if df.loc[pw, 'pval'] < 0.05 and df.loc[pw, 'abs_rho'] > 0.05:
                n_sig += 1
                rhos.append(df.loc[pw, 'rho'])
    if rhos:
        pathway_consensus.append({
            'pathway': pw,
            'n_significant': n_sig,
            'mean_rho': np.mean(rhos),
            'mean_abs_rho': np.mean(np.abs(rhos))
        })

consensus_df = pd.DataFrame(pathway_consensus).sort_values(
    ['n_significant', 'mean_abs_rho'], ascending=[False, False]
)

# Select top 20
CONSENSUS_PATHWAYS = consensus_df.head(20)['pathway'].tolist()

print(f"\nTop 20 Consensus Pathways:")
print(f"{'─'*70}")
for _, row in consensus_df.head(20).iterrows():
    direction = "↑" if row['mean_rho'] > 0 else "↓"
    print(f"  {direction} [{int(row['n_significant'])}/{len(patient_data)}] {row['pathway'][:55]} (ρ={row['mean_rho']:.3f})")

# Verify all patients have these columns
for pid in patient_data:
    adata_p = patient_data[pid]['adata']
    for p in CONSENSUS_PATHWAYS:
        if p not in adata_p.obs.columns:
            adata_p.obs[p] = 0.0

In [ ]:
# SELECT CONSISTENT FLUX MODULES

print("SELECTING CONSISTENT FLUX MODULES")

all_flux_vars = {}
for pid in patient_data:
    flux = patient_data[pid]['flux']
    for col in flux.columns:
        if col not in all_flux_vars:
            all_flux_vars[col] = []
        all_flux_vars[col].append(flux[col].var())

# Modules present in ALL patients
module_avg_var = {m: np.mean(v) for m, v in all_flux_vars.items() 
                  if len(v) == len(patient_data)}
sorted_modules = sorted(module_avg_var.items(), key=lambda x: x[1], reverse=True)

N_FLUX_MODULES = 30
CONSISTENT_FLUX_COLS = [m[0] for m in sorted_modules[:N_FLUX_MODULES]]

print(f"  Modules in ALL patients: {len(module_avg_var)}")
print(f"  Selected top {N_FLUX_MODULES} by variance")
print(f"\nTop 10 flux modules:")
for m, v in sorted_modules[:10]:
    print(f"  {m}: variance = {v:.4f}")

In [ ]:
import torch
import numpy as np

def evaluate_fixed(model, loader, criterion, device):
    """Evaluate model with fixed squeeze dimension."""
    model.eval()
    total_loss = 0
    n_batches = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            logits = model(X_batch)
            # FIX: Use squeeze(-1) instead of squeeze() to preserve batch dim
            logits_squeezed = logits.squeeze(-1) if logits.dim() > 1 else logits
            
            loss = criterion(logits_squeezed, y_batch)
            
            probs = torch.sigmoid(logits_squeezed)
            all_preds.append(probs.cpu().numpy())
            all_labels.append(y_batch.cpu().numpy())
            
            total_loss += loss.item()
            n_batches += 1
    
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    
    return total_loss / n_batches, preds, labels

# Monkey-patch the module
import metabolic_transformer
metabolic_transformer.evaluate = evaluate_fixed

print("Patched metabolic_transformer.evaluate() to fix squeeze bug")

In [ ]:
# SPATIAL CROSS-VALIDATION

print("SPATIAL CROSS-VALIDATION")


# Select patient for CV
CV_PATIENT = 2 if 2 in patient_data else list(patient_data.keys())[0]
cv_data = patient_data[CV_PATIENT]
cv_adata = cv_data['adata']
cv_flux = cv_data['flux']

print(f"Patient {CV_PATIENT}: {cv_adata.n_obs} spots")

# Create spatial blocks
if 'spatial' in cv_adata.obsm:
    coords = cv_adata.obsm['spatial']
elif 'x_coord' in cv_adata.obs.columns:
    coords = cv_adata.obs[['x_coord', 'y_coord']].values
else:
    # Parse from barcodes or use index
    coords = np.column_stack([
        np.arange(cv_adata.n_obs),
        np.arange(cv_adata.n_obs)
    ])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
spatial_blocks = kmeans.fit_predict(coords)
print(f"Spatial blocks: {dict(zip(*np.unique(spatial_blocks, return_counts=True)))}")

# Prepare features
X_expr = cv_adata.obs[CONSENSUS_PATHWAYS].values

X_flux_aligned = pd.DataFrame(0.0, index=cv_adata.obs_names, columns=CONSISTENT_FLUX_COLS)
common = cv_adata.obs_names.intersection(cv_flux.index)
for c in CONSISTENT_FLUX_COLS:
    if c in cv_flux.columns:
        X_flux_aligned.loc[common, c] = cv_flux.loc[common, c]
X_flux = X_flux_aligned.values

cov_avail = [c for c in COVARIATE_COLS if c in cv_adata.obs.columns]
X_cov = cv_adata.obs[cov_avail].values

X_combined = np.hstack([X_expr, X_flux, X_cov])
y = cv_adata.obs['target_proliferation_binary'].values

feature_names = CONSENSUS_PATHWAYS + CONSISTENT_FLUX_COLS + cov_avail
feature_types = (['expression'] * len(CONSENSUS_PATHWAYS) + 
                 ['flux'] * len(CONSISTENT_FLUX_COLS) + 
                 ['covariate'] * len(cov_avail))

print(f"\nFeature matrix: {X_combined.shape}")
print(f"  Expression: {len(CONSENSUS_PATHWAYS)} | Flux: {len(CONSISTENT_FLUX_COLS)} | Covariates: {len(cov_avail)}")
print(f"Class distribution: {np.bincount(y.astype(int))}")

# Run spatial CV using the module function
cv_results = spatial_cv_transformer(
    X=X_combined,
    y=y,
    spatial_blocks=spatial_blocks,
    feature_names=feature_names,
    feature_types=feature_types,
    n_splits=5,
    d_token=48,
    n_heads=4,
    n_layers=2,
    dropout=0.3,
    lr=5e-4,
    weight_decay=1e-2,
    epochs=150,
    batch_size=256,
    patience=20,
    device=DEVICE,
    verbose=True
)

# Save
os.makedirs('results/transformer', exist_ok=True)
cv_results['fold_results'].to_csv('results/transformer/spatial_cv_results.csv', index=False)
print(f"\nSaved: results/transformer/spatial_cv_results.csv")

In [ ]:
from scipy.stats import spearmanr, ks_2samp
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

print("DOMAIN SHIFT ANALYSIS: Standard Visium vs Visium HD")


# Separate patients by platform
standard_patients = [p for p in patient_data.keys() if p != 7]
hd_patients = [7] if 7 in patient_data else []

print(f"\nStandard Visium (55µm): Patients {standard_patients}")
print(f"Visium HD (8µm): Patients {hd_patients}")

# ─── Compare feature distributions ───
print(f"\n{'─'*70}")
print("FEATURE DISTRIBUTION COMPARISON")
print(f"{'─'*70}")

# Collect features from both platforms
X_standard, X_hd = [], []
for pid in standard_patients:
    data = patient_data[pid]
    x_e = data['adata'].obs[CONSENSUS_PATHWAYS].values
    fa = pd.DataFrame(0.0, index=data['adata'].obs_names, columns=CONSISTENT_FLUX_COLS)
    common = data['adata'].obs_names.intersection(data['flux'].index)
    for c in CONSISTENT_FLUX_COLS:
        if c in data['flux'].columns:
            fa.loc[common, c] = data['flux'].loc[common, c]
    x_f = fa.values
    X_standard.append(np.hstack([x_e, x_f]))

if 7 in patient_data:
    data = patient_data[7]
    x_e = data['adata'].obs[CONSENSUS_PATHWAYS].values
    fa = pd.DataFrame(0.0, index=data['adata'].obs_names, columns=CONSISTENT_FLUX_COLS)
    common = data['adata'].obs_names.intersection(data['flux'].index)
    for c in CONSISTENT_FLUX_COLS:
        if c in data['flux'].columns:
            fa.loc[common, c] = data['flux'].loc[common, c]
    x_f = fa.values
    X_hd.append(np.hstack([x_e, x_f]))

X_standard = np.vstack(X_standard)
X_hd = np.vstack(X_hd) if X_hd else None

print(f"\nStandard Visium: {X_standard.shape[0]} spots")
print(f"Visium HD: {X_hd.shape[0] if X_hd is not None else 0} spots")

# KS test for distribution shift
if X_hd is not None:
    feature_names_check = CONSENSUS_PATHWAYS + CONSISTENT_FLUX_COLS
    ks_results = []
    for i, fname in enumerate(feature_names_check):
        stat, pval = ks_2samp(X_standard[:, i], X_hd[:, i])
        ks_results.append({'feature': fname, 'ks_stat': stat, 'pval': pval})
    
    ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)
    
    print(f"\nTop 10 features with LARGEST distribution shift (KS test):")
    for _, row in ks_df.head(10).iterrows():
        sig = "***" if row['pval'] < 0.001 else "**" if row['pval'] < 0.01 else "*" if row['pval'] < 0.05 else ""
        print(f"  {row['feature'][:50]}: KS={row['ks_stat']:.3f} {sig}")
    
    print(f"\nFeatures with SMALLEST shift (most transferable):")
    for _, row in ks_df.tail(5).iterrows():
        print(f"  {row['feature'][:50]}: KS={row['ks_stat']:.3f}")

In [ ]:
# PLATFORM-SPECIFIC VALIDATION — STANDARD VISIUM ONLY

print("LOPO VALIDATION — STANDARD VISIUM (Patients 1-6)")

standard_patients = [1, 2, 3, 4, 5, 6]
lopo_standard_results = []

for test_pid in standard_patients:
    train_pids = [p for p in standard_patients if p != test_pid]
    
    print(f"Test: Patient {test_pid} | Train: {train_pids}")
    
    # Build training data
    X_train_parts, y_train_parts = [], []
    for p in train_pids:
        data = patient_data[p]
        x_e = data['adata'].obs[CONSENSUS_PATHWAYS].values
        
        fa = pd.DataFrame(0.0, index=data['adata'].obs_names, columns=CONSISTENT_FLUX_COLS)
        common = data['adata'].obs_names.intersection(data['flux'].index)
        for c in CONSISTENT_FLUX_COLS:
            if c in data['flux'].columns:
                fa.loc[common, c] = data['flux'].loc[common, c]
        x_f = fa.values
        x_c = data['adata'].obs[cov_avail].values
        
        X_train_parts.append(np.hstack([x_e, x_f, x_c]))
        y_train_parts.append(data['adata'].obs['target_proliferation_binary'].values)
    
    X_train = np.vstack(X_train_parts)
    y_train = np.concatenate(y_train_parts)
    
    # Test data
    td = patient_data[test_pid]
    x_e_t = td['adata'].obs[CONSENSUS_PATHWAYS].values
    fa_t = pd.DataFrame(0.0, index=td['adata'].obs_names, columns=CONSISTENT_FLUX_COLS)
    common_t = td['adata'].obs_names.intersection(td['flux'].index)
    for c in CONSISTENT_FLUX_COLS:
        if c in td['flux'].columns:
            fa_t.loc[common_t, c] = td['flux'].loc[common_t, c]
    x_f_t = fa_t.values
    x_c_t = td['adata'].obs[cov_avail].values
    
    X_test = np.hstack([x_e_t, x_f_t, x_c_t])
    y_test = td['adata'].obs['target_proliferation_binary'].values
    
    print(f"  Train: {X_train.shape} | Test: {X_test.shape}")
    print(f"  Train class: {np.bincount(y_train.astype(int))}")
    
    # Standardize
    scaler = StandardScaler()
    X_train_s = np.nan_to_num(scaler.fit_transform(X_train), 0)
    X_test_s = np.nan_to_num(scaler.transform(X_test), 0)
    
    # Class weight
    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    pos_weight = torch.FloatTensor([n_neg / max(n_pos, 1)]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    # DataLoaders
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train_s), torch.FloatTensor(y_train)),
        batch_size=256, shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_test_s), torch.FloatTensor(y_test)),
        batch_size=256
    )
    
    # Model
    model = MetabolicTransformer(
        n_expression_features=len(CONSENSUS_PATHWAYS), n_flux_features=len(CONSISTENT_FLUX_COLS), n_covariate_features=len(cov_avail),
        d_token=48, n_heads=4, n_layers=2, dropout=0.3
    ).to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)
    early_stop = EarlyStopping(patience=25)
    
    for epoch in range(150):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b).squeeze(-1), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        
        model.eval()
        vp, vl = [], []
        with torch.no_grad():
            for X_b, y_b in test_loader:
                vp.append(torch.sigmoid(model(X_b.to(DEVICE)).squeeze(-1)).cpu().numpy())
                vl.append(y_b.numpy())
        vp, vl = np.concatenate(vp), np.concatenate(vl)
        
        try: auc = roc_auc_score(vl, vp)
        except: auc = 0.5
        early_stop(auc, model)
        if early_stop.early_stop:
            print(f"  Early stop at epoch {epoch+1} | Best AUC: {early_stop.best_score:.4f}")
            break
    
    early_stop.load_best(model)
    model.eval()
    tp, tl = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            tp.append(torch.sigmoid(model(X_b.to(DEVICE)).squeeze(-1)).cpu().numpy())
            tl.append(y_b.numpy())
    tp, tl = np.concatenate(tp), np.concatenate(tl)
    
    # Optimal threshold
    best_f1, best_th = 0, 0.5
    for th in np.arange(0.2, 0.8, 0.05):
        f = f1_score(tl, (tp >= th).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_th = f, th
    
    auc = roc_auc_score(tl, tp)
    pb = (tp >= best_th).astype(int)
    acc = accuracy_score(tl, pb)
    prec, rec, f1, _ = precision_recall_fscore_support(tl, pb, average='binary', zero_division=0)
    
    lopo_standard_results.append({
        'patient': test_pid, 'auc': auc, 'f1': f1, 'accuracy': acc,
        'precision': prec, 'recall': rec, 'threshold': best_th, 'n_spots': len(tl)
    })
    
    print(f"  Patient {test_pid}: AUC={auc:.4f} | F1={f1:.4f} | Acc={acc:.4f}")
    
    del model, optimizer; gc.collect()

lopo_standard_df = pd.DataFrame(lopo_standard_results)

# FINAL SUMMARY
print("LOPO SUMMARY — STANDARD VISIUM (Patients 1-6)")
print(f"  Mean AUC:  {lopo_standard_df['auc'].mean():.4f} ± {lopo_standard_df['auc'].std():.4f}")
print(f"  Mean F1:   {lopo_standard_df['f1'].mean():.4f} ± {lopo_standard_df['f1'].std():.4f}")
print(f"  Mean Acc:  {lopo_standard_df['accuracy'].mean():.4f} ± {lopo_standard_df['accuracy'].std():.4f}")

print("\nPer-Patient Results:")
print(lopo_standard_df[['patient', 'auc', 'f1', 'accuracy', 'n_spots']].to_string(index=False))

# Save
lopo_standard_df.to_csv('results/transformer/lopo_standard_visium.csv', index=False)
print(f"\nSaved: results/transformer/lopo_standard_visium.csv")

In [ ]:
# USE ORIGINAL ADATA FOR PATIENT 7

print("USING ORIGINAL ADATA FOR PATIENT 7")

# Target labels -- needed below regardless of whether the specific
# 'Unwinding of DNA' pathway column happens to be present in adata.obs.
y_orig = adata.obs['target_proliferation_binary'].values

# Check the correlation of the key pathway, if available
if 'Reactome_Unwinding Of DNA R-HSA-176974' in adata.obs.columns:
    unwinding = adata.obs['Reactome_Unwinding Of DNA R-HSA-176974'].values
    rho, pval = spearmanr(unwinding, y_orig)
    print(f"\n'Reactome_Unwinding Of DNA R-HSA-176974':")
    print(f"  Correlation with target: ρ = {rho:.3f} (p = {pval:.2e})")
else:
    print("\n  'Reactome_Unwinding Of DNA R-HSA-176974' not found in adata.obs -- skipping.")

print(f"\nOriginal adata: {adata.n_obs} spots")
print(f"Target distribution: {np.bincount(y_orig.astype(int))}")

# Find all pathway columns in original adata
orig_pathway_cols = [c for c in adata.obs.columns 
                     if any(db in c for db in ['Reactome', 'KEGG', 'Hallmark', 'BioCarta', 'GOBP', 'h_'])]
print(f"Pathway columns: {len(orig_pathway_cols)}")

# Score pathways by correlation
orig_corrs = []
for col in orig_pathway_cols:
    vals = adata.obs[col].values
    if vals.std() > 0:
        rho, pval = spearmanr(vals, y_orig)
        if not np.isnan(rho):
            orig_corrs.append({'pathway': col, 'rho': rho, 'abs_rho': abs(rho), 'pval': pval})

orig_corr_df = pd.DataFrame(orig_corrs).sort_values('abs_rho', ascending=False)

print("TOP 20 PATHWAYS IN ORIGINAL ADATA:")
for _, row in orig_corr_df.head(20).iterrows():
    direction = "↑" if row['rho'] > 0 else "↓"
    print(f"  {direction} {row['pathway'][:55]} (ρ={row['rho']:.3f})")

# Select top 20
ORIG_P7_PATHWAYS = orig_corr_df.head(20)['pathway'].tolist()

print(f"\n Max correlation: ρ = {orig_corr_df['abs_rho'].iloc[0]:.3f}")

In [ ]:
# SPATIAL CV ON PATIENT 7

print("SPATIAL CV ON Patient 7")


# Prepare features from original adata
X_expr_orig = adata.obs[ORIG_P7_PATHWAYS].values

# Flux — align with original adata barcodes
p7_flux = patient_data[7]['flux']
X_flux_orig = pd.DataFrame(0.0, index=adata.obs_names, columns=CONSISTENT_FLUX_COLS)
common_orig = adata.obs_names.intersection(p7_flux.index)
print(f"Flux barcode overlap: {len(common_orig)} / {adata.n_obs}")

for c in CONSISTENT_FLUX_COLS:
    if c in p7_flux.columns:
        X_flux_orig.loc[common_orig, c] = p7_flux.loc[common_orig, c]
X_flux_orig = X_flux_orig.values

# Covariates
cov_avail_orig = [c for c in ['log1p_total_counts', 'pct_counts_mt', 'pct_counts_ribo'] 
                  if c in adata.obs.columns]
X_cov_orig = adata.obs[cov_avail_orig].values

X_orig = np.hstack([X_expr_orig, X_flux_orig, X_cov_orig])
y_orig = adata.obs['target_proliferation_binary'].values

print(f"Feature matrix: {X_orig.shape}")
print(f"  Expression: {len(ORIG_P7_PATHWAYS)} | Flux: {len(CONSISTENT_FLUX_COLS)} | Cov: {len(cov_avail_orig)}")
print(f"Target: {np.bincount(y_orig.astype(int))}")

# Spatial blocks from HD coordinates
rows_orig, cols_orig = [], []
for bc in adata.obs_names:
    parts = bc.replace('-1', '').split('_')
    if len(parts) >= 4:
        rows_orig.append(int(parts[2]))
        cols_orig.append(int(parts[3]))
    else:
        rows_orig.append(0)
        cols_orig.append(0)

coords_orig = np.column_stack([rows_orig, cols_orig])
kmeans_orig = KMeans(n_clusters=5, random_state=42, n_init=10)
spatial_blocks_orig = kmeans_orig.fit_predict(coords_orig)

# Run CV with ORIGINAL architecture
print(f"\nRunning spatial CV with original architecture (d=64, layers=3)...")

cv_results_orig = spatial_cv_transformer(
    X=X_orig,
    y=y_orig,
    spatial_blocks=spatial_blocks_orig,
    feature_names=ORIG_P7_PATHWAYS + CONSISTENT_FLUX_COLS + cov_avail_orig,
    feature_types=['expression']*len(ORIG_P7_PATHWAYS) + ['flux']*len(CONSISTENT_FLUX_COLS) + ['covariate']*len(cov_avail_orig),
    
    # Original architecture
    d_token=64,
    n_heads=4,
    n_layers=3,
    dropout=0.2,
    
    # Original training
    lr=1e-4,
    weight_decay=1e-2,
    epochs=150,
    batch_size=256,
    patience=20,
    
    n_splits=5,
    device=DEVICE,
    verbose=True
)

print("SPATIAL CV SUMMARY (Patient 7)")
fold_df = cv_results_orig['fold_results']
print(f"  Mean AUC:  {fold_df['auc'].mean():.4f} ± {fold_df['auc'].std():.4f}")
print(f"  Mean F1:   {fold_df['f1'].mean():.4f} ± {fold_df['f1'].std():.4f}")
print(f"  Mean Acc:  {fold_df['accuracy'].mean():.4f} ± {fold_df['accuracy'].std():.4f}")

# Save
fold_df.to_csv('results/transformer/spatial_cv_patient7_original.csv', index=False)
print("\nSaved: results/transformer/spatial_cv_patient7_original.csv")

In [ ]:
# MODEL COMPARISON: LOGISTIC REGRESSION vs METABOLIC TRANSFORMER

print("MODEL COMPARISON: LOGISTIC REGRESSION vs METABOLIC TRANSFORMER")


from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import GroupKFold

all_patients = [1, 2, 3, 4, 5, 6, 7]
standard_patients = [1, 2, 3, 4, 5, 6]

lr_results = []
transformer_results = []


#  LOPO FOR STANDARD VISIUM (Patients 1-6)


print(" LOPO VALIDATION — Standard Visium (Patients 1-6)")


for test_pid in standard_patients:
    train_pids = [p for p in standard_patients if p != test_pid]
    
    # Build training data
    X_train_parts, y_train_parts = [], []
    for p in train_pids:
        data = patient_data[p]
        x_e = data['adata'].obs[CONSENSUS_PATHWAYS].values
        
        fa = pd.DataFrame(0.0, index=data['adata'].obs_names, columns=CONSISTENT_FLUX_COLS)
        common = data['adata'].obs_names.intersection(data['flux'].index)
        for c in CONSISTENT_FLUX_COLS:
            if c in data['flux'].columns:
                fa.loc[common, c] = data['flux'].loc[common, c]
        x_f = fa.values
        x_c = data['adata'].obs[cov_avail].values
        
        X_train_parts.append(np.hstack([x_e, x_f, x_c]))
        y_train_parts.append(data['adata'].obs['target_proliferation_binary'].values)
    
    X_train = np.vstack(X_train_parts)
    y_train = np.concatenate(y_train_parts)
    
    # Build test data
    td = patient_data[test_pid]
    x_e_t = td['adata'].obs[CONSENSUS_PATHWAYS].values
    fa_t = pd.DataFrame(0.0, index=td['adata'].obs_names, columns=CONSISTENT_FLUX_COLS)
    common_t = td['adata'].obs_names.intersection(td['flux'].index)
    for c in CONSISTENT_FLUX_COLS:
        if c in td['flux'].columns:
            fa_t.loc[common_t, c] = td['flux'].loc[common_t, c]
    x_f_t = fa_t.values
    x_c_t = td['adata'].obs[cov_avail].values
    
    X_test = np.hstack([x_e_t, x_f_t, x_c_t])
    y_test = td['adata'].obs['target_proliferation_binary'].values
    
    # Standardize
    scaler = StandardScaler()
    X_train_s = np.nan_to_num(scaler.fit_transform(X_train), 0)
    X_test_s = np.nan_to_num(scaler.transform(X_test), 0)
    
    # Logistic Regression 
    lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced', C=1.0)
    lr.fit(X_train_s, y_train)
    lr_proba = lr.predict_proba(X_test_s)[:, 1]
    
    best_f1_lr, best_th_lr = 0, 0.5
    for th in np.arange(0.2, 0.8, 0.05):
        f = f1_score(y_test, (lr_proba >= th).astype(int), zero_division=0)
        if f > best_f1_lr: best_f1_lr, best_th_lr = f, th
    
    lr_auc = roc_auc_score(y_test, lr_proba)
    lr_pred = (lr_proba >= best_th_lr).astype(int)
    lr_acc = accuracy_score(y_test, lr_pred)
    _, _, lr_f1, _ = precision_recall_fscore_support(y_test, lr_pred, average='binary', zero_division=0)
    
    lr_results.append({
        'patient': test_pid, 'platform': 'Standard', 'validation': 'LOPO',
        'auc': lr_auc, 'f1': lr_f1, 'accuracy': lr_acc
    })
    
    #  Metabolic Transformer
    n_pos = y_train.sum()
    pos_weight = torch.FloatTensor([(len(y_train) - n_pos) / max(n_pos, 1)]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train_s), torch.FloatTensor(y_train)),
        batch_size=256, shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_test_s), torch.FloatTensor(y_test)),
        batch_size=256
    )
    
    model = MetabolicTransformer(
        n_expression_features=len(CONSENSUS_PATHWAYS),
        n_flux_features=len(CONSISTENT_FLUX_COLS),
        n_covariate_features=len(cov_avail),
        d_token=48, n_heads=4, n_layers=2, dropout=0.3
    ).to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)
    early_stop = EarlyStopping(patience=25)
    
    for epoch in range(150):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b).squeeze(-1), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        
        model.eval()
        vp, vl = [], []
        with torch.no_grad():
            for X_b, y_b in test_loader:
                vp.append(torch.sigmoid(model(X_b.to(DEVICE)).squeeze(-1)).cpu().numpy())
                vl.append(y_b.numpy())
        vp, vl = np.concatenate(vp), np.concatenate(vl)
        
        try: auc = roc_auc_score(vl, vp)
        except: auc = 0.5
        early_stop(auc, model)
        if early_stop.early_stop:
            break
    
    early_stop.load_best(model)
    model.eval()
    tp, tl = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            tp.append(torch.sigmoid(model(X_b.to(DEVICE)).squeeze(-1)).cpu().numpy())
            tl.append(y_b.numpy())
    tp, tl = np.concatenate(tp), np.concatenate(tl)
    
    best_f1_tr, best_th_tr = 0, 0.5
    for th in np.arange(0.2, 0.8, 0.05):
        f = f1_score(tl, (tp >= th).astype(int), zero_division=0)
        if f > best_f1_tr: best_f1_tr, best_th_tr = f, th
    
    tr_auc = roc_auc_score(tl, tp)
    tr_acc = accuracy_score(tl, (tp >= best_th_tr).astype(int))
    _, _, tr_f1, _ = precision_recall_fscore_support(tl, (tp >= best_th_tr).astype(int), average='binary', zero_division=0)
    
    transformer_results.append({
        'patient': test_pid, 'platform': 'Standard', 'validation': 'LOPO',
        'auc': tr_auc, 'f1': tr_f1, 'accuracy': tr_acc
    })
    
    print(f"  Patient {test_pid}: LR AUC={lr_auc:.4f} | Transformer AUC={tr_auc:.4f}")
    
    del model, optimizer; gc.collect()



#  SPATIAL CV FOR VISIUM HD (Patient 7)

p7_data = patient_data[7]
p7_adata = p7_data['adata']
p7_flux = p7_data['flux']

# Prepare features
X_expr_p7 = p7_adata.obs[CONSENSUS_PATHWAYS].values
X_flux_p7 = pd.DataFrame(0.0, index=p7_adata.obs_names, columns=CONSISTENT_FLUX_COLS)
common_p7 = p7_adata.obs_names.intersection(p7_flux.index)
for c in CONSISTENT_FLUX_COLS:
    if c in p7_flux.columns:
        X_flux_p7.loc[common_p7, c] = p7_flux.loc[common_p7, c]
X_flux_p7 = X_flux_p7.values
X_cov_p7 = p7_adata.obs[cov_avail].values

X_p7 = np.hstack([X_expr_p7, X_flux_p7, X_cov_p7])
y_p7 = p7_adata.obs['target_proliferation_binary'].values

# Create spatial blocks
rows_p7, cols_p7 = [], []
for bc in p7_adata.obs_names:
    parts = bc.replace('-1', '').split('_')
    if len(parts) >= 4:
        rows_p7.append(int(parts[2]))
        cols_p7.append(int(parts[3]))
    else:
        rows_p7.append(0)
        cols_p7.append(0)

coords_p7 = np.column_stack([rows_p7, cols_p7])
kmeans_p7 = KMeans(n_clusters=5, random_state=42, n_init=10)
spatial_blocks_p7 = kmeans_p7.fit_predict(coords_p7)

print(f"  Patient 7: {p7_adata.n_obs} spots | Class: {np.bincount(y_p7.astype(int))}")

# 5-Fold Spatial CV
cv_p7 = GroupKFold(n_splits=5)
lr_aucs_p7, tr_aucs_p7 = [], []
lr_f1s_p7, tr_f1s_p7 = [], []

for fold, (train_idx, test_idx) in enumerate(cv_p7.split(X_p7, y_p7, groups=spatial_blocks_p7)):
    X_train, X_test = X_p7[train_idx], X_p7[test_idx]
    y_train, y_test = y_p7[train_idx], y_p7[test_idx]
    
    scaler = StandardScaler()
    X_train_s = np.nan_to_num(scaler.fit_transform(X_train), 0)
    X_test_s = np.nan_to_num(scaler.transform(X_test), 0)
    
    # Logistic Regression 
    lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
    lr.fit(X_train_s, y_train)
    lr_proba = lr.predict_proba(X_test_s)[:, 1]
    lr_aucs_p7.append(roc_auc_score(y_test, lr_proba))
    
    best_f1_lr = 0
    for th in np.arange(0.2, 0.8, 0.05):
        f = f1_score(y_test, (lr_proba >= th).astype(int), zero_division=0)
        if f > best_f1_lr: best_f1_lr = f
    lr_f1s_p7.append(best_f1_lr)
    
    #  Metabolic Transformer 
    n_pos = y_train.sum()
    pos_weight = torch.FloatTensor([(len(y_train) - n_pos) / max(n_pos, 1)]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train_s), torch.FloatTensor(y_train)),
        batch_size=256, shuffle=True
    )
    test_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_test_s), torch.FloatTensor(y_test)),
        batch_size=256
    )
    
    model = MetabolicTransformer(
        n_expression_features=len(CONSENSUS_PATHWAYS),
        n_flux_features=len(CONSISTENT_FLUX_COLS),
        n_covariate_features=len(cov_avail),
        d_token=48, n_heads=4, n_layers=2, dropout=0.3
    ).to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)
    early_stop = EarlyStopping(patience=20)
    
    for epoch in range(150):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b).squeeze(-1), y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        
        model.eval()
        vp, vl = [], []
        with torch.no_grad():
            for X_b, y_b in test_loader:
                vp.append(torch.sigmoid(model(X_b.to(DEVICE)).squeeze(-1)).cpu().numpy())
                vl.append(y_b.numpy())
        vp, vl = np.concatenate(vp), np.concatenate(vl)
        
        try: auc = roc_auc_score(vl, vp)
        except: auc = 0.5
        early_stop(auc, model)
        if early_stop.early_stop:
            break
    
    early_stop.load_best(model)
    model.eval()
    tp, tl = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            tp.append(torch.sigmoid(model(X_b.to(DEVICE)).squeeze(-1)).cpu().numpy())
            tl.append(y_b.numpy())
    tp, tl = np.concatenate(tp), np.concatenate(tl)
    
    tr_aucs_p7.append(roc_auc_score(tl, tp))
    
    best_f1_tr = 0
    for th in np.arange(0.2, 0.8, 0.05):
        f = f1_score(tl, (tp >= th).astype(int), zero_division=0)
        if f > best_f1_tr: best_f1_tr = f
    tr_f1s_p7.append(best_f1_tr)
    
    print(f"  Fold {fold+1}: LR AUC={lr_aucs_p7[-1]:.4f} | Transformer AUC={tr_aucs_p7[-1]:.4f}")
    
    del model, optimizer; gc.collect()

# Add Patient 7 results (mean of CV folds)
lr_results.append({
    'patient': 7, 'platform': 'Visium HD', 'validation': 'Spatial CV',
    'auc': np.mean(lr_aucs_p7), 'f1': np.mean(lr_f1s_p7), 'accuracy': np.nan
})
transformer_results.append({
    'patient': 7, 'platform': 'Visium HD', 'validation': 'Spatial CV',
    'auc': np.mean(tr_aucs_p7), 'f1': np.mean(tr_f1s_p7), 'accuracy': np.nan
})


# RESULTS TABLE

lr_df = pd.DataFrame(lr_results)
transformer_df = pd.DataFrame(transformer_results)

comparison_df = pd.merge(
    lr_df[['patient', 'platform', 'validation', 'auc', 'f1']].rename(
        columns={'auc': 'lr_auc', 'f1': 'lr_f1'}),
    transformer_df[['patient', 'auc', 'f1']].rename(
        columns={'auc': 'transformer_auc', 'f1': 'transformer_f1'}),
    on='patient'
)
comparison_df['delta_auc'] = comparison_df['transformer_auc'] - comparison_df['lr_auc']


print("RESULTS: LOGISTIC REGRESSION vs METABOLIC TRANSFORMER")


print(f"\n{'Patient':<10}{'Platform':<12}{'Validation':<12}{'LR AUC':<12}{'Transformer AUC':<18}{'Δ AUC':<10}")


for _, row in comparison_df.iterrows():
    print(f"Patient {row['patient']:<3}{row['platform']:<12}{row['validation']:<12}{row['lr_auc']:<12.4f}{row['transformer_auc']:<18.4f}{row['delta_auc']:+.4f}")



# Summary statistics
lr_mean_all = lr_df['auc'].mean()
tr_mean_all = transformer_df['auc'].mean()
lr_std_all = lr_df['auc'].std()
tr_std_all = transformer_df['auc'].std()

lr_mean_std = lr_df[lr_df['platform'] == 'Standard']['auc']
tr_mean_std = transformer_df[transformer_df['platform'] == 'Standard']['auc']


print("SUMMARY STATISTICS")

print(f"\n  Standard Visium (n=6, LOPO):")
print(f"    Logistic Regression:   AUC = {lr_mean_std.mean():.4f} ± {lr_mean_std.std():.4f}")
print(f"    Metabolic Transformer: AUC = {tr_mean_std.mean():.4f} ± {tr_mean_std.std():.4f}")

print(f"\n  Visium HD (n=1, Spatial CV):")
print(f"    Logistic Regression:   AUC = {np.mean(lr_aucs_p7):.4f} ± {np.std(lr_aucs_p7):.4f}")
print(f"    Metabolic Transformer: AUC = {np.mean(tr_aucs_p7):.4f} ± {np.std(tr_aucs_p7):.4f}")

print(f"\n  Overall (n=7):")
print(f"    Logistic Regression:   AUC = {lr_mean_all:.4f} ± {lr_std_all:.4f}")
print(f"    Metabolic Transformer: AUC = {tr_mean_all:.4f} ± {tr_std_all:.4f}")

# Statistical test (Standard Visium only, n=6)
from scipy.stats import wilcoxon
std_comparison = comparison_df[comparison_df['platform'] == 'Standard']
try:
    stat, pval = wilcoxon(std_comparison['transformer_auc'], std_comparison['lr_auc'])
    print(f"\n  Wilcoxon signed-rank test (Standard Visium): p = {pval:.4f}")
except Exception as e:
    print(f"\n  Wilcoxon test: {e}")

# Save results
os.makedirs('results/transformer', exist_ok=True)
lr_df.to_csv('results/transformer/lopo_logistic_regression.csv', index=False)
transformer_df.to_csv('results/transformer/lopo_transformer.csv', index=False)
comparison_df.to_csv('results/transformer/model_comparison.csv', index=False)


print("Saved:")
print("  results/transformer/lopo_logistic_regression.csv")
print("  results/transformer/lopo_transformer.csv")
print("  results/transformer/model_comparison.csv")

In [ ]:
# LOPO cross-validation results 

lopo_cmp = comparison_df[comparison_df['platform'] == 'Standard'].sort_values('patient').reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# (A) AUC-ROC per patient, LR vs Transformer, plus cross-patient mean +/- SD
ax = axes[0]
patient_labels = [f"P{p}" for p in lopo_cmp['patient']]
x = np.arange(len(patient_labels) + 1)  
width = 0.35

lr_vals = list(lopo_cmp['lr_auc']) + [lopo_cmp['lr_auc'].mean()]
tr_vals = list(lopo_cmp['transformer_auc']) + [lopo_cmp['transformer_auc'].mean()]
lr_err = [0] * len(patient_labels) + [lopo_cmp['lr_auc'].std()]
tr_err = [0] * len(patient_labels) + [lopo_cmp['transformer_auc'].std()]

ax.bar(x - width / 2, lr_vals, width, yerr=lr_err, capsize=3,
       color=COLOR_SECONDARY, label='Logistic Regression')
ax.bar(x + width / 2, tr_vals, width, yerr=tr_err, capsize=3,
       color=COLOR_PRIMARY, label='Metabolic Transformer')
ax.axhline(0.5, color=COLOR_REFERENCE, linestyle='--', linewidth=1, label='Chance (AUC = 0.5)')
ax.axvline(len(patient_labels) - 0.5, color='black', linewidth=0.8, alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(patient_labels + ['Mean \u00b1 SD'])
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.4, 1.0)
ax.set_title(' LOPO cross-validation: AUC-ROC by patient')
ax.legend(fontsize=8, loc='lower right')

# (B) Per-patient delta AUC (Transformer - LR), ranked
ax = axes[1]
ranked = lopo_cmp.sort_values('delta_auc', ascending=True)
bar_colors = [COLOR_PRIMARY if d > 0 else COLOR_SECONDARY for d in ranked['delta_auc']]
ax.barh([f"P{p}" for p in ranked['patient']], ranked['delta_auc'], color=bar_colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('$\\Delta$AUC (Transformer $-$ LR)')
ax.set_title('Per-patient improvement over baseline')

plt.tight_layout()
os.makedirs('figures', exist_ok=True)
plt.savefig('figures/Fig_LOPO_validation.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: figures/Fig_LOPO_validation.png")
print(f"Mean AUC \u2014 LR: {lopo_cmp['lr_auc'].mean():.4f} \u00b1 {lopo_cmp['lr_auc'].std():.4f} | "
      f"Transformer: {lopo_cmp['transformer_auc'].mean():.4f} \u00b1 {lopo_cmp['transformer_auc'].std():.4f}")


In [ ]:
# FIGURE: Visium HD (Patient 7) spatial cross-validation results (Table 4 -> Figure)

n_folds = len(lr_aucs_p7)
fold_labels = [f"Fold {i + 1}" for i in range(n_folds)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# (A) AUC-ROC per fold, LR vs Transformer, plus cross-fold mean +/- SD
ax = axes[0]
x = np.arange(n_folds + 1)
width = 0.35

lr_vals = list(lr_aucs_p7) + [np.mean(lr_aucs_p7)]
tr_vals = list(tr_aucs_p7) + [np.mean(tr_aucs_p7)]
lr_err = [0] * n_folds + [np.std(lr_aucs_p7)]
tr_err = [0] * n_folds + [np.std(tr_aucs_p7)]

ax.bar(x - width / 2, lr_vals, width, yerr=lr_err, capsize=3,
       color=COLOR_SECONDARY, label='Logistic Regression')
ax.bar(x + width / 2, tr_vals, width, yerr=tr_err, capsize=3,
       color=COLOR_PRIMARY, label='Metabolic Transformer')
ax.axhline(0.5, color=COLOR_REFERENCE, linestyle='--', linewidth=1, label='Chance (AUC = 0.5)')
ax.axvline(n_folds - 0.5, color='black', linewidth=0.8, alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(fold_labels + ['Mean \u00b1 SD'])
ax.set_ylabel('AUC-ROC')
ax.set_ylim(0.4, 1.0)
ax.set_title(' Visium HD (Patient 7) spatial CV: AUC-ROC by fold')
ax.legend(fontsize=8, loc='lower right')

# (B) Per-fold delta AUC (Transformer - LR), ranked
ax = axes[1]
deltas_p7 = np.array(tr_aucs_p7) - np.array(lr_aucs_p7)
order = np.argsort(deltas_p7)
bar_colors = [COLOR_PRIMARY if d > 0 else COLOR_SECONDARY for d in deltas_p7[order]]
ax.barh([fold_labels[i] for i in order], deltas_p7[order], color=bar_colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('$\\Delta$AUC (Transformer $-$ LR)')
ax.set_title(' Per-fold improvement over baseline')

plt.tight_layout()
plt.savefig('figures/Fig_VisiumHD_validation.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: figures/Fig_VisiumHD_validation.png")
print(f"Mean AUC \u2014 LR: {np.mean(lr_aucs_p7):.4f} \u00b1 {np.std(lr_aucs_p7):.4f} | "
      f"Transformer: {np.mean(tr_aucs_p7):.4f} \u00b1 {np.std(tr_aucs_p7):.4f}")
